# PTM Interpretability Pipeline -- N1: Corpus & Labels (M1-M3 + M_E)

First of four notebooks in the split pipeline (N1 -> N2 x2 passes -> N3 -> N4). Strictly dataset
construction: acquisition (M1), dataset exploration (M_E), corpus construction (M2), label
construction (M3). No model loading, no GPU, no SAE code.

Builds the human corpus plus OOD corpora/labels for mouse, rat and yeast.

**Output contract:** see the final cell. Save a notebook version, then Kaggle "New Dataset" ->
"Notebook Output", and attach to N2 (both passes) and separately to N3.

**Changelog:** every `[N1-CL#N]` marker refers to an entry in `N1_CHANGELOG.md`. Dataset-level
facts and quirks live in `N1_DATASET_GLOSSARY.md`.


In [1]:
import shutil
import subprocess
import sys


def _check_cdhit():
    """[N1-CL#56] Local-machine setup: cd-hit is a system binary (not pip-installable), and
    unlike a fresh Kaggle container this machine persists between runs -- check first rather
    than blindly re-running an installer (which also needs sudo, unavailable by default here).
    Install once via the OS package manager (see the accompanying setup notes), then this
    passes silently on every future run."""
    if shutil.which("cd-hit") is None or shutil.which("cd-hit-2d") is None:
        raise RuntimeError(
            "cd-hit / cd-hit-2d not found on PATH. Install via your OS package manager before "
            "running M2 (e.g. `sudo apt install cd-hit` on Debian/Ubuntu, `brew install cd-hit` "
            "on macOS, or build from https://github.com/weizhongli/cdhit). "
            "Everything before M2 (M1, M_E) will still run without it."
        )
    print(f"cd-hit: {shutil.which('cd-hit')}")
    print(f"cd-hit-2d: {shutil.which('cd-hit-2d')}")


_check_cdhit()

# MOCK_MODE = True  -> synthetic fixtures only, no network needed, safe anywhere.
# MOCK_MODE = False -> real acquisition. Requires open internet and QPTM_DIRECT_URL set as an
#                       environment variable (see 01_m1_acquisition's N1-CL#6/#55).
MOCK_MODE = False

# FORCE_REBUILD = False -> reuse any valid cache (the normal case).
# FORCE_REBUILD = True  -> ignore every cache, rebuild from scratch (use once after editing
#                          parsing/filtering LOGIC, since fingerprints hash config, not code).
FORCE_REBUILD = False


cd-hit: /usr/bin/cd-hit
cd-hit-2d: /usr/bin/cd-hit-2d


## M0 -- Configuration & Design Invariants

In [2]:
"""
N1 — Part 0: Configuration & Design Invariants
===============================================================================
Single source of truth for M1-M3+M_E. Trimmed from the original pipeline's Config to only
the fields this notebook's modules actually read (verified against config_fingerprint()'s own
field list, which already only covered M1-M3) -- no ARMS, no SAE/GPU/statistics/causal fields,
since none of that is N1's concern. N2 (M4-M6) defines its own arm registry independently.

Defines: Paths, Config/CFG, Manifest, and the fingerprint/cache helpers M1/M2/M3/M_E share.
"""
from __future__ import annotations
import dataclasses
import hashlib
import json
import shutil
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Dict, List, Optional

# ---------------------------------------------------------------------------
# Paths -- Kaggle convention (/kaggle/working persists as notebook output, 20GB cap;
# /kaggle/temp is scratch, not persisted). Falls back to local dirs off-Kaggle.
# [N1-CL#1] Paths.INPUT set to the standard Kaggle mount point (/kaggle/input) -- the source
# notebooks disagreed with each other here (see N1_CHANGELOG.md #1); this resolves it.
# ---------------------------------------------------------------------------
_ON_KAGGLE = Path("/kaggle").exists()

class Paths:
    ROOT = Path("/kaggle/working") if _ON_KAGGLE else Path("./ptm_working")
    TEMP = Path("/kaggle/temp") if _ON_KAGGLE else Path("./ptm_temp")
    INPUT = Path("/kaggle/input") if _ON_KAGGLE else Path("./ptm_input")

    RAW = TEMP / "raw"                                     # M1 downloads, scratch-only
    MANIFEST = ROOT / "manifest.json"                       # M1 SHA/provenance log

    CORPUS = ROOT / "corpus.parquet"                        # M2
    STRATUM_COUNTS = ROOT / "stratum_counts.json"           # M2
    SPLIT_MANIFEST = ROOT / "split_manifest.json"           # M2

    LABELS_STRATIFIED = ROOT / "labels_stratified.parquet"  # M3
    LABELS_NAIVE = ROOT / "labels_naive.parquet"            # M3
    EXCLUSION_MASK = ROOT / "exclusion_mask.parquet"        # M3
    TABLES = ROOT / "tables"                                # T1, T2, ... as .csv
    FIGURES = ROOT / "figures"                              # F2, ... as .png
    EXPLORATION = ROOT / "exploration"                      # M_E's per-source profile output (new)

    for _p in [ROOT, TEMP, RAW, TABLES, FIGURES, EXPLORATION]:
        _p.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Config -- only fields M1/M2/M3/M_E read. [N1-CL#2] DEBUG_SMALL_RUN and every *_debug field
# dropped: none of M1-M3's fields have a debug-scaled variant (confirmed against the original
# Config.__post_init__'s field list), so that whole mechanism was dead weight here.
# ---------------------------------------------------------------------------
@dataclasses.dataclass(frozen=True)
class Config:
    # Corpus
    max_protein_length: int = 1022
    organism_primary: str = "human"
    swissprot_reviewed_only: bool = True

    # Label filters
    min_sites_per_type: int = 200
    min_sites_headline: int = 1000
    # [N1-CL#34] Sites reported by >= this many independent sources define the high-confidence
    # subset (the plan's "free robustness analysis"). Not a filter -- source_count is stored per
    # site, and this is the threshold downstream analyses should use when subsetting.
    min_source_count_high_confidence: int = 2
    # [N1-CL#50] lp_class_positive / lp_class_masked REMOVED. CPLM's real bulk download has no
    # `lp` column at all (measured: 100% null), so the LP Class II/III ambiguity mask could never
    # fire. Keeping dead config invited the belief that CPLM sites were confidence-filtered when
    # they were not. If a future CPLM release restores the field, re-add both fields and step 3b
    # together -- see N1_CHANGELOG.md #50.

    # [N1-CL#22] qPTM's 0-5 reliability score floor (sites seen in >=N independent datasets).
    # [N1-CL#24] 5 -> 2. The real sensitivity sweep showed independence collapsing as the floor
    # rises: 62.9% of qPTM sites also in dbPTM at floor 0, 91.2% at floor 2, 98.2% at floor 5.
    # Floor 5 would have made the CF-26 "independent replication" ~98% circular. Floor 2 keeps
    # 186,525 unique human sites with ~16,400 genuinely dbPTM-independent -- the best available
    # balance between evidence quality and actual independence.
    qptm_reliability_floor: int = 2

    # [N1-CL#35] Out-of-distribution species -- FOUR, matching the original pipeline's own
    # CPLM_SPECIES_OOD list (mouse/rat/yeast/E.coli) and its v12a_cross_species_scaffold note.
    # E. coli was in the original and is restored here; its succinylation coverage (dbPTM 4,147
    # rows + CPLM) is the sharpest cross-species test the plan identifies (CF-31).
    # `cplm_species` strings are the original's verified strain-qualified names -- do not
    # simplify them; CPLM's download paths are strain-exact.
    # A None label means that source does not cover that species at all (measured, not assumed):
    # qPTM covers only Human/Mouse/Rat/Yeast; O-GlcNAcAtlas has no E. coli.
    # NOTE: N-GlycositeAtlas is NOT an OOD source -- its endpoint is human-only (HumanAll).
    ood_species: dict = dataclasses.field(default_factory=lambda: {
        "mouse": {"taxon_id": "10090", "uniprot_suffix": "MOUSE",
                  "cplm_species": "Mus musculus",
                  "qptm_label": "mouse", "oglcnac_label": "mouse"},
        "rat": {"taxon_id": "10116", "uniprot_suffix": "RAT",
                "cplm_species": "Rattus norvegicus",
                "qptm_label": "rat", "oglcnac_label": "rat"},
        "yeast": {"taxon_id": "559292", "uniprot_suffix": "YEAST",
                  "cplm_species": "Saccharomyces cerevisiae (strain ATCC 204508 or S288c)",
                  "qptm_label": "yeast", "oglcnac_label": "yeast"},
        "ecoli": {"taxon_id": "83333", "uniprot_suffix": "ECOLI",
                  "cplm_species": "Escherichia coli (strain K12)",
                  "qptm_label": None, "oglcnac_label": None},
    })

    # [N1-CL#36] CF-22: a (species, type) pair is only tested when that species has >= this many
    # sites of that type. Running every type against every species produces meaningless nulls --
    # E. coli largely lacks N-glycosylation; several lysine acylations are far better
    # characterised in bacteria than in yeast. Absence of a type in a species is a fact about
    # biology, not a failure of the latent.
    ood_min_sites_per_type: int = 200
    # Redundancy
    # [N1-CL#3] cdhit_identity: 0.50 -> 0.40 per explicit instruction; run_cdhit's word-size
    # band logic already supports 0.40 (the documented [0.4, 0.5) -> n=2 band), no code change
    # needed there -- only this default.
    cdhit_identity: float = 0.40
    cdhit_cluster_before_split: bool = True

    # Discovery / held-out split (cluster-level, assigned in M2)
    holdout_fraction: float = 0.20
    holdout_seed: int = 42
    # [N1-CL#34] DESCRIPTIVE ONLY -- recorded in split_manifest.json for provenance, never read
    # by iterative_stratified_split. Since N1-CL#26 stratification is always multi-label over
    # every canonical type; this field cannot change that. Left in (it is genuine provenance) but
    # do not mistake it for a switch.
    holdout_stratify_by: str = "ptm_type"
    holdout_min_sites_per_type: int = 200

    # Residue strata -> chemistry-valid target residues (M2/M3, and M7 downstream).
    # [N1-CL#14] E/M/W/Q added for Category B dbPTM types -- each verified against real
    # biochemistry literature before adding, see N1_CHANGELOG.md.
    stratum_residues: dict = dataclasses.field(default_factory=lambda: {
        "K": {"K"}, "ST": {"S", "T"}, "N": {"N"}, "C": {"C"}, "R": {"R"}, "Y": {"Y"},
        "E": {"E"}, "M": {"M"}, "W": {"W"}, "Q": {"Q"},
    })
    # PTM type -> stratum. Only the primary three are load-bearing for headline claims; the
    # rest exist so M3's "other PTM" pooling and crosstalk bookkeeping are complete.
    ptm_type_to_stratum: dict = dataclasses.field(default_factory=lambda: {
        # Lysine (K) -- CPLM's 29 verified types
        "succinylation": "K", "acetylation": "K", "crotonylation": "K", "malonylation": "K",
        "2-hydroxyisobutyrylation": "K", "beta-hydroxybutyrylation": "K", "butyrylation": "K",
        "propionylation": "K", "glutarylation": "K", "lactylation": "K", "formylation": "K",
        "benzoylation": "K", "HMGylation": "K", "MGcylation": "K", "MGylation": "K",
        "ubiquitination": "K", "sumoylation": "K", "pupylation": "K", "neddylation": "K",
        "methylation_K": "K", "glycation": "K", "hydroxylation_K": "K",
        "phosphoglycerylation": "K", "carboxymethylation": "K", "lipoylation": "K",
        "carboxylation": "K", "dietylphosphorylation": "K", "biotinylation": "K",
        "carboxyethylation": "K",
        # Ser/Thr
        "phosphorylation_ST": "ST", "O-GlcNAcylation": "ST", "O-glycosylation": "ST",
        # Asparagine
        "N-glycosylation": "N", "deamidation": "N",
        # Cysteine
        "S-nitrosylation": "C", "palmitoylation": "C", "glutathionylation": "C",
        "S-sulfhydration": "C",
        # Arginine
        "methylation_R": "R", "citrullination": "R", "ADP-ribosylation": "R",
        # Tyrosine
        "phosphorylation_Y": "Y", "nitration": "Y", "sulfation": "Y",
        # [N1-CL#13] Category A additions -- new canonical types, existing strata (C/Y).
        "S-cysteinylation": "C", "S-linked Glycosylation": "C", "Iodination": "Y",
        # [N1-CL#14] Category B additions -- new strata (E/M/W/Q). Residue verified via web
        # search before adding, not assumed -- see N1_CHANGELOG.md.
        "gamma-carboxyglutamic-acid": "E", "sulfoxidation": "M",
        "C-linked Glycosylation": "W", "serotonylation": "Q",
    })
    # The three primary/headline types this thesis leads with.
    primary_types: tuple = ("N-glycosylation", "O-GlcNAcylation", "succinylation")


CFG = Config()


# ---------------------------------------------------------------------------
# [N1-CL#17] Type canonicalisation, hoisted here from M3 (which originally owned it) so M_E can
# use the same mapping to report a type's stratum -- M_E now runs before M3 in this notebook, so
# M3-only ownership would have made the mapping unavailable when M_E needs it. Single source of
# truth for "how do we recognize what type a raw PTM label refers to", used by M_E's exploration
# and M3's real cascade alike, instead of two copies drifting apart.
# ---------------------------------------------------------------------------
PTM_TYPE_ALIASES: Dict[str, str] = {
    # normalised (lowercase, hyphens/spaces collapsed) source spelling -> canonical cfg key
    "n-glycosylation": "N-glycosylation", "n-linked-glycosylation": "N-glycosylation",
    "asparagine-linked-glycosylation": "N-glycosylation",
    "o-glcnacylation": "O-GlcNAcylation", "o-glcnac": "O-GlcNAcylation",
    "o-glycosylation": "O-glycosylation", "o-linked-glycosylation": "O-glycosylation",
    "succinylation": "succinylation", "acetylation": "acetylation",
    "crotonylation": "crotonylation", "malonylation": "malonylation",
    "ubiquitination": "ubiquitination", "sumoylation": "sumoylation",
    "phosphorylation": "phosphorylation_ST",  # residue split resolved by step2's chemistry check
    "methylation": "methylation_K",  # K vs R ambiguous at alias stage; step2 corrects R rows
    "s-nitrosylation": "S-nitrosylation", "palmitoylation": "palmitoylation",
    "citrullination": "citrullination", "nitration": "nitration", "sulfation": "sulfation",
    "deamidation": "deamidation",
    "s-palmitoylation": "palmitoylation",
    "glutathionylation": "glutathionylation",  # cfg key already lowercase; alias just ensures the mapping exists
    "sulfhydration": "S-sulfhydration",  # dbPTM's unprefixed name -> cfg's S-prefixed key
    "hydroxylation": "hydroxylation_K",  # CPLM is lysine-only, so its bare name is unambiguous
    "β-hydroxybutyrylation": "beta-hydroxybutyrylation",  # literal Greek beta vs ASCII spelling
    "ubiquitylation": "ubiquitination",  # [N1-CL#20] qPTM's spelling of the same modification
}


# ---------------------------------------------------------------------------
# [N1-CL#20] Residue-resolved types: source labels that cover MORE THAN ONE chemistry, and can
# only be assigned correctly by looking at the actual residue at the site. These are checked
# BEFORE PTM_TYPE_ALIASES and win when the residue is known.
#
# Without this, a single alias collapses every chemistry under one label into one canonical type
# (e.g. all dbPTM "Methylation" -> methylation_K), and step2 then silently DROPS every row whose
# real residue disagrees -- destroying the data and leaving the sibling type (methylation_R)
# permanently empty. Measured cost of that collapse on real data: dbPTM Methylation 29.9%
# chemistry-match, dbPTM Phosphorylation 83.7%, qPTM "Glycosylation" unmappable entirely.
#
# Only labels whose BOTH destinations already exist as canonical types are listed here. Labels
# that would need a new stratum first (dbPTM Hydroxylation -> mostly proline; Deamidation -> Q)
# are deliberately absent -- they fall through to the plain alias and behave exactly as before,
# pending the residue-failure EDA that says whether those strata are worth adding.
# ---------------------------------------------------------------------------
PTM_TYPE_RESIDUE_RESOLVED: Dict[str, Dict[str, str]] = {
    "glycosylation": {"N": "N-glycosylation", "S": "O-glycosylation", "T": "O-glycosylation"},
    "phosphorylation": {"S": "phosphorylation_ST", "T": "phosphorylation_ST", "Y": "phosphorylation_Y"},
    "methylation": {"K": "methylation_K", "R": "methylation_R"},
}


def _normalize_type_key(s: str) -> str:
    return str(s).strip().lower().replace("_", "-").replace("  ", " ").replace(" ", "-")


def normalize_ptm_type(raw_type: str, known_types: Optional[set] = None,
                        residue: Optional[str] = None) -> str:
    """Resolution order: residue-resolved table (only when `residue` is supplied and the label is
    multi-chemistry) -> alias table -> case/formatting variant of an existing canonical key
    (e.g. CPLM's "Carboxylation" vs config's "carboxylation"); that last step cannot invent a
    mapping, only resolve formatting noise around a key already present. Returns `raw_type`
    unchanged if nothing matches, so the caller's fail-loud unmapped-type check stays exactly as
    strict as before.

    `residue` is the actual amino acid at the site. When it's None (caller has no corpus
    sequence) or the label isn't residue-ambiguous, behaviour is identical to the alias-only
    lookup -- so passing it is strictly additive, never a regression."""
    key = _normalize_type_key(raw_type)
    if residue and key in PTM_TYPE_RESIDUE_RESOLVED:
        resolved = PTM_TYPE_RESIDUE_RESOLVED[key].get(residue)
        if resolved is not None:
            return resolved
        # Residue outside this label's known chemistries -- fall through to the plain alias so
        # step2 rejects it as a mismatch (visible in failure_breakdown), rather than silently
        # inventing a canonical type for a residue nobody has characterised for this label.
    if key in PTM_TYPE_ALIASES:
        return PTM_TYPE_ALIASES[key]
    if known_types:
        for canonical in known_types:
            if _normalize_type_key(canonical) == key:
                return canonical
    return raw_type


def type_to_stratum(raw_type: str) -> str:
    """Convenience for exploration/reporting: canonical stratum for a raw type string, or
    'UNMAPPED' if it doesn't (yet) resolve to any cfg.ptm_type_to_stratum entry."""
    canonical = normalize_ptm_type(raw_type, known_types=set(CFG.ptm_type_to_stratum))
    return CFG.ptm_type_to_stratum.get(canonical, "UNMAPPED")


# ---------------------------------------------------------------------------
# [N1-CL#42] Diagnostic printing. Everything diagnostic must appear IN THE NOTEBOOK OUTPUT, not
# only on disk -- the saved .ipynb is the only artefact that can be reviewed without the (large)
# output dataset. `display()` silently truncates rows AND columns (the first real run's T3 came
# out column-wrapped with backslashes), so every diagnostic table goes through show_table(),
# which uses to_string() and is therefore complete and diff-able in the saved notebook.
# ---------------------------------------------------------------------------
import pandas as _pd

_pd.set_option("display.max_rows", 500)
_pd.set_option("display.max_columns", 200)
_pd.set_option("display.width", 250)
_pd.set_option("display.max_colwidth", 60)


def show_table(df, title: Optional[str] = None, max_rows: Optional[int] = None):
    """Print a DataFrame in full -- no row or column truncation."""
    if title:
        print(f"\n{title}")
    if df is None or not len(df):
        print("  (empty)")
        return
    shown = df if max_rows is None or len(df) <= max_rows else df.head(max_rows)
    with _pd.option_context("display.max_rows", None, "display.max_columns", None,
                            "display.width", 250, "display.max_colwidth", 60):
        print(shown.to_string(index=False))
    if max_rows is not None and len(df) > max_rows:
        print(f"  ... {len(df) - max_rows} more row(s) not shown (full table saved to tables/)")


def show_json(obj, title: Optional[str] = None, indent: int = 2):
    """Print a nested dict/log in full, so step-level counts survive into the saved notebook."""
    if title:
        print(f"\n{title}")
    print(json.dumps(obj, indent=indent, default=str))


def cluster_size_report(corpus) -> dict:
    """CD-HIT diagnostics beyond a bare cluster count: size distribution, singleton fraction and
    the largest clusters. A clustering that collapsed (one giant cluster) or did nothing (all
    singletons) both produce a plausible-looking cluster COUNT, so the count alone cannot tell
    you whether CD-HIT worked."""
    sizes = corpus.groupby("cluster_id").size()
    largest = sizes.sort_values(ascending=False).head(10)
    return {
        "n_proteins": int(len(corpus)), "n_clusters": int(len(sizes)),
        "mean_cluster_size": round(float(sizes.mean()), 3),
        "median_cluster_size": int(sizes.median()),
        "max_cluster_size": int(sizes.max()),
        "n_singleton_clusters": int((sizes == 1).sum()),
        "pct_proteins_in_singletons": round(100 * int((sizes == 1).sum()) / len(corpus), 2),
        "size_distribution": {str(k): int(v) for k, v in sizes.value_counts().sort_index().head(15).items()},
        "largest_clusters": {str(k): int(v) for k, v in largest.items()},
    }


def save_config(path: Optional[Path] = None) -> Path:
    """Release artefact: plain-JSON dump of CFG, for exact reproducibility of N1's output."""
    path = path or (Paths.ROOT / "config.json")
    path.write_text(json.dumps(dataclasses.asdict(CFG), indent=2, default=str))
    return path


# ---------------------------------------------------------------------------
# Caching layer, shared by M1/M2/M3. Three-zone Kaggle filesystem: /kaggle/input (attached
# datasets, persists forever, checked first) -> /kaggle/working (this session's own prior
# output) -> /kaggle/temp (scratch). Every cached artefact is paired with a fingerprint
# sidecar, checked before reuse, so a config change can never silently reuse stale output.
# ---------------------------------------------------------------------------
def fingerprint(*parts) -> str:
    """Short stable hash of JSON-serialisable objects; used to invalidate caches."""
    h = hashlib.sha256()
    for p in parts:
        h.update(json.dumps(p, sort_keys=True, default=str).encode())
    return h.hexdigest()[:16]


def find_in_inputs(filename: str) -> Optional[Path]:
    """Search every attached Kaggle Dataset under Paths.INPUT, recursively, for `filename`."""
    if not Paths.INPUT.exists():
        return None
    for match in Paths.INPUT.rglob(filename):
        if match.is_file():
            return match
    return None


def find_cached(filename: str, also_search: Optional[List[Path]] = None) -> Optional[Path]:
    """Priority: Paths.ROOT (recursive) -> attached input datasets -> caller-supplied dirs."""
    if Paths.ROOT.exists():
        for match in Paths.ROOT.rglob(filename):
            if match.is_file():
                return match
    hit = find_in_inputs(filename)
    if hit is not None:
        return hit
    for d in (also_search or []):
        candidate = d / filename
        if candidate.exists():
            return candidate
    return None


@dataclass
class CacheResult:
    path: Path
    was_cached: bool
    source: str  # "working" | "input_dataset" | "built"


def ensure_artifact(filename: str, build_fn: "Callable[[Path], None]", *,
                     dest_dir: Optional[Path] = None, also_search: Optional[List[Path]] = None,
                     expected_fingerprint: Optional[str] = None, force: bool = False) -> CacheResult:
    """Core cache-or-build primitive. A hit under an attached input dataset is copied into
    Paths.ROOT so every downstream path assumption ("artefacts live under Paths.ROOT") holds
    regardless of where the cache hit came from."""
    dest_dir = dest_dir or Paths.ROOT
    dest = dest_dir / filename

    if not force:
        hit = find_cached(filename, also_search=also_search)
        if hit is not None:
            fp_ok = True
            if expected_fingerprint is not None:
                fp_sidecar = hit.with_suffix(hit.suffix + ".fp")
                fp_ok = fp_sidecar.exists() and fp_sidecar.read_text().strip() == expected_fingerprint
            if fp_ok:
                if hit != dest:
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(hit, dest)
                    if expected_fingerprint is not None:
                        dest.with_suffix(dest.suffix + ".fp").write_text(expected_fingerprint)
                source = "input_dataset" if Paths.INPUT in hit.parents else "working"
                return CacheResult(path=dest, was_cached=True, source=source)
            print(f"[cache] {filename}: cached copy found but fingerprint is stale -- rebuilding.")

    dest.parent.mkdir(parents=True, exist_ok=True)
    build_fn(dest)
    if expected_fingerprint is not None:
        dest.with_suffix(dest.suffix + ".fp").write_text(expected_fingerprint)
    return CacheResult(path=dest, was_cached=False, source="built")


def config_fingerprint(*extra, cfg: "Config" = None) -> str:
    """Fingerprint of every config field that affects M2/M3's output content. Pass upstream
    artefact hashes (e.g. an M1 manifest digest) as `extra` to also invalidate on raw-input
    changes. Takes `cfg` explicitly rather than always reading the global CFG, so a caller
    testing a modified config doesn't get a silently-wrong fingerprint."""
    cfg = cfg if cfg is not None else CFG
    relevant = {
        "max_protein_length": cfg.max_protein_length,
        "swissprot_reviewed_only": cfg.swissprot_reviewed_only,
        "cdhit_identity": cfg.cdhit_identity,
        "holdout_fraction": cfg.holdout_fraction,
        "holdout_seed": cfg.holdout_seed,
        "holdout_stratify_by": cfg.holdout_stratify_by,
        "min_sites_per_type": cfg.min_sites_per_type,
        "min_sites_headline": cfg.min_sites_headline,
        "ptm_type_to_stratum": cfg.ptm_type_to_stratum,
        "primary_types": cfg.primary_types,
        # [N1-CL#47] These three were MISSING and are output-affecting:
        #   stratum_residues        -> changes stratum_counts.json (g_ell) AND step2's chemistry
        #                              validation, so it changes both M2 and M3 output.
        #   qptm_reliability_floor  -> changes labels_qptm directly.
        #   ood_min_sites_per_type  -> changes T5's testable flags.
        # Without them, editing any of these and re-running would silently reuse stale cached
        # output -- and the reliability floor in particular is a documented tunable.
        # `stratum_residues` values are sets (unordered, not JSON-serialisable) so they are
        # sorted into lists first; otherwise the hash would vary run to run for identical config.
        "stratum_residues": {k: sorted(v) for k, v in cfg.stratum_residues.items()},
        "qptm_reliability_floor": cfg.qptm_reliability_floor,
        "ood_min_sites_per_type": cfg.ood_min_sites_per_type,
    }
    return fingerprint(relevant, *extra)


# ---------------------------------------------------------------------------
# Size reporting -- shared by M1's raw-download report and the final packaging cell.
# ---------------------------------------------------------------------------
def path_size_bytes(p: Optional[Path]) -> int:
    """A file's own size, or a directory's total (recursive). 0 for None/missing paths."""
    if p is None or not p.exists():
        return 0
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


def human_mb(n_bytes: int) -> str:
    return f"{n_bytes / 1e6:,.2f} MB"


# ---------------------------------------------------------------------------
# Manifest -- every acquired artefact's provenance (M1's other emission).
# ---------------------------------------------------------------------------
def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


@dataclass
class Manifest:
    entries: List[dict] = field(default_factory=list)

    def log(self, source: str, url: str, dest: Path, notes: str = "") -> dict:
        if dest.exists() and dest.is_file():
            sha, size = sha256_file(dest), dest.stat().st_size
        elif dest.exists() and dest.is_dir():
            sha, size = None, sum(f.stat().st_size for f in dest.rglob("*") if f.is_file())
        else:
            sha, size = None, None
        entry = {
            "source": source, "url": url, "dest": str(dest),
            "sha256": sha, "bytes": size,
            "accessed": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "notes": notes,
        }
        self.entries.append(entry)
        return entry

    def save(self, path: Path):
        path.write_text(json.dumps(self.entries, indent=2, default=str))


## M1 -- Acquisition

Swiss-Prot (human + 3 OOD proteomes), CPLM, dbPTM (37 types), qPTM, O-GlcNAcAtlas, N-GlycositeAtlas, N-GlyDE.

In [3]:
"""
M1 -- Acquisition
=================
Inputs: none. Emits: raw data in Paths.RAW; manifest.json (SHA sums + access dates).
Cost: minutes to ~1h depending on bandwidth. No GPU.

Sources: UniProt/Swiss-Prot, CPLM 4.0 (human), dbPTM 2025 (37 types), qPTM (Human/Mouse/Rat/
Yeast), O-GlcNAcAtlas 5.0, N-GlycositeAtlas, N-GlyDE (via StackGlyEmbed). [N1-CL#4] ESM-2/SAE model+checkpoint acquisition
moved OUT of M1 into N2 -- N1 is strictly dataset construction, and nothing here needs model
weights; see N1_CHANGELOG.md #4 for the full rationale.

IMPORTANT -- this sandbox's network allowlist does not include any real-data domain below, so
these download calls cannot be executed or tested here; exercised only in MOCK_MODE, which tests
every non-network code path (parsing, schema validation, manifest logging) against small
synthetic fixtures shaped like each real file's documented schema. Run with mock=False only on
Kaggle, which has open internet access. Full source-by-source endpoint history: N1_CHANGELOG.md.
"""
from __future__ import annotations
import gzip
import io
import shutil
import tarfile
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests

# Paths/CFG/Manifest/fingerprint helpers come from 00_config (same kernel, executed first).


DBPTM_TYPE_URLS: Dict[str, str] = {
    "N-linked Glycosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/N-linked Glycosylation.gz",
    "O-linked Glycosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/O-linked Glycosylation.gz",
    "Phosphorylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Phosphorylation.gz",
    "Deamidation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Deamidation.gz",
    "S-nitrosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/S-nitrosylation.gz",
    "S-palmitoylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/S-palmitoylation.gz",
    "Glutathionylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Glutathionylation.gz",
    "Sulfhydration": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Sulfhydration.gz",
    "Methylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Methylation.gz",
    "Citrullination": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Citrullination.gz",
    "ADP-ribosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/ADP-ribosylation.gz",
    "Nitration": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Nitration.gz",
    "Sulfation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Sulfation.gz",
    # [N1-CL#13] Category A additions -- existing K-stratum canonical types (via CPLM); dbPTM
    # becomes a SECOND source for each, feeding M3's cross-source dedup (source_count), not a
    # new config entry (type name already matches a cfg.ptm_type_to_stratum key exactly).
    "Acetylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Acetylation.gz",
    "Biotinylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Biotinylation.gz",
    "Butyrylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Butyrylation.gz",
    "Carboxyethylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Carboxyethylation.gz",
    "Carboxylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Carboxylation.gz",
    "Crotonylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Crotonylation.gz",
    "Formylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Formylation.gz",
    "Glutarylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Glutarylation.gz",
    "Lactylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Lactylation.gz",
    "Lipoylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Lipoylation.gz",
    "Malonylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Malonylation.gz",
    "Neddylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Neddylation.gz",
    "Propionylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Propionylation.gz",
    "Sumoylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Sumoylation.gz",
    "Succinylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Succinylation.gz",
    "Ubiquitination": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Ubiquitination.gz",
    "Hydroxylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Hydroxylation.gz",
    # Category A additions needing one new cfg.ptm_type_to_stratum entry each (existing stratum,
    # new canonical type name).
    "S-cysteinylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/S-cysteinylation.gz",
    "S-linked Glycosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/S-linked Glycosylation.gz",
    "Iodination": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Iodination.gz",
    # [N1-CL#14] Category B additions -- genuinely new strata (Glu/Met/Trp/Gln), see
    # N1_CHANGELOG.md for the residue verification behind each.
    "Gamma-carboxyglutamic acid": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Gamma-carboxyglutamic acid.gz",
    "Sulfoxidation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Sulfoxidation.gz",
    "C-linked Glycosylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/C-linked Glycosylation.gz",
    "Serotonylation": "https://biomics.lab.nycu.edu.tw/dbPTM/download/experiment/Serotonylation.gz",
}

CPLM_BASE = "https://cplm.biocuckoo.cn/Download/"
CPLM_SPECIES_HUMAN = "Homo sapiens"

UNIPROT_STREAM = "https://rest.uniprot.org/uniprotkb/stream"
UNIPROT_FEATURE_FIELDS = "accession,length,sequence,ft_carbohyd,ft_mod_res,ft_lipid,ft_signal,ft_transit"

def download_file(url: str, dest: Path, timeout: int = 120, chunk_size: int = 1 << 20,
                   max_retries: int = 4, backoff_base: float = 5.0) -> Path:
    """[N1-CL#57, found on a real local run] Streaming download with parent-dir creation,
    retry-with-backoff, and an ATOMIC write. Raises requests.HTTPError on non-2xx.

    Two real, compounding bugs this fixes, found from an actual local (non-Kaggle) run's
    traceback -- a UniProt reviewed-proteome stream (OOD species acquisition) hit
    `ChunkedEncodingError: Response ended prematurely` partway through a large TSV transfer,
    which the original version had no defence against at all:

    1. Zero retry logic -- a single dropped connection (transient, and apparently more common
       on a home connection than whatever Kaggle's network path enjoyed) crashed the entire M1
       cell outright, discarding six other sources' worth of already-completed acquisition work
       in the same cell. Now retries transient network errors (ConnectionError, Timeout,
       ChunkedEncodingError) up to `max_retries` times with exponential backoff, and re-raises
       immediately (no retry) for anything that means "this URL/request is wrong" rather than
       "the network hiccuped" -- HTTPError (4xx/5xx) is NOT retried, since retrying a 404 five
       times just wastes five timeouts before failing anyway.
    2. NON-atomic write -- chunks were streamed directly into `dest`. A crash mid-download left
       a truncated, partial file sitting exactly at the real destination path -- and
       `_resolve_raw`'s own cache check only tests `candidate.exists()`, never file
       completeness. Re-running the cell without manually deleting the partial file would have
       silently treated that truncated file as a valid cache hit and fed corrupted data into M2,
       with no error at all. Fixed by writing to a `.part` sibling file and renaming to `dest`
       ONLY after a fully successful download -- `dest` itself either doesn't exist or is
       complete; there is no truncated-but-present state anymore."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with open(tmp_dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
            tmp_dest.rename(dest)
            return dest
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            last_exc = e
            tmp_dest.unlink(missing_ok=True)
            if attempt == max_retries:
                break
            wait = backoff_base * (2 ** (attempt - 1))
            print(f"[download_file] attempt {attempt}/{max_retries} failed ({type(e).__name__}: "
                  f"{e}) for {url} -- retrying in {wait:.0f}s.")
            time.sleep(wait)
        except requests.exceptions.HTTPError:
            tmp_dest.unlink(missing_ok=True)
            raise  # non-2xx is a real, non-transient failure -- never retry this one
    tmp_dest.unlink(missing_ok=True)
    raise RuntimeError(
        f"[download_file] Giving up after {max_retries} attempts for {url} -- "
        f"last error: {type(last_exc).__name__}: {last_exc}") from last_exc


def _download_and_extract_zip_member(url: str, dest: Path, timeout: int = 600,
                                      chunk_size: int = 1 << 20) -> Path:
    """[N1-CL#5] New: downloads a ZIP, extracts its largest member to `dest` as raw bytes (no
    decode/re-encode -- decoding is deferred to whatever reads `dest` later, same policy as
    every other parser here). Same 'largest member = the real data file' heuristic as
    parse_cplm_zip, for the same reason (robust to an unknown exact inner filename). Streams the
    extracted member straight to disk rather than holding it in memory, since qPTM's real file
    is ~1.55GB uncompressed.

    [N1-CL#57] The ZIP download itself now goes through download_file() -- retry-with-backoff
    plus an atomic write -- instead of its own unguarded requests.get(...) loop. This is the
    single largest download in the whole pipeline (~1.55GB uncompressed; the zip itself is
    smaller but still the biggest transfer here), making it the most likely place to hit the
    same "ChunkedEncodingError: Response ended prematurely" a real local run already hit on a
    much smaller UniProt TSV. Only the zip-extraction step below (zipfile-local, no network)
    keeps its own ad hoc `.ziptmp` handling -- that part was never the source of the risk."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    zip_path = dest.with_suffix(dest.suffix + ".ziptmp")
    download_file(url, zip_path, timeout=timeout, chunk_size=chunk_size)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        if not members:
            raise ValueError(f"_download_and_extract_zip_member: {url} zip has no file members")
        member = max(members, key=lambda m: m.file_size)
        with zf.open(member) as src, open(dest, "wb") as out:
            shutil.copyfileobj(src, out, length=chunk_size)
    zip_path.unlink(missing_ok=True)
    return dest

# ---------------------------------------------------------------------------
def _resolve_raw(rel_path: str) -> Optional[Path]:
    """rel_path is relative to raw_dir. Checked in Paths.RAW first (same-session cache), then
    by filename anywhere under an attached input dataset."""
    filename = Path(rel_path).name
    candidate = Paths.RAW / rel_path
    if candidate.exists():
        return candidate
    return find_in_inputs(filename)


def acquire_uniprot_reviewed(raw_dir: Path, manifest: Manifest, organism_id: str, organism_label: str,
                             dest_name: str, mock: bool = False, force: bool = False,
                             mock_writer=None) -> Path:
    """Generalized so the same reviewed-proteome + PTM-feature-fields query can be reused for
    other organisms without a second hard-coded query string."""
    dest = raw_dir / dest_name
    query = f"(reviewed:true) AND (organism_id:{organism_id}) AND (length:[1 TO 1022])"
    url = (f"{UNIPROT_STREAM}?query={requests.utils.quote(query)}"
          f"&fields={requests.utils.quote(UNIPROT_FEATURE_FIELDS)}&format=tsv")
    cached = None if force else _resolve_raw(dest_name)
    if cached is not None:
        manifest.log(f"UniProt/Swiss-Prot ({organism_label})", url, cached, "CACHED (reused, not re-downloaded)")
        return cached
    if mock:
        (mock_writer or _write_mock_swissprot)(dest)
    else:
        download_file(url, dest)
    manifest.log(f"UniProt/Swiss-Prot ({organism_label})", url, dest,
                 f"Reviewed {organism_label}, length<=1022, TSV with ft_carbohyd/ft_mod_res/ft_lipid/"
                 f"ft_signal/ft_transit")
    return dest


def acquire_uniprot_swissprot(raw_dir: Path, manifest: Manifest, mock: bool = False, force: bool = False) -> Path:
    return acquire_uniprot_reviewed(raw_dir, manifest, organism_id="9606", organism_label="human",
                                    dest_name="swissprot_human_reviewed.tsv", mock=mock, force=force,
                                    mock_writer=_write_mock_swissprot)


def acquire_cplm(raw_dir: Path, manifest: Manifest, species: List[str], mock: bool = False,
                  force: bool = False) -> Dict[str, Path]:
    """Downloads CPLM per-species zips (Section 1 -- Lysine Modification Data -- only, never
    Section 2/Annotations)."""
    out = {}
    for sp in species:
        rel = f"cplm/{sp.replace('/', '_')}.zip"
        dest = raw_dir / rel
        url = CPLM_BASE + requests.utils.quote(f"{sp}.zip")
        cached = None if force else _resolve_raw(rel)
        if cached is not None:
            manifest.log(f"CPLM 4.0 [{sp}]", url, cached, "CACHED (reused, not re-downloaded)")
            out[sp] = cached
            continue
        if mock:
            _write_mock_cplm_zip(dest, n_sites=200)
        else:
            download_file(url, dest)
        manifest.log(f"CPLM 4.0 [{sp}]", url, dest, "Section 1 (Lysine Modification Data) only")
        out[sp] = dest
    return out


# [N1-CL#6] Real qPTM bulk-download link -- replaces the gated request form this used to
# require. [N1-CL#55] Moved OUT of source: read from the QPTM_DIRECT_URL environment variable
# instead of a hard-coded literal, now that this notebook runs on a persistent personal machine
# rather than a fresh per-session Kaggle container -- a hard-coded access code would otherwise
# sit in the notebook (and any git history / backup of it) indefinitely. Set it once per machine,
# e.g. in a .env file loaded before this cell runs, or `export QPTM_DIRECT_URL=...` in the shell.
# Re-request from qptm.omicsbio.info if it ever needs rotating.
import os as _os
QPTM_FORM_URL = "https://qptm.omicsbio.info/download.php"
QPTM_DIRECT_URL: Optional[str] = _os.environ.get("QPTM_DIRECT_URL")
QPTM_NOTE = (
    "QPTM_DIRECT_URL is unset -- qptm.omicsbio.info/download.php is a gated request form, not a "
    "static file endpoint. Submit that form once by hand to get a real link, then set it as the "
    "QPTM_DIRECT_URL environment variable (not in this file). "
    "acquire_qptm() raises rather than silently downloading the form page."
)


def acquire_qptm(raw_dir: Path, manifest: Manifest, mock: bool = False, force: bool = False) -> Path:
    """CF-26 (required): the independent-curation replication label set -- Human/Mouse/Rat/Yeast
    site-level PTMs in one table. [N1-CL#7] Real endpoint confirmed: a ZIP containing one large
    TSV ('qptm_all_data', ~1.55GB uncompressed); extracted via _download_and_extract_zip_member
    so downstream parsing reads a plain flat file like every other source, never the zip."""
    rel = "qptm/qptm_all_data.tsv"
    dest = raw_dir / rel
    cached = None if force else _resolve_raw(rel)
    if cached is not None:
        manifest.log("qPTM (CF-26 replication set)", QPTM_DIRECT_URL or QPTM_FORM_URL, cached,
                     "CACHED (reused, not re-downloaded)")
        return cached
    if mock:
        _write_mock_qptm(dest, n_sites=180)
    else:
        if not QPTM_DIRECT_URL:
            raise RuntimeError(QPTM_NOTE)
        _download_and_extract_zip_member(QPTM_DIRECT_URL, dest)
    manifest.log("qPTM (CF-26 replication set)", QPTM_DIRECT_URL or QPTM_FORM_URL, dest,
                 "Human/Mouse/Rat/Yeast; species filter to human applied downstream (M3)")
    return dest


def parse_qptm_flatfile(path: Path) -> pd.DataFrame:
    """qPTM site table -> (accession, position, type, species, pmid). Filtered to human
    downstream by M3's build_qptm_labels (qPTM's raw table spans Human/Mouse/Rat/Yeast).
    [N1-CL#8] Column matcher fixed for the real confirmed header (Organism/PMID/UniProt
    accession/Gene name/Position/PTM/Sequence window/Raw peptide/Sample/Condition/Log2Ratio.../
    Reliability/fdr...) -- the 'type' matcher previously only recognised 'modification'/'type'
    and would have missed the real 'PTM' column entirely."""
    raw = _read_flatfile_tsv(path)
    col_map = {
        "accession": _match_column(list(raw.columns), ["uniprot", "accession", "protein", "protein id"]),
        "position": _match_column(list(raw.columns), ["position", "site"]),
        "type": _match_column(list(raw.columns), ["modification", "ptm type", "modification type", "ptm", "type"]),
        "species": _match_column(list(raw.columns), ["species", "organism"]),
        "pmid": _match_column(list(raw.columns), ["pmid", "reference"]),
        # [N1-CL#22] qPTM's own 0-5 confidence score (how many independent datasets saw the
        # site). Carried through so M3 can apply a floor -- previously dropped, which left qPTM
        # the ONLY source with no quality filter at all (it has no `evidence` column, so
        # step1_evidence_filter is a no-op on it).
        "reliability": _match_column(list(raw.columns), ["reliability"]),
    }
    missing = [k for k, v in col_map.items() if v is None and k not in ("pmid", "reliability")]
    if missing:
        raise ValueError(f"parse_qptm_flatfile: missing columns {missing} in {path.name}: {list(raw.columns)}")
    out = pd.DataFrame({
        "accession": raw[col_map["accession"]].astype(str),
        "position": pd.to_numeric(raw[col_map["position"]], errors="coerce").astype("Int64"),
        "type": raw[col_map["type"]].astype(str),
        "species": raw[col_map["species"]].astype(str),
        "pmid": raw[col_map["pmid"]].astype(str) if col_map["pmid"] else pd.NA,
        "reliability": (pd.to_numeric(raw[col_map["reliability"]], errors="coerce").astype("Int64")
                        if col_map["reliability"] else pd.NA),
    })
    return out.dropna(subset=["accession", "position"])


def acquire_dbptm(raw_dir: Path, manifest: Manifest, mock: bool = False, force: bool = False) -> Dict[str, Path]:
    if not mock and not DBPTM_TYPE_URLS:
        raise RuntimeError("DBPTM_TYPE_URLS is empty -- see N1_CHANGELOG.md for the confirmed endpoint list.")
    out = {}
    urls = DBPTM_TYPE_URLS if (not mock or DBPTM_TYPE_URLS) else {
        "N-linked Glycosylation": "MOCK", "Succinylation": "MOCK", "O-linked Glycosylation": "MOCK",
    }
    for ptm_type, url in urls.items():
        rel = f"dbptm/{ptm_type.replace(' ', '_')}.txt.gz"  # real bytes are gzip/tar, not literal .txt -- parsing sniffs content
        dest = raw_dir / rel
        cached = None if force else _resolve_raw(rel)
        if cached is not None:
            manifest.log(f"dbPTM 2025 [{ptm_type}]", url, cached, "CACHED (reused, not re-downloaded)")
            out[ptm_type] = cached
            continue
        if mock:
            _write_mock_dbptm_flatfile(dest, ptm_type, n_sites=150)
        else:
            download_file(url, dest)
        manifest.log(f"dbPTM 2025 [{ptm_type}]", url, dest, "Experimentally validated only")
        out[ptm_type] = dest
    return out


OGLCNAC_URLS: Dict[str, str] = {
    "Dataset-I": "https://oglcnac.org/static/dataset/Atlas%205.0_unambiguous%20sites_20251208.csv?v=20260523",
    "Dataset-II": "https://oglcnac.org/static/dataset/Atlas%205.0_ambiguous%20sites_20251208.csv?v=20260523",
}


def acquire_oglcnac_atlas(raw_dir: Path, manifest: Manifest, mock: bool = False,
                           force: bool = False) -> Dict[str, Path]:
    """Dataset-I = unambiguous sites (feeds positives); Dataset-II = ambiguous sites, routed to
    M3's ambiguity mask purely by the raw_sources dict key containing 'oglcnac_dataset_ii'."""
    out = {}
    for dataset in ["Dataset-I", "Dataset-II"]:
        rel = f"oglcnac_atlas/{dataset}.csv"
        dest = raw_dir / rel
        url = OGLCNAC_URLS[dataset]
        cached = None if force else _resolve_raw(rel)
        if cached is not None:
            manifest.log(f"O-GlcNAcAtlas 5.0 [{dataset}]", url, cached, "CACHED (reused, not re-downloaded)")
            out[dataset] = cached
            continue
        if mock:
            _write_mock_oglcnac(dest, dataset, n_sites=100)
        else:
            download_file(url, dest)
        manifest.log(f"O-GlcNAcAtlas 5.0 [{dataset}]", url, dest,
                     "Dataset-I=unambiguous, Dataset-II=ambiguous (drives the ambiguity mask, M3 step 3a)")
        out[dataset] = dest
    return out


STACKGLYEMBED_BASE = "https://raw.githubusercontent.com/nafcoder/StackGlyEmbed/main/Dataset/"
GLYCOSITE_ATLAS_URL = "http://nglycositeatlas.biomarkercenter.org/download/HumanAll/"


def acquire_glycosite_atlas(raw_dir: Path, manifest: Manifest, mock: bool = False,
                            force: bool = False) -> Path:
    """Supplementary N-glycosylation evidence (unioned with dbPTM's own 'N-linked
    Glycosylation' entries), fetched directly from the database's own site as XLSX."""
    rel = "n_glycosite_atlas/HumanAll.xlsx"
    dest = raw_dir / rel
    cached = None if force else _resolve_raw(rel)
    if cached is not None:
        manifest.log("N-GlycositeAtlas", GLYCOSITE_ATLAS_URL, cached, "CACHED (reused, not re-downloaded)")
        return cached
    if mock:
        _write_mock_glycosite_atlas_xlsx(dest, n_sites=120)
    else:
        download_file(GLYCOSITE_ATLAS_URL, dest)
    manifest.log("N-GlycositeAtlas", GLYCOSITE_ATLAS_URL, dest,
                 "XLSX, confirmed schema: Database Accession (UniProtKB)/Glycosylation location/...")
    return dest


def acquire_glyde(raw_dir: Path, manifest: Manifest, mock: bool = False, force: bool = False) -> Dict[str, Path]:
    """The only Gold-tier negative source (via build_gold_negatives_nglyde). Sourced from
    StackGlyEmbed's GitHub reproduction of the N-GlyDE benchmark, since the live N-GlyDE web app
    is reported down. Uses the '_original' identity-threshold variant (matching the 2019 paper's
    own benchmark), not StackGlyEmbed's own newer 40%/60% variants."""
    out = {}
    for split in ["train", "test"]:
        rel = f"n_glyde/fasta_{split}_NGlyDE_original.txt"
        dest = raw_dir / rel
        url = STACKGLYEMBED_BASE + f"fasta_{split}_NGlyDE_original.txt"
        cached = None if force else _resolve_raw(rel)
        if cached is not None:
            manifest.log(f"N-GlyDE [{split}, via StackGlyEmbed repo]", url, cached,
                         "CACHED (reused, not re-downloaded)")
            out[split] = cached
            continue
        if mock:
            _write_mock_stackglyembed_fasta_pairs(dest, n_proteins=15, seed=5 if split == "train" else 50)
        else:
            download_file(url, dest)
        manifest.log(f"N-GlyDE [{split}, via StackGlyEmbed repo]", url, dest,
                     "Gold-tier positives+negatives -- see build_gold_negatives_nglyde")
        out[split] = dest
    return out


# ---------------------------------------------------------------------------
# Parsers -- each source's raw file(s) -> a normalized (accession, position, type, ...) table.
# ---------------------------------------------------------------------------
def _match_column(columns: List[str], candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower().strip(): c for c in columns}
    for cand in candidates:
        for lc, orig in cols_lower.items():
            if cand in lc:
                return orig
    return None


def _decode_bytes_robust(data: bytes) -> str:
    """UTF-8, falling back to cp1252 then latin-1 for files that aren't UTF-8 (a real
    O-GlcNAcAtlas download needed this -- lab-exported CSVs commonly carry stray cp1252
    bytes in free-text columns). latin-1 is the final fallback since it cannot raise
    UnicodeDecodeError (every byte 0x00-0xFF maps to a code point)."""
    for enc in ("utf-8", "cp1252"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    return data.decode("latin-1")


def _read_csv_robust(path_or_bytes, **kwargs) -> pd.DataFrame:
    """pd.read_csv with the encoding fallback above. Accepts a Path or already-read bytes
    (e.g. a zip/tar member), so every parser here shares one encoding policy."""
    if isinstance(path_or_bytes, (bytes, bytearray)):
        data = bytes(path_or_bytes)
    else:
        with open(path_or_bytes, "rb") as f:
            data = f.read()
    return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


def _read_flatfile_tsv(path: Path, **kwargs) -> pd.DataFrame:
    """Reads a downloaded flat file as TSV regardless of what its name/extension claims.
    dbPTM's per-type '.gz' links are actually gzip-compressed TAR archives (PAX format, the
    default 'tar czf x.gz onefile.txt' output), not bare gzip -- content is sniffed (tarfile's
    own signature check, then raw gzip magic bytes) rather than trusted from the filename, so
    this is correct for real downloads, for files fetched under an earlier differently-shaped
    version of this pipeline, and for MOCK_MODE's plain-text fixtures, with one code path."""
    try:
        with tarfile.open(path, mode="r:*") as tf:
            members = [m for m in tf.getmembers() if m.isfile()]
            if not members:
                raise ValueError(f"_read_flatfile_tsv: tar archive at {path.name} has no regular file members")
            member = max(members, key=lambda m: m.size)  # dbPTM ships exactly one data file per archive
            with tf.extractfile(member) as f:
                data = f.read()
            return _read_csv_robust(data, sep="\t", **kwargs)
    except tarfile.ReadError:
        pass  # not a tar archive -- fall through to gzip/plain-text handling

    with open(path, "rb") as f:
        magic = f.read(2)
    if magic == b"\x1f\x8b":
        with gzip.open(path, "rb") as f:
            data = f.read()
        return _read_csv_robust(data, sep="\t", **kwargs)
    return _read_csv_robust(path, sep="\t", **kwargs)


CPLM_POSITIONAL_COLUMNS = ["cplm_id", "accession", "position", "type", "gene", "species",
                          "sequence", "evidence_type", "pmid"]


def parse_cplm_zip(zip_path: Path) -> pd.DataFrame:
    """CPLM's real per-species zip has no header row (confirmed) -- read positionally against
    CPLM_POSITIONAL_COLUMNS rather than name-matching text that doesn't exist in the file."""
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        data_member = max(members, key=lambda m: m.file_size)  # one primary data file per species zip
        with zf.open(data_member) as f:
            raw = _read_csv_robust(f.read(), sep="\t", header=None, names=CPLM_POSITIONAL_COLUMNS)

    out = pd.DataFrame({
        "accession": raw["accession"].astype(str),
        "position": pd.to_numeric(raw["position"], errors="coerce").astype("Int64"),
        "type": raw["type"].astype(str),
        "pmid": raw["pmid"].astype(str),
        "flanking_peptide": pd.NA, # not column 6 ("sequence") -- see CPLM_POSITIONAL_COLUMNS
    })
    out = out.dropna(subset=["accession", "position", "type"])
    return out


DBPTM_POSITIONAL_COLUMNS = ["entry_name", "accession", "position", "type", "pmid", "flanking_peptide"]


def parse_dbptm_flatfile(path: Path, ptm_type: str) -> pd.DataFrame:
    """dbPTM per-type experimental flat file -> (accession, position, type, pmid, evidence).
    No header row -- read positionally. `evidence` is hard-coded "experimental" since every row
    in this file is already dbPTM's own experimentally-validated tier by construction (this is
    the site's dedicated /download/experiment/ endpoint)."""
    raw = _read_flatfile_tsv(path, header=None, names=DBPTM_POSITIONAL_COLUMNS)
    entry_name = raw["entry_name"].astype(str)
    out = pd.DataFrame({
        "accession": raw["accession"].astype(str),
        "position": pd.to_numeric(raw["position"], errors="coerce").astype("Int64"),
        "type": ptm_type,
        "pmid": raw["pmid"].astype(str),
        "evidence": "experimental",
        # [N1-CL#25] entry_name's suffix is dbPTM's ONLY species signal (the per-type files are
        # not species-filtered at source). Previously discarded; kept now so OOD corpora can be
        # built for mouse/rat/yeast from the same download.
        "species_suffix": entry_name.str.rsplit("_", n=1).str[-1],
    })
    return out.dropna(subset=["accession", "position"])


def parse_oglcnac_atlas_flatfile(path: Path, species: str = "human") -> pd.DataFrame:
    """O-GlcNAcAtlas 5.0 Dataset-I/II CSV -> (accession, position, type, evidence). Covers
    multiple species and both UniProt/NCBI accessions in one file; filtered here to
    species=="human" AND accession_source=="UniProt" AND a non-empty position_in_protein (not
    position_in_peptide, which is within-peptide, not within-protein -- an easy silent
    off-by-large-number error if swapped)."""
    raw = _read_csv_robust(path)  # comma-separated, not tab
    # [N1-CL#35] `species` parameterised (was hard-coded "human") so the same parser serves the
    # OOD species. Dataset-I spans 29 species; accession_source spans 8 registries, so the
    # UniProt filter is what keeps accessions joinable to a UniProt-derived corpus.
    human = raw[(raw["species"].astype(str).str.strip().str.lower() == str(species).strip().lower())
               & (raw["accession_source"].astype(str).str.strip().str.lower() == "uniprot")].copy()
    human["position_in_protein"] = pd.to_numeric(human["position_in_protein"], errors="coerce")
    human = human.dropna(subset=["accession", "position_in_protein"])
    out = pd.DataFrame({
        "accession": human["accession"].astype(str),
        "position": human["position_in_protein"].astype(int),
        "type": "O-GlcNAcylation",
        "evidence": "MS/MS",
    })
    return out.dropna(subset=["accession", "position"])


def parse_stackglyembed_fasta_pairs(path: Path) -> pd.DataFrame:
    """Two lines per protein: an odd header 'accession,label,position[,label,position,...]'
    (1=glycosylated, 0=validated non-glycosylated sequon), even line = raw sequence (unused --
    this pipeline gets sequences from the UniProt-derived corpus, matched by accession)."""
    lines = _decode_bytes_robust(path.read_bytes()).strip().split("\n")
    rows = []
    for i in range(0, len(lines) - 1, 2):
        header = lines[i].strip()
        parts = header.split(",")
        if not parts or not parts[0]:
            continue
        accession, pairs = parts[0], parts[1:]
        for j in range(0, len(pairs) - 1, 2):
            label_str, pos_str = pairs[j].strip(), pairs[j + 1].strip()
            if not pos_str.lstrip("-").isdigit():
                continue
            rows.append({"UniProt Accession": accession, "Position": int(pos_str),
                        "Label": "glycosylated" if label_str == "1" else "non-glycosylated"})
    return pd.DataFrame(rows, columns=["UniProt Accession", "Position", "Label"])


def _find_excel_header_row(raw_no_header: pd.DataFrame, must_contain: List[str],
                            max_scan_rows: int = 10) -> int:
    """Scans the first rows for the one containing all of `must_contain` (case-insensitive
    substrings), since journal supplementary XLSX exports commonly ship a title row before the
    real header row and a plain header=0 read would misparse the title as column names."""
    n_scan = min(max_scan_rows, len(raw_no_header))
    for row_idx in range(n_scan):
        row_text = " | ".join(str(v).strip().lower() for v in raw_no_header.iloc[row_idx].tolist())
        if all(token in row_text for token in must_contain):
            return row_idx
    raise ValueError(f"_find_excel_header_row: no row in the first {n_scan} rows contains all of "
                     f"{must_contain}; first {n_scan} rows were: "
                     f"{raw_no_header.head(n_scan).values.tolist()}")


def parse_glycosite_atlas_flatfile(path: Path) -> pd.DataFrame:
    """N-GlycositeAtlas's HumanAll.xlsx -> (accession, position, type, evidence). Real header
    row located dynamically (see _find_excel_header_row) rather than assumed at a fixed offset,
    since the sheet has a title row above the real column-name row."""
    raw_no_header = pd.read_excel(path, header=None)
    header_row = _find_excel_header_row(raw_no_header, must_contain=["uniprotkb", "glycosylation location"])
    raw = pd.read_excel(path, header=header_row)
    col_map = {
        "accession": _match_column(list(raw.columns), ["uniprotkb", "accession", "database accession"]),
        "position": _match_column(list(raw.columns), ["glycosylation location", "position", "location"]),
    }
    missing = [k for k, v in col_map.items() if v is None]
    if missing:
        raise ValueError(f"parse_glycosite_atlas_flatfile: missing columns {missing} in {path.name}: "
                         f"{list(raw.columns)}")
    out = pd.DataFrame({
        "accession": raw[col_map["accession"]].astype(str),
        "position": pd.to_numeric(raw[col_map["position"]], errors="coerce").astype("Int64"),
        "type": "N-glycosylation",
        "evidence": "MS/MS",
    })
    return out.dropna(subset=["accession", "position"])


def parse_glyde_flatfile(path: Path) -> pd.DataFrame:
    """N-GlyDE (via StackGlyEmbed) -> (UniProt Accession, Position, Label). Feeds the Gold-tier
    NEGATIVE set only (Label containing "non"); positives carried through too since M3 filters."""
    return parse_stackglyembed_fasta_pairs(path)


# ---------------------------------------------------------------------------
# Mock fixtures -- exercise every parser above with no network access. Shapes mirror the
# documented real schemas.
# ---------------------------------------------------------------------------
_AA20 = "ACDEFGHIKLMNPQRSTVWY"

def _random_seq(rng, length):
    return "".join(rng.choice(list(_AA20)) for _ in range(length))


def _write_mock_swissprot(dest: Path, n_proteins: int = 200, seed: int = 1):
    import random
    rng = random.Random(seed)
    rows = []
    for i in range(n_proteins):
        length = rng.randint(50, 800)
        rows.append({"Entry": f"P{10000 + i}", "Length": length, "Sequence": _random_seq(rng, length)})
    dest.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(dest, sep="\t", index=False)


def _write_mock_cplm_zip(dest: Path, n_sites: int = 200, seed: int = 2):
    import random
    rng = random.Random(seed)
    types = ["Succinylation", "Acetylation", "Ubiquitination", "Hydroxylation", "β-Hydroxybutyrylation"]
    lines = []
    for i in range(n_sites):
        lines.append("\t".join([
            str(i), f"P{10000 + rng.randint(0, 199)}", str(rng.randint(1, 500)),
            rng.choice(types), f"GENE{i}", "Homo sapiens", _random_seq(rng, 13),
            "MS/MS", str(20000000 + i),
        ]))
    dest.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(dest, "w") as zf:
        zf.writestr("Homo_sapiens.txt", "\n".join(lines))


def _write_mock_dbptm_flatfile(dest: Path, ptm_type: str, n_sites: int = 150, seed: int = 3):
    import random
    rng = random.Random(seed + hash(ptm_type) % 1000)
    lines = []
    for i in range(n_sites):
        lines.append("\t".join([
            f"MOCK{i}_HUMAN", f"P{20000 + rng.randint(0, 149)}", str(rng.randint(1, 500)),
            ptm_type, str(30000000 + i), _random_seq(rng, 13),
        ]))
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text("\n".join(lines))


def _write_mock_oglcnac(dest: Path, dataset: str, n_sites: int = 100, seed: int = 4):
    import random
    rng = random.Random(seed)
    rows = []
    for i in range(n_sites):
        # [N1-CL#19] Row 3's comment deliberately contains a right-single-quote character that
        # isn't valid UTF-8 once the whole file is written as cp1252 below -- reproduces the
        # exact decode failure a real O-GlcNAcAtlas download hit (see N1_CHANGELOG.md #19),
        # so a mock-mode test run now exercises the same failure mode instead of only clean ASCII.
        comment = "authors\u2019 note" if i == 3 else ""
        rows.append({
            "id": i, "species": "human", "sample_type": "HeLa", "accession": f"P{30000 + (i % 40)}",
            "accession_source": "UniProt", "entry_name": f"MOCK{i}_HUMAN", "protein_name": "Mock protein",
            "gene_name": f"MOCK{i}", "peptide_seq": _random_seq(rng, 15), "site_residue": "S",
            "position_in_peptide": rng.randint(1, 15), "position_in_protein": rng.randint(1, 500),
            "method": "MS", "analytical_throughput": "HTP", "pmid": 20000000 + i,
            "condition": "mock/control", "log2Ratio": round(rng.uniform(-2, 2), 3), "p_value": "",
            "comments": comment,
        })
    dest.parent.mkdir(parents=True, exist_ok=True)
    # Written as cp1252, not UTF-8 -- a real lab-exported CSV with stray Windows encoding, not a
    # pure-ASCII fixture, is what actually exposed this bug; matching that here is the point.
    dest.write_bytes(pd.DataFrame(rows).to_csv(index=False).encode("cp1252"))


def _write_mock_qptm(dest: Path, n_sites: int = 180, seed: int = 6):
    """[N1-CL#9] Rewritten to match the real confirmed header (Organism/PMID/UniProt
    accession/Gene name/Position/PTM/Sequence window/Raw peptide/Sample/Condition/Log2Ratio
    (peptide)/P value (peptide)/Log2Ratio (protein)/P value (protein)/Reliability/fdr (peptide)),
    including all three real species so M_E's organism breakdown has something to show even in
    mock mode."""
    import random
    rng = random.Random(seed)
    organisms = ["Human"] * int(n_sites * 0.8) + ["Mouse"] * int(n_sites * 0.15) + ["Yeast"] * int(n_sites * 0.05)
    while len(organisms) < n_sites:
        organisms.append("Human")
    rng.shuffle(organisms)
    ptm_types = ["Phosphorylation", "Acetylation", "Ubiquitination", "Succinylation"]
    rows = []
    for i, org in enumerate(organisms):
        rows.append({
            "Organism": org, "PMID": 20000000 + i, "UniProt accession": f"P{40000 + (i % 60)}",
            "Gene name": f"MOCK{i % 60}", "Position": rng.randint(1, 500),
            "PTM": rng.choice(ptm_types), "Sequence window": _random_seq(rng, 15),
            "Raw peptide": _random_seq(rng, 12), "Sample": "MockCellLine",
            "Condition": "Treatment/Ctr", "Log2Ratio (peptide)": round(rng.uniform(-3, 3), 3),
            "P value (peptide)": round(rng.random(), 4), "Log2Ratio (protein)": "-",
            "P value (protein)": "-", "Reliability": rng.choice([2, 5]),
            "fdr (peptide)": round(rng.random() * 0.01, 6),
        })
    dest.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(dest, sep="\t", index=False)


def _write_mock_stackglyembed_fasta_pairs(dest: Path, n_proteins: int = 15, seed: int = 4):
    """Positions land on real 'N' residues so the mock data is chemistry-valid like the real
    data (every glycosylation site sits on an asparagine)."""
    import random
    rng = random.Random(seed)
    lines = []
    for i in range(n_proteins):
        length = rng.randint(60, 200)
        seq = list(_random_seq(rng, length))
        n_positions = rng.sample(range(1, length + 1), k=min(4, length))
        for p in n_positions:
            seq[p - 1] = "N"
        seq_str = "".join(seq)
        pairs = []
        for p in n_positions:
            label = 1 if rng.random() > 0.4 else 0
            pairs.extend([str(label), str(p)])
        lines.append(f"P{70000 + i}," + ",".join(pairs))
        lines.append(seq_str)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text("\n".join(lines) + "\n")


def _write_mock_glycosite_atlas_xlsx(dest: Path, n_sites: int = 120, seed: int = 4):
    import random
    rng = random.Random(seed)
    rows = []
    for i in range(n_sites):
        peptide = list(_random_seq(rng, 14))
        site_idx = rng.randrange(len(peptide))
        peptide[site_idx] = "n"  # lowercase marks the modified residue, matching the real format
        rows.append({
            "Database Accession (UniProtKB)": f"P{50000 + (i % 40)}", "Gene names": f"MOCK{i % 40}",
            "Protein names": "Mock protein", "Glycosylation location": rng.randint(1, 500),
            "Glycosite-containing peptides": "".join(peptide), "Glycosite_20AAs": _random_seq(rng, 20),
            "Identification resources": rng.choice(["Liver", "Plasma", "Kidney"]),
            "Years": rng.randint(2010, 2022), "References": rng.randint(1, 100), "Reviewed": "",
        })
    dest.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_excel(dest, index=False)


def acquire_ood_proteomes(raw_dir: Path, manifest: Manifest, ood_species: dict,
                           mock: bool = False, force: bool = False) -> Dict[str, Path]:
    """[N1-CL#25] Reviewed proteome per OOD species. Uses the same generalized
    acquire_uniprot_reviewed the human corpus uses -- the reason that function was written
    organism-parameterised rather than hard-coding the human query."""
    out = {}
    for species, meta in ood_species.items():
        out[species] = acquire_uniprot_reviewed(
            raw_dir, manifest, organism_id=meta["taxon_id"], organism_label=species,
            dest_name=f"swissprot_{species}_reviewed.tsv", mock=mock, force=force,
            mock_writer=_write_mock_swissprot)
    return out


def acquire_cplm_ood(raw_dir: Path, manifest: Manifest, ood_species: dict, mock: bool = False,
                      force: bool = False) -> Dict[str, Path]:
    """[N1-CL#35] CPLM per-species zips for the OOD species. The `cplm_species` strings come from
    the ORIGINAL pipeline's own CPLM_SPECIES_OOD list, which documented that all four have
    prepared CPLM files -- and its V12a/CF-31 E. coli replication called
    acquire_cplm(species=["Escherichia coli (strain K12)"]) in working code. These names are
    verified, strain-qualified, and must not be simplified.

    A failure is reported but not fatal: dbPTM covers the same strata by species suffix, so a
    missing CPLM species degrades OOD lysine coverage rather than blocking the run."""
    out = {}
    for species, meta in ood_species.items():
        sp_name = meta.get("cplm_species")
        if not sp_name:
            continue
        try:
            got = acquire_cplm(raw_dir, manifest, species=[sp_name], mock=mock, force=force)
            out[species] = got[sp_name]
        except Exception as e:
            print(f"[M1] CPLM {species} ({sp_name}) unavailable -- skipped, not fatal: "
                  f"{type(e).__name__}: {e}")
    return out


# ---------------------------------------------------------------------------
# [N1-CL#12] New: raw download size reporting.
# ---------------------------------------------------------------------------
# [N1-CL#41] ood_proteomes/ood_cplm added -- the first real run reported a 1,634 MB total
# that silently excluded eight OOD downloads that had in fact been fetched.
RAW_DOWNLOAD_SOURCES = ["swissprot", "cplm", "dbptm", "qptm", "oglcnac_atlas",
                        "glycosite_atlas", "glyde", "ood_proteomes", "ood_cplm"]


def report_raw_download_sizes(resolved: Dict[str, object]) -> pd.DataFrame:
    """Per-source and total raw download size, read directly off disk (Path.stat()) so it's
    accurate whether a source was freshly downloaded this session or reused from cache.
    Excludes cplm_compact (derived, not downloaded) and anything model/SAE-related (not
    acquired here -- see N1-CL#4)."""
    rows = []
    for key in RAW_DOWNLOAD_SOURCES:
        val = resolved.get(key)
        if val is None:
            rows.append({"source": key, "n_files": 0, "bytes": 0})
            continue
        if isinstance(val, dict):
            rows.append({"source": key, "n_files": len(val),
                        "bytes": sum(path_size_bytes(p) for p in val.values())})
        else:
            rows.append({"source": key, "n_files": 1, "bytes": path_size_bytes(val)})
    df = pd.DataFrame(rows)
    total_bytes = int(df["bytes"].sum())

    print(f"\n{'source':<20}{'n_files':>10}{'size':>16}")
    for _, r in df.iterrows():
        note = " (not acquired)" if r["n_files"] == 0 else ""
        print(f"{r['source']:<20}{int(r['n_files']):>10}{human_mb(r['bytes']):>16}{note}")
    print(f"{'-' * 46}")
    print(f"{'TOTAL raw downloaded':<20}{'':>10}{human_mb(total_bytes):>16}")
    df["size"] = df["bytes"].map(human_mb)
    return df


# ---------------------------------------------------------------------------
# Orchestrator
# ---------------------------------------------------------------------------
def run_m1(paths, mock: bool = False, force: bool = False, ood_species: Optional[dict] = None
           ) -> Tuple[Manifest, Dict[str, object]]:
    """Returns (manifest, resolved_paths) -- resolved_paths is the source of truth for where
    each artefact actually ended up (Paths.RAW for a fresh build, or directly inside an attached
    input dataset for a cache hit, never copied). Downstream code MUST read paths from this
    dict, never hard-code Paths.RAW / "some_file"."""
    from tqdm.auto import tqdm
    manifest = Manifest()
    raw = paths.RAW
    print(f"[M1] mock={mock} force={force} raw_dir={raw}")

    pbar = tqdm(total=7, desc="[M1] Acquisition", unit="source")
    resolved: Dict[str, object] = {}
    pbar.set_postfix_str("swissprot"); resolved["swissprot"] = acquire_uniprot_swissprot(raw, manifest, mock=mock, force=force); pbar.update(1)
    pbar.set_postfix_str("cplm"); resolved["cplm"] = acquire_cplm(raw, manifest, species=[CPLM_SPECIES_HUMAN], mock=mock, force=force); pbar.update(1)
    pbar.set_postfix_str("dbptm"); resolved["dbptm"] = acquire_dbptm(raw, manifest, mock=mock, force=force); pbar.update(1)
    pbar.set_postfix_str("qptm")
    try:
        resolved["qptm"] = acquire_qptm(raw, manifest, mock=mock, force=force)  # CF-26: required, not optional
    except RuntimeError as e:
        if not mock:
            print(f"[M1] HELD: qPTM acquisition skipped ({e}). Re-run M1 once QPTM_DIRECT_URL is set.")
        else:
            raise
    pbar.update(1)
    pbar.set_postfix_str("oglcnac_atlas"); resolved["oglcnac_atlas"] = acquire_oglcnac_atlas(raw, manifest, mock=mock, force=force); pbar.update(1)
    pbar.set_postfix_str("glycosite_atlas"); resolved["glycosite_atlas"] = acquire_glycosite_atlas(raw, manifest, mock=mock, force=force); pbar.update(1)
    pbar.set_postfix_str("glyde"); resolved["glyde"] = acquire_glyde(raw, manifest, mock=mock, force=force); pbar.update(1)
    pbar.close()

    if ood_species:
        print(f"[M1] OOD species: {list(ood_species)}")
        # [N1-CL#35] The human corpus's own CPLM download stays "Homo sapiens" only; these are
        # SEPARATE per-species files for the OOD corpora, using the original's verified names.
        resolved["ood_proteomes"] = acquire_ood_proteomes(raw, manifest, ood_species, mock=mock, force=force)
        resolved["ood_cplm"] = acquire_cplm_ood(raw, manifest, ood_species, mock=mock, force=force)

    cplm_paths = resolved["cplm"]
    cplm_zip_hash = sha256_file(cplm_paths[CPLM_SPECIES_HUMAN])
    result = ensure_artifact(
        "cplm_human_compact.parquet",
        build_fn=lambda dest: parse_cplm_zip(cplm_paths[CPLM_SPECIES_HUMAN]).to_parquet(dest, index=False),
        expected_fingerprint=fingerprint({"cplm_zip_sha256": cplm_zip_hash}),
        force=force,
    )
    resolved["cplm_compact"] = result.path
    manifest.log("CPLM compact parquet (derived)", "n/a", result.path,
                 f"{'CACHED' if result.was_cached else 'built'} ({result.source})")

    manifest.save(paths.MANIFEST)
    print(f"[M1] {len(manifest.entries)} artefacts logged -> {paths.MANIFEST} "
          f"({sum(1 for e in manifest.entries if 'CACHED' in e['notes'])} reused from cache)")
    return manifest, resolved


In [4]:
# ---------------------------------------------------------------------------
# M1
# ---------------------------------------------------------------------------
manifest, resolved = run_m1(Paths, mock=MOCK_MODE, force=FORCE_REBUILD, ood_species=CFG.ood_species)
n_cached = sum(1 for e in manifest.entries if "CACHED" in e["notes"])
print(f"M1: {len(manifest.entries)} artefacts -> {Paths.MANIFEST} ({n_cached} reused from cache)")

download_sizes = report_raw_download_sizes(resolved)


/home/mdsr/ptm-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[M1] mock=False force=False raw_dir=ptm_temp/raw


[M1] Acquisition: 100%|██████████| 7/7 [00:08<00:00,  1.27s/source, glyde]          


[M1] OOD species: ['mouse', 'rat', 'yeast', 'ecoli']
[M1] 54 artefacts logged -> ptm_working/manifest.json (53 reused from cache)
M1: 54 artefacts -> ptm_working/manifest.json (53 reused from cache)

source                 n_files            size
swissprot                    1        12.87 MB
cplm                         1        16.74 MB
dbptm                       37        43.55 MB
qptm                         1     1,548.24 MB
oglcnac_atlas                2         9.51 MB
glycosite_atlas              1         2.95 MB
glyde                        2         0.70 MB
ood_proteomes                4        20.96 MB
ood_cplm                     4        23.48 MB
----------------------------------------------
TOTAL raw downloaded               1,679.00 MB


## M_E -- Dataset Exploration & Statistics

In [5]:
"""
M_E -- Dataset Exploration & Statistics (new stage, between M1 and M2)
=======================================================================
Inputs: M1's resolved raw source paths. Emits: one profile block per source (printed + saved as
JSON under Paths.EXPLORATION), plus a single cross-source overview table (TE).
Cost: minutes, CPU only (dominated by qPTM's ~1.55GB flat file).

WHY THIS STAGE EXISTS: several config thresholds (cdhit_identity, min_sites_per_type,
min_sites_headline, holdout_fraction) were previously set from assumption, before ever looking
at what these raw databases actually contain -- column shapes, PTM-type/organism composition,
score-field ranges. This stage gives that real view BEFORE M2/M3 run, cheaply, so those
thresholds can be revisited from evidence rather than assumption.

Each source is read through the SAME parser M1/M2/M3 already use for it (no separate parsing
logic here -- see PARSERS below), then run through a fixed set of generic checks
(profile_dataframe) plus, where relevant, a PTM-type / organism / numeric-field breakdown. This
is a first pass: profile_dataframe's checks are deliberately generic and dataset-agnostic;
additional source-specific breakdowns are easy to add once real output is in hand (columns/
values only become known from a real, non-mock run -- see the module docstring's own caveat).

MEMORY NOTE: qPTM's real flat file is ~1.55GB uncompressed and is read in full via
parse_qptm_flatfile (same as M3 will do downstream) -- comfortably within Kaggle's ~30GB RAM at
this stage (nothing else large is loaded yet), but the single biggest cost in this module by far.
"""
from __future__ import annotations
import json
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import pandas as pd

# Paths come from 00_config; parse_* functions come from 01_m1_acquisition (same kernel).


# ---------------------------------------------------------------------------
# Generic, fixed profile -- run identically on every source's parsed DataFrame.
# ---------------------------------------------------------------------------
def profile_dataframe(df: pd.DataFrame, label: str, max_unique_display: int = 15) -> dict:
    """Fixed set of checks: shape, per-column dtype, memory footprint, null counts, and
    cardinality (capped -- a high-cardinality column like `accession` reports its count, not
    every value). Returns a JSON-serialisable dict; does not print (see print_profile)."""
    n_rows, n_cols = df.shape
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    columns = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        n_unique = int(s.nunique(dropna=True))
        entry = {
            "column": col, "dtype": str(s.dtype), "n_null": n_null,
            "pct_null": round(100 * n_null / n_rows, 2) if n_rows else 0.0,
            "n_unique": n_unique,
        }
        if 0 < n_unique <= max_unique_display:
            entry["unique_values"] = sorted(s.dropna().astype(str).unique().tolist())
        columns.append(entry)
    return {
        "label": label, "n_rows": n_rows, "n_cols": n_cols,
        "memory_mb": round(mem_mb, 2), "columns": columns,
        "sample_rows": df.head(3).astype(str).to_dict(orient="records"),
    }


def print_profile(stats: dict):
    print(f"\n{'=' * 78}\n[{stats['label']}] {stats['n_rows']:,} rows x {stats['n_cols']} cols "
          f"({stats['memory_mb']:,.1f} MB in memory)\n{'-' * 78}")
    print(f"{'column':<28}{'dtype':<12}{'null%':>8}{'n_unique':>12}")
    for c in stats["columns"]:
        print(f"{c['column']:<28}{c['dtype']:<12}{c['pct_null']:>7.1f}%{c['n_unique']:>12,}")
        if "unique_values" in c:
            print(f"{'':<28}values: {c['unique_values']}")


def value_counts_report(df: pd.DataFrame, col: str, label: str, top_n: int = 30) -> Optional[pd.DataFrame]:
    """PTM-type / organism / label breakdown for one categorical column. Returns None (and
    prints nothing but a note) if `col` isn't present, since not every source has every field."""
    if col not in df.columns:
        print(f"[{label}] no '{col}' column -- skipping value-count breakdown.")
        return None
    vc = df[col].value_counts(dropna=False)
    out = vc.reset_index()
    out.columns = [col, "n"]
    out["pct"] = (100 * out["n"] / out["n"].sum()).round(2)
    print(f"\n[{label}] '{col}' breakdown ({vc.shape[0]} distinct values, top {min(top_n, len(out))} shown):")
    print(out.head(top_n).to_string(index=False))
    return out


def numeric_summary(df: pd.DataFrame, col: str, label: str) -> Optional[pd.Series]:
    """describe() for a numeric score/reliability/fdr-style column, coercing non-numeric
    placeholder strings (real qPTM/CPLM sources use '-' for missing numeric values)."""
    if col not in df.columns:
        print(f"[{label}] no '{col}' column -- skipping numeric summary.")
        return None
    vals = pd.to_numeric(df[col], errors="coerce")
    n_coerced_na = int(vals.isna().sum() - df[col].isna().sum())
    desc = vals.describe()
    print(f"\n[{label}] '{col}' numeric summary "
          f"({n_coerced_na} non-numeric value(s) treated as missing):")
    print(desc.to_string())
    return desc


# ---------------------------------------------------------------------------
# Per-source drivers. Each returns the parsed DataFrame(s) it profiled, so the orchestrator can
# build the cross-source overview table without re-parsing.
# ---------------------------------------------------------------------------
def explore_swissprot(swissprot_path: Path) -> pd.DataFrame:
    df = _read_csv_robust(swissprot_path, sep="\t")
    stats = profile_dataframe(df, "Swiss-Prot (raw TSV)")
    print_profile(stats)
    length_col = next((c for c in df.columns if c.lower() == "length"), None)
    if length_col:
        numeric_summary(df, length_col, "Swiss-Prot")
    _save_profile(stats, "swissprot")
    return df


def explore_cplm(cplm_zip_path: Path, parse_cplm_zip: Callable) -> pd.DataFrame:
    df = parse_cplm_zip(cplm_zip_path)
    stats = profile_dataframe(df, "CPLM 4.0 (human, Section 1)")
    print_profile(stats)
    value_counts_report(df, "type", "CPLM")
    if "lp" in df.columns:
        numeric_summary(df, "lp", "CPLM")
    _save_profile(stats, "cplm_human")
    return df


def explore_dbptm(dbptm_paths: Dict[str, Path], parse_dbptm_flatfile: Callable) -> pd.DataFrame:
    """dbPTM ships one file per PTM type -- profiled individually, then combined into one
    aggregate breakdown (row count per type) since that combined view is what actually informs
    min_sites_per_type/min_sites_headline."""
    parts = []
    per_type_counts = {}
    for ptm_type, path in dbptm_paths.items():
        df = parse_dbptm_flatfile(path, ptm_type)
        per_type_counts[ptm_type] = len(df)
        parts.append(df)
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=["accession", "position", "type", "pmid", "evidence"])
    stats = profile_dataframe(combined, "dbPTM 2025 (all types combined)")
    print_profile(stats)
    print(f"\n[dbPTM] rows per type (from acquisition, {len(per_type_counts)} files):")
    for t, n in sorted(per_type_counts.items(), key=lambda kv: -kv[1]):
        print(f"  {t:<28}{n:>8,}")
    _save_profile(stats, "dbptm_combined")
    _save_profile({"per_type_row_counts": per_type_counts}, "dbptm_per_type_counts")
    return combined


def explore_qptm(qptm_path: Path, parse_qptm_flatfile: Callable) -> pd.DataFrame:
    """The 1.55GB source -- profiled at both the raw-column level and the domain level
    (organism split, PTM-type breakdown, Reliability/fdr distributions), since this is the
    dataset with the least prior visibility (qPTM access was only obtained this session)."""
    raw = _read_qptm_raw(qptm_path)
    stats_raw = profile_dataframe(raw, "qPTM (raw columns, pre-parse)")
    print_profile(stats_raw)
    organism_col = next((c for c in raw.columns if c.lower() == "organism"), None)
    if organism_col:
        value_counts_report(raw, organism_col, "qPTM (raw)")
    ptm_col = next((c for c in raw.columns if c.lower() == "ptm"), None)
    if ptm_col:
        value_counts_report(raw, ptm_col, "qPTM (raw)", top_n=50)
    reliability_col = next((c for c in raw.columns if c.lower() == "reliability"), None)
    if reliability_col:
        value_counts_report(raw, reliability_col, "qPTM (raw)")
    fdr_col = next((c for c in raw.columns if "fdr" in c.lower()), None)
    if fdr_col:
        numeric_summary(raw, fdr_col, "qPTM (raw)")
    _save_profile(stats_raw, "qptm_raw")

    parsed = parse_qptm_flatfile(qptm_path)
    stats_parsed = profile_dataframe(parsed, "qPTM (parsed: accession/position/type/species/pmid)")
    print_profile(stats_parsed)
    value_counts_report(parsed, "species", "qPTM (parsed)")
    value_counts_report(parsed, "type", "qPTM (parsed)", top_n=50)
    _save_profile(stats_parsed, "qptm_parsed")
    return parsed


def _read_qptm_raw(path: Path) -> pd.DataFrame:
    """Plain TSV read for the raw-column profile, separate from parse_qptm_flatfile's
    normalized output -- lets M_E show the real header names verbatim (Organism/PMID/PTM/...)
    before M1's column-matching renames anything. Deliberately NOT routed through
    _read_csv_robust here: that helper reads the whole file into memory as bytes and decodes it
    to a string before pandas ever sees it, which would add a second full-file copy in memory on
    top of pandas' own parse -- real cost on qPTM specifically (1.55GB raw, 8+ GB in memory once
    parsed, the single largest file in this pipeline). A real run already confirmed qPTM's file
    parses cleanly as plain UTF-8 with no decode error, so the robust fallback buys nothing here
    and the memory cost isn't worth paying pre-emptively."""
    return pd.read_csv(path, sep="\t", low_memory=False)


def explore_oglcnac_atlas(oglcnac_paths: Dict[str, Path], parse_oglcnac_atlas_flatfile: Callable) -> pd.DataFrame:
    parts = []
    for dataset, path in oglcnac_paths.items():
        raw_csv = _read_csv_robust(path)
        stats = profile_dataframe(raw_csv, f"O-GlcNAcAtlas 5.0 [{dataset}] (raw)")
        print_profile(stats)
        if "species" in raw_csv.columns:
            value_counts_report(raw_csv, "species", f"O-GlcNAcAtlas [{dataset}]")
        _save_profile(stats, f"oglcnac_{dataset.lower().replace('-', '_')}")
        parsed = parse_oglcnac_atlas_flatfile(path)
        parsed["dataset"] = dataset
        parts.append(parsed)
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return combined


def explore_glycosite_atlas(glycosite_path: Path, parse_glycosite_atlas_flatfile: Callable) -> pd.DataFrame:
    df = parse_glycosite_atlas_flatfile(glycosite_path)
    stats = profile_dataframe(df, "N-GlycositeAtlas (parsed)")
    print_profile(stats)
    _save_profile(stats, "glycosite_atlas")
    return df


def explore_glyde(glyde_paths: Dict[str, Path], parse_glyde_flatfile: Callable) -> pd.DataFrame:
    parts = []
    for split, path in glyde_paths.items():
        df = parse_glyde_flatfile(path)
        df["split"] = split
        parts.append(df)
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    stats = profile_dataframe(combined, "N-GlyDE (parsed, train+test)")
    print_profile(stats)
    value_counts_report(combined, "Label", "N-GlyDE")
    _save_profile(stats, "glyde")
    return combined


def _save_profile(stats: dict, name: str):
    Paths.EXPLORATION.mkdir(parents=True, exist_ok=True)
    (Paths.EXPLORATION / f"{name}.json").write_text(json.dumps(stats, indent=2, default=str))


# ---------------------------------------------------------------------------
# [N1-CL#18] New: human-coverage, stratum-mapping, chemistry-match-rate, and cross-dataset
# overlap diagnostics -- added per explicit request, to answer three questions before M2/M3 run:
# how many unique human proteins does each dataset actually cover, how many sites does each PTM
# type have, and does the type->stratum mapping look right in practice (not just on paper).
# ---------------------------------------------------------------------------
def load_human_swissprot_lookup(swissprot_path: Path) -> Dict[str, str]:
    """accession -> sequence for every row in the raw acquired Swiss-Prot TSV. Deliberately reads
    the RAW file directly (not M2's filtered/CD-HIT'd corpus, which doesn't exist yet at this
    point in the notebook) -- this is the full reviewed-human set M1 downloaded, before M2's
    length filter. A handful of accessions that M2 later drops (>1022 aa) will show as "human but
    not in the eventual corpus"; that's expected and fine for a pre-M2 coverage check."""
    df = _read_csv_robust(swissprot_path, sep="\t")
    acc_col = next(c for c in df.columns if c.lower() in ("entry", "accession", "entry name"))
    seq_col = next(c for c in df.columns if c.lower() == "sequence")
    return dict(zip(df[acc_col].astype(str), df[seq_col].astype(str).str.upper()))


def _classify_sites(accs, poss, human_lookup: Dict[str, str]):
    """[N1-CL#23] Single pass over (accession, position) pairs, classifying each site into
    non_human / out_of_range / in_range(+which residue it actually landed on). Separating
    out_of_range from residue_mismatch is the whole point: the previous version lumped them
    together, so a wrong-stratum mapping and an isoform/coordinate problem were indistinguishable
    -- and both show up as the same "low chemistry match rate" number.

    Plain Python loop rather than pandas .apply(axis=1): ~10x faster on the row counts here
    (dbPTM 2.2M, qPTM 11.5M), and the per-type loop in the caller keeps only one type's data
    live at a time."""
    from collections import Counter
    n_non_human = n_out_of_range = 0
    residue_counts = Counter()
    for acc, pos in zip(accs, poss):
        seq = human_lookup.get(acc)
        if seq is None:
            n_non_human += 1
            continue
        if pos != pos or pos is None:  # NaN
            n_out_of_range += 1
            continue
        p = int(pos)
        if p < 1 or p > len(seq):
            n_out_of_range += 1
            continue
        residue_counts[seq[p - 1]] += 1
    return n_non_human, n_out_of_range, residue_counts


def _suggest_stratum(residue_counts) -> Tuple[str, float]:
    """Which configured stratum best explains where this type's sites ACTUALLY land, and what
    fraction it would capture. This is the empirical answer to "is the config's mapping right",
    independent of what the config currently claims."""
    total = sum(residue_counts.values())
    if not total:
        return ("n/a", float("nan"))
    best, best_frac = "n/a", 0.0
    for stratum, residues in CFG.stratum_residues.items():
        frac = sum(residue_counts.get(r, 0) for r in residues) / total
        if frac > best_frac:
            best, best_frac = stratum, frac
    return (best, round(100 * best_frac, 1))


def chemistry_failure_breakdown(df: pd.DataFrame, dataset_label: str, human_lookup: Dict[str, str],
                                 accession_col: str = "accession", position_col: str = "position",
                                 type_col: str = "type", top_n_residues: int = 5) -> pd.DataFrame:
    """THE diagnostic for residue-collapse and coordinate problems. Per (dataset, type):
      - pct_ok / pct_out_of_range / pct_residue_mismatch among human-accession sites
      - top_residues_landed_on: what the mismatching sites actually sit on
      - suggested_stratum / suggested_stratum_pct: the stratum that best fits the real data
    Read it as: scattered mismatch residues => coordinate/isoform problem; ONE dominant residue
    => the config's stratum mapping is wrong (or the label covers two chemistries); high
    out_of_range => isoform/length mismatch, not chemistry at all."""
    if type_col not in df.columns:
        df = df.copy()
        df[type_col] = dataset_label
    rows = []
    for ptm_type, sub in df.groupby(type_col):
        stratum = type_to_stratum(ptm_type)
        valid = CFG.stratum_residues.get(stratum, set())
        n_non_human, n_oor, residue_counts = _classify_sites(
            sub[accession_col].astype(str).values,
            pd.to_numeric(sub[position_col], errors="coerce").values,
            human_lookup)
        n_human_sites = len(sub) - n_non_human
        n_in_range = sum(residue_counts.values())
        n_ok = sum(residue_counts.get(r, 0) for r in valid)
        n_mismatch = n_in_range - n_ok
        mismatch_counts = {r: c for r, c in residue_counts.items() if r not in valid}
        top_mismatch = dict(sorted(mismatch_counts.items(), key=lambda kv: -kv[1])[:top_n_residues])
        suggested, suggested_pct = _suggest_stratum(residue_counts)
        rows.append({
            "dataset": dataset_label, "type": ptm_type, "config_stratum": stratum,
            "n_human_sites": n_human_sites,
            "pct_ok": round(100 * n_ok / n_human_sites, 1) if n_human_sites else None,
            "pct_out_of_range": round(100 * n_oor / n_human_sites, 1) if n_human_sites else None,
            "pct_residue_mismatch": round(100 * n_mismatch / n_human_sites, 1) if n_human_sites else None,
            "top_residues_landed_on": top_mismatch,
            "suggested_stratum": suggested, "suggested_stratum_pct": suggested_pct,
            "mapping_looks_wrong": (suggested != stratum and suggested_pct > 60),
        })
    out = pd.DataFrame(rows).sort_values("n_human_sites", ascending=False).reset_index(drop=True)
    print(f"\n[{dataset_label}] chemistry failure breakdown "
          f"(mapping_looks_wrong=True => config's stratum disagrees with where sites actually land):")
    print(out.to_string(index=False))
    return out


def isoform_suffix_audit(df: pd.DataFrame, dataset_label: str, human_lookup: Dict[str, str],
                          accession_col: str = "accession") -> dict:
    """[N1-CL#23] How many of this source's accessions carry a UniProt isoform suffix (P12345-2),
    and how many currently-unmatched accessions WOULD match the human corpus if the suffix were
    stripped. A large recoverable count means a cheap fix is available; near-zero means the
    accession gap is genuinely non-human/TrEMBL/long proteins and stripping buys nothing."""
    accs = set(df[accession_col].astype(str).unique())
    unmatched = {a for a in accs if a not in human_lookup}
    with_suffix = {a for a in accs if "-" in a and a.rsplit("-", 1)[-1].isdigit()}
    recoverable = {a for a in unmatched if a.rsplit("-", 1)[0] in human_lookup and a not in human_lookup}
    report = {
        "dataset": dataset_label, "n_unique_accessions": len(accs),
        "n_unmatched": len(unmatched), "n_with_isoform_suffix": len(with_suffix),
        "n_recoverable_by_stripping_suffix": len(recoverable),
        "pct_of_unmatched_recoverable": round(100 * len(recoverable) / len(unmatched), 1) if unmatched else 0.0,
    }
    print(f"[{dataset_label}] isoform-suffix audit: {report}")
    return report


def glyde_negative_contradiction_check(glyde_df: pd.DataFrame,
                                        positive_sets: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """[N1-CL#23] Do any N-GlyDE NON-glycosylated sites (the project's ONLY Gold-tier negatives)
    appear as annotated POSITIVES in dbPTM or N-GlycositeAtlas? Any hit is a direct contradiction
    in the gold negative set -- exactly the kind of thing that silently corrupts a validation tier,
    and nothing in the pipeline currently checks it."""
    neg = glyde_df[glyde_df["Label"].astype(str).str.contains("non", case=False)]
    neg_keys = set(zip(neg["UniProt Accession"].astype(str), neg["Position"].astype(int)))
    rows = []
    for name, pos_df in positive_sets.items():
        if not len(pos_df):
            continue
        pos_keys = set(zip(pos_df["accession"].astype(str), pos_df["position"].astype(int)))
        contradictions = neg_keys & pos_keys
        rows.append({"gold_negative_set": "N-GlyDE non-glycosylated", "vs_positive_source": name,
                     "n_gold_negatives": len(neg_keys), "n_contradictions": len(contradictions),
                     "pct_contradicted": round(100 * len(contradictions) / len(neg_keys), 2) if neg_keys else 0.0,
                     "example_contradictions": sorted(contradictions)[:5]})
    out = pd.DataFrame(rows)
    print("\nGold-tier negative contradiction check (any non-zero n_contradictions is a real "
          "problem in the only Gold-tier negative set this project has):")
    print(out.to_string(index=False) if len(out) else "(no positive sets to compare against)")
    return out


def qptm_reliability_sensitivity(qptm_parsed: pd.DataFrame, dbptm_nglyco: Optional[pd.DataFrame] = None
                                  ) -> pd.DataFrame:
    """[N1-CL#23] Retention at every reliability floor 0-5, human rows only -- plus, where a
    dbPTM comparison set is supplied, how much of the retained set overlaps dbPTM at each floor.
    The second number is the one that matters for CF-26: a stricter floor keeps better-evidenced
    sites, but better-evidenced sites are also more likely to already be in dbPTM, which erodes
    the independence that makes qPTM a replication set at all. This shows that tradeoff instead
    of assuming it away."""
    if "reliability" not in qptm_parsed.columns:
        print("[qPTM] no reliability column in parsed output -- skipping sensitivity sweep.")
        return pd.DataFrame()
    human = qptm_parsed[qptm_parsed["species"].astype(str).str.strip().str.lower().isin({"human", "homo sapiens"})]
    rel = pd.to_numeric(human["reliability"], errors="coerce")
    dbptm_keys = (set(zip(dbptm_nglyco["accession"].astype(str), dbptm_nglyco["position"].astype(int)))
                  if dbptm_nglyco is not None and len(dbptm_nglyco) else None)
    rows = []
    n_total = len(human)
    for floor in range(6):
        kept = human[rel >= floor]
        row = {"reliability_floor": floor, "n_human_rows_kept": len(kept),
               "pct_of_human_kept": round(100 * len(kept) / n_total, 1) if n_total else 0.0}
        if dbptm_keys is not None:
            kept_keys = set(zip(kept["accession"].astype(str), kept["position"].astype(int)))
            shared = kept_keys & dbptm_keys
            row["n_unique_sites_kept"] = len(kept_keys)
            row["pct_of_kept_also_in_dbptm"] = round(100 * len(shared) / len(kept_keys), 2) if kept_keys else 0.0
        rows.append(row)
    out = pd.DataFrame(rows)
    print("\n[qPTM] reliability-floor sensitivity (higher floor = better evidence, but higher "
          "dbPTM overlap = less independent replication -- CF-26 tradeoff):")
    print(out.to_string(index=False))
    return out


def type_coverage_report(df: pd.DataFrame, dataset_label: str, human_lookup: Dict[str, str],
                          accession_col: str = "accession", position_col: str = "position",
                          type_col: str = "type") -> pd.DataFrame:
    """Per-(dataset, type) row: site count, unique accessions, how many of those accessions are
    actually in the human Swiss-Prot set M1 acquired, and -- of the ones that are -- what
    fraction have the chemically-correct residue at the reported position. Low
    pct_accessions_human flags a source that isn't human-filtered (e.g. dbPTM's per-type files,
    which are NOT species-restricted at the source -- confirmed by checking dbPTM's own
    methodology, not assumed). Low chemistry_match_rate (among accessions that ARE human) flags
    either a wrong stratum mapping or a coordinate/off-by-one problem, independent of species."""
    rows = []
    if type_col not in df.columns:
        df = df.copy()
        df[type_col] = dataset_label  # single-type source with no explicit type column (e.g. glycosite atlas)
    for ptm_type, sub in df.groupby(type_col):
        accs = sub[accession_col].astype(str)
        n_sites = len(sub)
        uniq_accs = set(accs.unique())
        human_accs = uniq_accs & human_lookup.keys()
        n_human = len(human_accs)

        human_sub = sub[accs.isin(human_accs)]
        if len(human_sub):
            residues = human_sub.apply(
                lambda r: human_lookup[str(r[accession_col])][int(r[position_col]) - 1]
                if 0 <= int(r[position_col]) - 1 < len(human_lookup[str(r[accession_col])]) else None,
                axis=1)
            stratum = type_to_stratum(ptm_type)
            valid_residues = CFG.stratum_residues.get(stratum, set())
            chem_match = residues.isin(valid_residues).mean() if valid_residues else float("nan")
        else:
            chem_match = float("nan")

        rows.append({
            "dataset": dataset_label, "type": ptm_type, "stratum": type_to_stratum(ptm_type),
            "n_sites": n_sites, "n_unique_accessions": len(uniq_accs),
            "n_unique_accessions_human": n_human,
            "pct_accessions_human": round(100 * n_human / len(uniq_accs), 1) if uniq_accs else 0.0,
            "chemistry_match_rate_among_human": round(100 * chem_match, 1) if chem_match == chem_match else None,
        })
    out = pd.DataFrame(rows).sort_values("n_sites", ascending=False).reset_index(drop=True)
    print(f"\n[{dataset_label}] per-type coverage:")
    print(out.to_string(index=False))
    return out


def dbptm_species_suffix_audit(dbptm_paths: Dict[str, Path]) -> pd.DataFrame:
    """dbPTM integrates 41 source databases mapped onto UniProtKB across every species those
    sources cover -- it is NOT filtered to human at the source (confirmed against dbPTM's own
    published methodology). The only species signal in the per-type flat file is the UniProt
    entry_name suffix (e.g. '..._HUMAN' vs '..._MOUSE'), which parse_dbptm_flatfile discards
    since M3 doesn't need it (chemistry validation against the human corpus already filters
    non-human rows out downstream). This reads entry_name back out directly, across every
    acquired type, so the real species mixture is visible before assuming it's human-only."""
    rows = []
    for ptm_type, path in dbptm_paths.items():
        raw = _read_flatfile_tsv(path, header=None, names=DBPTM_POSITIONAL_COLUMNS)
        suffix = raw["entry_name"].astype(str).str.rsplit("_", n=1).str[-1]
        vc = suffix.value_counts()
        n_human = int(vc.get("HUMAN", 0))
        rows.append({"type": ptm_type, "n_rows": len(raw), "n_human_suffix": n_human,
                    "pct_human_suffix": round(100 * n_human / len(raw), 1) if len(raw) else 0.0,
                    "top_other_suffixes": dict(vc.drop(labels=["HUMAN"], errors="ignore").head(3))})
    out = pd.DataFrame(rows).sort_values("pct_human_suffix").reset_index(drop=True)
    print("\n[dbPTM] entry_name species-suffix audit (lower pct_human_suffix = more non-human "
         "rows in that type's raw file -- expected, since dbPTM is not human-filtered at the "
         "source; this is what step2's chemistry check against the human corpus is protecting "
         "against downstream):")
    print(out.to_string(index=False))
    return out


def cross_dataset_overlap(site_sets: Dict[str, pd.DataFrame], accession_col: str = "accession",
                          position_col: str = "position") -> pd.DataFrame:
    """Pairwise (accession, position) overlap between named site-sets. For sources meant to
    corroborate the SAME modification (e.g. dbPTM N-linked Glycosylation vs N-GlycositeAtlas),
    high overlap is reassuring. For sources that sound similar but are meant to be chemically
    DISTINCT (e.g. dbPTM's broad 'O-linked Glycosylation' vs O-GlcNAcAtlas's specific
    O-GlcNAcylation, kept as separate canonical types precisely because O-GlcNAc is an
    intracellular nucleocytoplasmic mark, not the extracellular mucin-type O-glycosylation dbPTM's
    bucket mostly captures) -- unexpectedly HIGH overlap here would be the signal that the two
    aren't as distinct as assumed and the type split needs revisiting."""
    keys = {name: set(zip(df[accession_col].astype(str), df[position_col].astype(int)))
            for name, df in site_sets.items() if len(df)}
    rows = []
    names = list(keys)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            shared = keys[a] & keys[b]
            rows.append({
                "set_a": a, "set_b": b, "n_a": len(keys[a]), "n_b": len(keys[b]),
                "n_shared": len(shared),
                "pct_of_a": round(100 * len(shared) / len(keys[a]), 1) if keys[a] else 0.0,
                "pct_of_b": round(100 * len(shared) / len(keys[b]), 1) if keys[b] else 0.0,
            })
    out = pd.DataFrame(rows)
    print("\nCross-dataset site overlap (same-type sources should overlap; look-alike-but-"
         "distinct types should mostly NOT):")
    print(out.to_string(index=False) if len(out) else "(nothing to compare)")
    return out


# ---------------------------------------------------------------------------
# Orchestrator
# ---------------------------------------------------------------------------
def run_me(resolved: Dict[str, object]) -> pd.DataFrame:
    """Runs every explore_* driver over M1's resolved paths, then assembles TE -- one row per
    source with the handful of numbers that most directly inform M2/M3's thresholds (row count,
    unique accessions, unique PTM types, unique proteins carrying >=1 site). Sources missing
    from `resolved` (e.g. qPTM before QPTM_DIRECT_URL is set) are skipped with a printed note,
    not a crash -- this stage must survive a partial M1 run.

    [N1-CL#18] Also runs, once every parsed dataframe is in hand: per-type human-coverage +
    chemistry-match-rate (type_coverage_report), dbPTM's species-suffix audit, and cross-dataset
    overlap for the N-glycosylation trio and the O-glycosylation/O-GlcNAc pair.
    """
    te_rows = []
    parsed: Dict[str, pd.DataFrame] = {}  # collected for the coverage + overlap diagnostics below

    def _te_row(name: str, df: pd.DataFrame, type_col: Optional[str] = "type"):
        acc_col = next((c for c in ("accession", "UniProt Accession") if c in df.columns), None)
        te_rows.append({
            "source": name, "n_rows": len(df),
            "n_unique_accessions": df[acc_col].nunique() if acc_col else None,
            "n_unique_types": df[type_col].nunique() if type_col and type_col in df.columns else None,
        })

    if "swissprot" in resolved:
        df = explore_swissprot(resolved["swissprot"])
        te_rows.append({"source": "swissprot", "n_rows": len(df),
                        "n_unique_accessions": df["accession"].nunique() if "accession" in df.columns else None,
                        "n_unique_types": None})
    else:
        print("[M_E] swissprot not in resolved -- skipped.")

    if "cplm" in resolved and CPLM_SPECIES_HUMAN in resolved["cplm"]:
        df = explore_cplm(resolved["cplm"][CPLM_SPECIES_HUMAN], parse_cplm_zip)
        _te_row("cplm_human", df)
        parsed["cplm_human"] = df
    else:
        print("[M_E] cplm not in resolved -- skipped.")

    if resolved.get("dbptm"):
        df = explore_dbptm(resolved["dbptm"], parse_dbptm_flatfile)
        _te_row("dbptm_combined", df)
        parsed["dbptm_combined"] = df
    else:
        print("[M_E] dbptm not in resolved -- skipped.")

    if "qptm" in resolved:
        df = explore_qptm(resolved["qptm"], parse_qptm_flatfile)
        _te_row("qptm_parsed", df)
        parsed["qptm_parsed"] = df
    else:
        print("[M_E] qptm not in resolved (QPTM_DIRECT_URL unset, or acquisition failed) -- skipped.")

    if resolved.get("oglcnac_atlas"):
        df = explore_oglcnac_atlas(resolved["oglcnac_atlas"], parse_oglcnac_atlas_flatfile)
        _te_row("oglcnac_atlas_combined", df)
        parsed["oglcnac_atlas_combined"] = df
    else:
        print("[M_E] oglcnac_atlas not in resolved -- skipped.")

    if "glycosite_atlas" in resolved:
        df = explore_glycosite_atlas(resolved["glycosite_atlas"], parse_glycosite_atlas_flatfile)
        _te_row("glycosite_atlas", df)
        parsed["glycosite_atlas"] = df
    else:
        print("[M_E] glycosite_atlas not in resolved -- skipped.")

    if resolved.get("glyde"):
        df = explore_glyde(resolved["glyde"], parse_glyde_flatfile)
        _te_row("glyde", df, type_col=None)
        parsed["glyde"] = df
    else:
        print("[M_E] glyde not in resolved -- skipped.")

    TE = pd.DataFrame(te_rows)
    Paths.TABLES.mkdir(parents=True, exist_ok=True)
    TE.to_csv(Paths.TABLES / "TE_dataset_overview.csv", index=False)
    print(f"\n{'=' * 78}\nTE -- cross-source overview (saved to {Paths.TABLES / 'TE_dataset_overview.csv'}):")
    print(TE.to_string(index=False))

    # --- [N1-CL#18] Human-coverage / stratum / chemistry diagnostics ---------------------------
    print(f"\n{'=' * 78}\nPer-type human-coverage, stratum mapping, and chemistry-match-rate:")
    if "swissprot" not in resolved:
        print("[M_E] swissprot not in resolved -- coverage diagnostics need it, skipping.")
    else:
        human_lookup = load_human_swissprot_lookup(resolved["swissprot"])
        coverage_tables = []
        if "cplm_human" in parsed:
            coverage_tables.append(type_coverage_report(parsed["cplm_human"], "CPLM (human)", human_lookup))
        if "dbptm_combined" in parsed:
            coverage_tables.append(type_coverage_report(parsed["dbptm_combined"], "dbPTM", human_lookup))
        if "qptm_parsed" in parsed:
            # qPTM's own species column already restricts what "unique accessions" means per type;
            # human_lookup intersection here still checks the accession is a REAL human UniProt
            # entry (qPTM's "species" self-report isn't independently verified against UniProt).
            coverage_tables.append(type_coverage_report(parsed["qptm_parsed"], "qPTM", human_lookup))
        if "oglcnac_atlas_combined" in parsed:
            coverage_tables.append(type_coverage_report(parsed["oglcnac_atlas_combined"], "O-GlcNAcAtlas", human_lookup))
        if "glycosite_atlas" in parsed:
            coverage_tables.append(type_coverage_report(parsed["glycosite_atlas"], "N-GlycositeAtlas", human_lookup))
        if "glyde" in parsed and len(parsed["glyde"]):
            glyde_typed = parsed["glyde"].rename(columns={"UniProt Accession": "accession", "Position": "position"}).copy()
            glyde_typed["type"] = glyde_typed["Label"]  # "glycosylated" / "non-glycosylated", not a PTM type, but reuses the same report shape
            coverage_tables.append(type_coverage_report(glyde_typed, "N-GlyDE", human_lookup))
        if coverage_tables:
            TE_TYPE = pd.concat(coverage_tables, ignore_index=True)
            TE_TYPE.to_csv(Paths.TABLES / "TE_type_coverage.csv", index=False)
            print(f"\n(saved to {Paths.TABLES / 'TE_type_coverage.csv'})")

    # --- [N1-CL#23] Chemistry failure breakdown: out_of_range vs residue_mismatch -------------
    if "swissprot" in resolved:
        print(f"\n{'=' * 78}\nChemistry failure breakdown (separates coordinate/isoform problems "
              f"from wrong-stratum mappings):")
        breakdowns = []
        for label, key in [("CPLM (human)", "cplm_human"), ("dbPTM", "dbptm_combined"),
                           ("qPTM", "qptm_parsed"), ("O-GlcNAcAtlas", "oglcnac_atlas_combined"),
                           ("N-GlycositeAtlas", "glycosite_atlas")]:
            if key in parsed:
                breakdowns.append(chemistry_failure_breakdown(parsed[key], label, human_lookup))
        if breakdowns:
            TE_FAIL = pd.concat(breakdowns, ignore_index=True)
            TE_FAIL.to_csv(Paths.TABLES / "TE_chemistry_failures.csv", index=False)
            flagged = TE_FAIL[TE_FAIL["mapping_looks_wrong"]]
            if len(flagged):
                print(f"\n>>> {len(flagged)} type(s) where the config's stratum disagrees with "
                      f"where sites actually land -- these are the residue-collapse candidates:")
                print(flagged[["dataset", "type", "config_stratum", "suggested_stratum",
                              "suggested_stratum_pct", "n_human_sites"]].to_string(index=False))
            print(f"\n(saved to {Paths.TABLES / 'TE_chemistry_failures.csv'})")

        # --- Isoform-suffix audit ---
        print(f"\n{'=' * 78}\nIsoform-suffix audit (can unmatched accessions be recovered cheaply?):")
        iso_rows = []
        for label, key in [("CPLM (human)", "cplm_human"), ("dbPTM", "dbptm_combined"),
                           ("qPTM", "qptm_parsed"), ("O-GlcNAcAtlas", "oglcnac_atlas_combined"),
                           ("N-GlycositeAtlas", "glycosite_atlas")]:
            if key in parsed:
                iso_rows.append(isoform_suffix_audit(parsed[key], label, human_lookup))
        if iso_rows:
            pd.DataFrame(iso_rows).to_csv(Paths.TABLES / "TE_isoform_audit.csv", index=False)

    # --- [N1-CL#23] Gold-tier negative contradiction check -------------------------------------
    if "glyde" in parsed and len(parsed["glyde"]):
        pos_sets = {}
        if "dbptm_combined" in parsed:
            d = parsed["dbptm_combined"]
            pos_sets["dbPTM N-linked Glycosylation"] = d[d["type"] == "N-linked Glycosylation"]
        if "glycosite_atlas" in parsed:
            pos_sets["N-GlycositeAtlas"] = parsed["glycosite_atlas"]
        if pos_sets:
            contra = glyde_negative_contradiction_check(parsed["glyde"], pos_sets)
            contra.to_csv(Paths.TABLES / "TE_gold_negative_contradictions.csv", index=False)

    # --- [N1-CL#23] qPTM reliability-floor sensitivity -----------------------------------------
    if "qptm_parsed" in parsed:
        dbptm_ref = None
        if "dbptm_combined" in parsed:
            d = parsed["dbptm_combined"]
            dbptm_ref = d[d["type"].isin(["Phosphorylation", "Acetylation", "Ubiquitination",
                                          "Sumoylation", "Methylation", "N-linked Glycosylation",
                                          "O-linked Glycosylation"])]
        rel_table = qptm_reliability_sensitivity(parsed["qptm_parsed"], dbptm_ref)
        if len(rel_table):
            rel_table.to_csv(Paths.TABLES / "TE_qptm_reliability_sensitivity.csv", index=False)

    # --- [N1-CL#18] dbPTM species-suffix audit -------------------------------------------------
    if resolved.get("dbptm"):
        species_audit = dbptm_species_suffix_audit(resolved["dbptm"])
        species_audit.to_csv(Paths.TABLES / "TE_dbptm_species_audit.csv", index=False)

    # --- [N1-CL#18] Cross-dataset overlap for look-alike / corroborating types ----------------
    overlap_sets = {}
    if "dbptm_combined" in parsed:
        dbptm_df = parsed["dbptm_combined"]
        if "N-linked Glycosylation" in dbptm_df["type"].values:
            overlap_sets["dbPTM N-linked Glycosylation"] = dbptm_df[dbptm_df["type"] == "N-linked Glycosylation"]
        if "O-linked Glycosylation" in dbptm_df["type"].values:
            overlap_sets["dbPTM O-linked Glycosylation"] = dbptm_df[dbptm_df["type"] == "O-linked Glycosylation"]
    if "glycosite_atlas" in parsed:
        overlap_sets["N-GlycositeAtlas"] = parsed["glycosite_atlas"]
    if "glyde" in parsed and len(parsed["glyde"]):
        glyde_pos = parsed["glyde"][parsed["glyde"]["Label"].astype(str).str.contains("glycosylated")
                                    & ~parsed["glyde"]["Label"].astype(str).str.contains("non")]
        overlap_sets["N-GlyDE (positives)"] = glyde_pos.rename(
            columns={"UniProt Accession": "accession", "Position": "position"})
    if "oglcnac_atlas_combined" in parsed:
        oglcnac_df = parsed["oglcnac_atlas_combined"]
        overlap_sets["O-GlcNAcAtlas"] = oglcnac_df[oglcnac_df.get("dataset", "Dataset-I") == "Dataset-I"] \
            if "dataset" in oglcnac_df.columns else oglcnac_df
    if len(overlap_sets) >= 2:
        overlap_table = cross_dataset_overlap(overlap_sets)
        overlap_table.to_csv(Paths.TABLES / "TE_cross_dataset_overlap.csv", index=False)

    return TE


In [6]:
# ---------------------------------------------------------------------------
# M_E -- run once, right after M1, before any of M2/M3 touch the raw data.
# ---------------------------------------------------------------------------
TE = run_me(resolved)



[Swiss-Prot (raw TSV)] 18,128 rows x 8 cols (17.8 MB in memory)
------------------------------------------------------------------------------
column                      dtype          null%    n_unique
Entry                       object          0.0%      18,128
Length                      int64           0.0%       1,006
Sequence                    object          0.0%      18,050
Glycosylation               object         77.7%       3,144
Modified residue            object         56.6%       7,447
Lipidation                  object         95.8%         643
Signal peptide              object         82.5%         694
Transit peptide             object         96.9%         310

[Swiss-Prot] 'Length' numeric summary (0 non-numeric value(s) treated as missing):
count    18128.000000
mean       412.313714
std        230.802821
min          2.000000
25%        228.000000
50%        372.000000
75%        556.000000
max       1022.000000

[CPLM 4.0 (human, Section 1)] 298,634 rows x 5

/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)



[qPTM (parsed: accession/position/type/species/pmid)] 11,482,553 rows x 6 cols (2,843.6 MB in memory)
------------------------------------------------------------------------------
column                      dtype          null%    n_unique
accession                   object          0.0%      40,730
position                    Int64           0.0%       9,455
type                        object          0.0%           6
                            values: ['Acetylation', 'Glycosylation', 'Methylation', 'Phosphorylation', 'SUMOylation', 'Ubiquitylation']
species                     object          0.0%           4
                            values: ['Human', 'Mouse', 'Rat', 'Yeast']
pmid                        object          0.0%         625
reliability                 Int64           0.0%           6
                            values: ['0', '1', '2', '3', '4', '5']

[qPTM (parsed)] 'species' breakdown (4 distinct values, top 4 shown):
species       n   pct
  Human 6087375 53.01
  

/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)
/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)



[O-GlcNAcAtlas 5.0 [Dataset-I] (raw)] 46,517 rows x 19 cols (40.5 MB in memory)
------------------------------------------------------------------------------
column                      dtype          null%    n_unique
id                          int64           0.0%      46,517
species                     object          0.0%          29
sample_type                 object          5.5%         191
accession                   object          0.0%       7,793
accession_source            object          0.0%           8
                            values: ['FlyBase', 'GenBank', 'MSU ID', 'NCBI', 'PDB', 'TAIR', 'UniProt', 'Uniprot']
entry_name                  object          5.3%       6,596
protein_name                object          1.4%       6,845
gene_name                   object          6.0%       6,267
peptide_seq                 object         27.4%      12,652
site_residue                object          7.9%           7
                            values: ['1', '8', 'C', 'S'

/tmp/ipykernel_251512/688393808.py:632: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  TE_TYPE = pd.concat(coverage_tables, ignore_index=True)



[CPLM (human)] chemistry failure breakdown (mapping_looks_wrong=True => config's stratum disagrees with where sites actually land):
     dataset                     type config_stratum  n_human_sites  pct_ok  pct_out_of_range  pct_residue_mismatch                        top_residues_landed_on suggested_stratum  suggested_stratum_pct  mapping_looks_wrong
CPLM (human)           Ubiquitination              K          87922    99.3               0.0                   0.6 {'L': 75, 'G': 50, 'S': 42, 'E': 41, 'R': 36}                 K                   99.4                False
CPLM (human)              Acetylation              K          38247    99.2               0.1                   0.7 {'L': 32, 'E': 31, 'A': 30, 'P': 21, 'V': 19}                 K                   99.3                False
CPLM (human)              Sumoylation              K          31074    99.3               0.0                   0.7 {'E': 23, 'L': 22, 'P': 20, 'S': 19, 'I': 16}                 K                

## M2 -- Corpus Construction

Length filter, exact per-stratum residue counts, CD-HIT at 40%, cluster-level stratified split (T3 reports achieved per-type fractions). OOD corpora get the same filter and counting, no CD-HIT/split.

In [7]:
"""
M2 -- Corpus construction
=========================
Inputs: M1 Swiss-Prot (+ a lightweight, unfiltered read of M1's raw PTM sources).
Emits: corpus.parquet (accession, sequence, length, cluster_id, split); stratum_counts.json;
       split_manifest.json.
Gate: protein count within 15,000-20,000; each primary type >=1000 sites in the discovery
      partition (else halve holdout_fraction to 0.15 and re-split).
Cost: hours, CPU only.

ORDERING NOTE: M3's split-inheritance step requires cluster->split assignment to already exist,
but a stratified split needs to know which proteins carry which PTM types before M3's full
filtering cascade has run. Resolved by doing a LIGHTWEIGHT, UNFILTERED raw positional join here
(no evidence filtering, no LP masking, no dedup -- just "does this accession have >=1 raw
reported site of primary type X") purely to drive cluster stratification; M3 then re-derives the
real, filtered labels independently and inherits the split column by cluster. Stratification
target is therefore approximate; T2's final label counts are exact and filtered.

[N1-CL#10] build_debug_corpus_subset (M6-onward debug-scale subsampling) removed from this
module -- it was dead code in the source pipeline (defined, never called) and its own docstring
scopes it to M6+, not M1-M3. Belongs in N2/N3 if and when debug-scale runs are rebuilt there.
"""
from __future__ import annotations
import json
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Paths/CFG/fingerprint/find_cached/sha256_file/config_fingerprint come from 00_config.


def filter_swissprot(swissprot_tsv: Path, max_length: int) -> pd.DataFrame:
    """M2 step 1: reviewed human, length <= max_length. Re-asserts the filter even though M1's
    UniProt query already applies it, so this function is correct fed any TSV."""
    df = pd.read_csv(swissprot_tsv, sep="\t")
    col_map = {
        "accession": next(c for c in df.columns if c.lower() in ("entry", "accession", "entry name")),
        "length": next(c for c in df.columns if c.lower() == "length"),
        "sequence": next(c for c in df.columns if c.lower() == "sequence"),
    }
    out = df.rename(columns={v: k for k, v in col_map.items()})[["accession", "length", "sequence"]].copy()
    out = out[out["length"] <= max_length].reset_index(drop=True)
    out["sequence"] = out["sequence"].astype(str).str.upper()
    return out


def compute_stratum_counts(corpus: pd.DataFrame, stratum_residues: Dict[str, set]) -> Dict[str, int]:
    """M2 steps 2-3: exact residue counts per stratum + total (must be exact, not sampled --
    every M7 enrichment value depends on g_ell being exact).
    [N1-CL#15] Takes `stratum_residues` from the caller (cfg.stratum_residues) instead of a
    second, hardcoded module-level copy -- the original pipeline defined this dict twice (here
    and in Config), a duplication that would have silently gone stale the moment one copy was
    edited and the other wasn't (exactly what happened when E/M/W/Q were added for Category B)."""
    from tqdm.auto import tqdm
    counts = {k: 0 for k in stratum_residues}
    total = 0
    for seq in tqdm(corpus["sequence"], desc="[M2] Stratum counts", unit="protein", leave=False):
        total += len(seq)
        for stratum, residues in stratum_residues.items():
            counts[stratum] += sum(seq.count(r) for r in residues)
    counts["total_residues"] = total
    counts["n_proteins"] = len(corpus)
    return counts


def run_cdhit(corpus: pd.DataFrame, identity: float = 0.40, tmp_dir: Path = None) -> Dict[str, int]:
    """
    M2 step 4: CD-HIT clustering at `identity`. Requires the `cd-hit` binary on PATH (apt
    package `cd-hit`). Word size follows CD-HIT's documented threshold bands:
    n=5 for [0.7,1.0], n=4 for [0.6,0.7), n=3 for [0.5,0.6), n=2 for [0.4,0.5).
    Returns {accession: cluster_id}.

    Two real behaviours handled explicitly, both silent otherwise:
    (1) CD-HIT's own "-l" (throw-away length) defaults to 10 -- any sequence <=10 residues is
        silently dropped with no warning. Fixed by setting -l to n-1, the lowest CD-HIT will
        accept for the chosen word length (confirmed empirically against cd-hit 4.8.1: n-2
        raises "Fatal Error: Too short -l, redefine it").
    (2) Independently of -l, a sequence shorter than word length n can never form an n-mer, so
        it can never be clustered at this word length by any -l setting. Fixed by routing
        sequences shorter than n around CD-HIT entirely and assigning each its own singleton
        cluster -- the correct redundancy-clustering answer for something too short to compare.
    """
    tmp_dir = Path(tmp_dir or tempfile.mkdtemp())
    tmp_dir.mkdir(parents=True, exist_ok=True)

    if identity >= 0.7:
        n = 5
    elif identity >= 0.6:
        n = 4
    elif identity >= 0.5:
        n = 3
    else:
        n = 2

    seq_lens = corpus["sequence"].str.len()
    too_short_mask = seq_lens < n
    cdhit_corpus = corpus.loc[~too_short_mask]
    too_short = corpus.loc[too_short_mask]

    fasta_path = tmp_dir / "corpus.fasta"
    out_path = tmp_dir / "corpus_cdhit"
    with open(fasta_path, "w") as f:
        for _, row in cdhit_corpus.iterrows():
            f.write(f">{row['accession']}\n{row['sequence']}\n")

    min_len_kept = max(1, n - 1)  # lowest -l CD-HIT accepts for this n -- see (1) above
    cmd = ["cd-hit", "-i", str(fasta_path), "-o", str(out_path), "-c", str(identity),
           "-n", str(n), "-M", "0", "-T", "0", "-d", "0", "-l", str(min_len_kept)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"cd-hit failed (rc={result.returncode}):\n{result.stdout}\n{result.stderr}")

    clstr_path = Path(str(out_path) + ".clstr")
    assignment: Dict[str, int] = {}
    current_cluster = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                current_cluster = int(line.strip().split()[-1])
            else:
                acc = line.split(">")[1].split("...")[0]  # "0   350aa, >P12345... *"
                assignment[acc] = current_cluster

    next_cluster = (max(assignment.values()) + 1) if assignment else 0
    for _, row in too_short.iterrows():  # each too-short sequence gets its own singleton cluster
        assignment[row["accession"]] = next_cluster
        next_cluster += 1

    return assignment


def raw_type_hits_by_accession(m1_raw_paths: Dict[str, Path], parsers: Dict[str, callable],
                                cfg, corpus_seq: Optional[Dict[str, str]] = None
                                ) -> Dict[str, Dict[str, int]]:
    """Lightweight unfiltered raw site counts per accession per canonical type, used only to
    drive M2's stratified split (see module docstring).

    [N1-CL#26] Now keeps EVERY canonical type, not just cfg.primary_types, and resolves types
    residue-aware where the corpus sequence is available. Reason: the real M_E run showed that
    restricting stratification to three types is too narrow now that phosphorylation_Y (~35,600
    recoverable sites), methylation_R (~4,150) and sulfoxidation (~3,900) clear the headline bar.
    Multi-label iterative stratification is designed for exactly this -- balancing many types at
    once -- and the achieved per-type fractions are now reported (T3) rather than assumed."""
    known = set(cfg.ptm_type_to_stratum)
    hits: Dict[str, Dict[str, int]] = {}
    for label, path in m1_raw_paths.items():
        if label not in parsers:
            continue
        df = parsers[label](path)
        if "type" not in df.columns or not len(df):
            continue
        df = df.copy()
        if corpus_seq is not None and "position" in df.columns:
            residues = []
            for acc, pos in zip(df["accession"].astype(str), df["position"]):
                seq = corpus_seq.get(acc)
                if seq is None or pd.isna(pos):
                    residues.append(None); continue
                p = int(pos)
                residues.append(seq[p - 1] if 1 <= p <= len(seq) else None)
            df["type"] = [normalize_ptm_type(t, known_types=known, residue=r)
                          for t, r in zip(df["type"], residues)]
        else:
            df["type"] = df["type"].map(lambda t: normalize_ptm_type(t, known_types=known))
        df = df[df["type"].isin(known)]
        for acc, sub in df.groupby("accession"):
            d = hits.setdefault(acc, {})
            for t, n in sub["type"].value_counts().items():
                d[t] = d.get(t, 0) + int(n)
    return hits


def iterative_stratified_split(cluster_type_counts: Dict[int, Dict[str, int]],
                                all_cluster_ids: List[int],
                                holdout_fraction: float, seed: int) -> Dict[int, str]:
    """Greedy multi-label iterative stratification (Sechidis et al. 2011, simplified) at cluster
    level. Clusters with no primary-type sites (most proteins) are split by plain ratio at the
    end, since they only affect background sizing, not per-type balance."""
    rng = np.random.RandomState(seed)
    labelled = [c for c in all_cluster_ids if cluster_type_counts.get(c)]
    unlabelled = [c for c in all_cluster_ids if not cluster_type_counts.get(c)]
    rng.shuffle(labelled)
    rng.shuffle(unlabelled)

    types = sorted({t for d in cluster_type_counts.values() for t in d})
    total_per_type = {t: sum(d.get(t, 0) for d in cluster_type_counts.values()) for t in types}
    target_holdout = {t: total_per_type[t] * holdout_fraction for t in types}
    running_holdout = {t: 0 for t in types}
    assignment: Dict[int, str] = {}

    def deficit(running):
        return sum(max(0.0, target_holdout[t] - running.get(t, 0)) ** 2 for t in types)

    n_holdout_target = int(round(len(all_cluster_ids) * holdout_fraction))
    n_holdout = 0
    for cid in labelled:
        counts = cluster_type_counts[cid]
        trial = dict(running_holdout)
        for t, c in counts.items():
            trial[t] = trial.get(t, 0) + c
        prefer_holdout = deficit(trial) < deficit(running_holdout)
        if n_holdout >= n_holdout_target * 1.2:
            prefer_holdout = False
        assignment[cid] = "holdout" if prefer_holdout else "discovery"
        if prefer_holdout:
            n_holdout += 1
            running_holdout = trial

    for cid in unlabelled:
        prefer_holdout = (n_holdout < n_holdout_target)
        assignment[cid] = "holdout" if prefer_holdout else "discovery"
        if prefer_holdout:
            n_holdout += 1

    return assignment


def build_corpus(swissprot_tsv: Path, m1_raw_paths: Dict[str, Path], parsers: Dict[str, callable],
                  cfg, paths, tmp_dir: Path = None) -> Tuple[pd.DataFrame, dict]:
    corpus = filter_swissprot(swissprot_tsv, cfg.max_protein_length)
    stratum_counts = compute_stratum_counts(corpus, cfg.stratum_residues)

    # Raw source parsing runs before CD-HIT (not after), so a parsing bug fails in seconds
    # instead of after paying CD-HIT's full clustering cost.
    corpus_seq = dict(zip(corpus["accession"], corpus["sequence"]))
    raw_hits = raw_type_hits_by_accession(m1_raw_paths, parsers, cfg, corpus_seq=corpus_seq)

    cluster_of = run_cdhit(corpus, identity=cfg.cdhit_identity, tmp_dir=tmp_dir)
    corpus["cluster_id"] = corpus["accession"].map(cluster_of)
    n_unclustered = corpus["cluster_id"].isna().sum()
    if n_unclustered:
        raise RuntimeError(f"{n_unclustered} proteins did not receive a CD-HIT cluster id")
    corpus["cluster_id"] = corpus["cluster_id"].astype(int)

    cluster_type_counts: Dict[int, Dict[str, int]] = {}
    for _, row in corpus.iterrows():
        acc, cid = row["accession"], row["cluster_id"]
        if acc in raw_hits:
            d = cluster_type_counts.setdefault(cid, {})
            for t, n in raw_hits[acc].items():
                d[t] = d.get(t, 0) + n

    all_cluster_ids = sorted(corpus["cluster_id"].unique().tolist())
    split_of_cluster = iterative_stratified_split(
        cluster_type_counts, all_cluster_ids, cfg.holdout_fraction, cfg.holdout_seed
    )
    corpus["split"] = corpus["cluster_id"].map(split_of_cluster)

    n_proteins = len(corpus)
    gate_protein_count_ok = 15000 <= n_proteins <= 20000

    # [N1-CL#26] Per-type split composition for EVERY type, not just the primary three. Iterative
    # stratification is a greedy heuristic with no optimality guarantee, so the ACHIEVED fraction
    # per type is the only honest way to judge it -- previously this lived only inside
    # split_manifest.json for 3 types and was effectively invisible.
    all_types = sorted({t for d in cluster_type_counts.values() for t in d})
    split_rows = []
    for t in all_types:
        disc = sum(d.get(t, 0) for cid, d in cluster_type_counts.items()
                   if split_of_cluster[cid] == "discovery")
        hold = sum(d.get(t, 0) for cid, d in cluster_type_counts.items()
                   if split_of_cluster[cid] == "holdout")
        total = disc + hold
        split_rows.append({
            "type": t, "stratum": cfg.ptm_type_to_stratum.get(t, "UNMAPPED"),
            "raw_sites_total": total, "raw_sites_discovery": disc, "raw_sites_holdout": hold,
            "achieved_holdout_fraction": round(hold / total, 4) if total else float("nan"),
            "target_holdout_fraction": cfg.holdout_fraction,
            "abs_deviation_from_target": (round(abs(hold / total - cfg.holdout_fraction), 4)
                                          if total else float("nan")),
            "is_primary_type": t in cfg.primary_types,
            "v12b_eligible": disc >= cfg.holdout_min_sites_per_type and hold >= cfg.holdout_min_sites_per_type,
        })
    T3_split = pd.DataFrame(split_rows).sort_values("raw_sites_total", ascending=False).reset_index(drop=True)

    per_type_discovery_sites = {t: int(T3_split.loc[T3_split["type"] == t, "raw_sites_discovery"].sum())
                                for t in cfg.primary_types}
    gate_report = {
        "n_proteins": n_proteins,
        "gate_protein_count_15k_20k": gate_protein_count_ok,
        "n_types_stratified_on": len(all_types),
        "per_type_discovery_raw_sites": per_type_discovery_sites,
        "target_holdout_fraction": cfg.holdout_fraction,
        "worst_abs_deviation_from_target": (float(T3_split["abs_deviation_from_target"].max())
                                            if len(T3_split) else float("nan")),
        "note": "raw/unfiltered counts for gating the SPLIT only; T2's real counts come from M3.",
    }
    return corpus, gate_report, T3_split


def save_corpus_artefacts(corpus: pd.DataFrame, stratum_counts: dict, gate_report: dict, paths, cfg,
                           fp: Optional[str] = None):
    corpus.to_parquet(paths.CORPUS, index=False)
    if fp is not None:
        paths.CORPUS.with_suffix(paths.CORPUS.suffix + ".fp").write_text(fp)
    paths.STRATUM_COUNTS.write_text(json.dumps(stratum_counts, indent=2))

    split_manifest = {
        "seed": cfg.holdout_seed,
        "holdout_fraction": cfg.holdout_fraction,
        "stratify_by": cfg.holdout_stratify_by,
        "cluster_to_split": corpus.drop_duplicates("cluster_id").set_index("cluster_id")["split"].to_dict(),
        "gate_report": gate_report,
    }
    paths.SPLIT_MANIFEST.write_text(json.dumps(split_manifest, indent=2, default=str))
    return paths.CORPUS, paths.STRATUM_COUNTS, paths.SPLIT_MANIFEST


def flag_human_homologs(ood_corpus: pd.DataFrame, human_corpus: pd.DataFrame, identity: float,
                         tmp_dir: Path = None) -> Tuple[pd.DataFrame, dict]:
    """[N1-CL#51] Flags every OOD protein that has a >= `identity` homolog in the HUMAN corpus,
    addressing the limitation that cross-species homology was previously not removed at all.

    Uses `cd-hit-2d`, which is built for exactly this: given db1 (human) and db2 (OOD), it reports
    which db2 sequences are NOT similar to anything in db1. Its output file contains the
    *non-redundant* OOD sequences; anything absent from that output has a human homolog.

    Adds a boolean `homologous_to_human` column rather than deleting rows, so BOTH analyses stay
    available from one corpus:
      * full OOD  -- every protein (the original, uncorrected view)
      * strict OOD -- `homologous_to_human == False` only (homology-corrected)
    Deleting would have forced a choice; flagging lets N4 report both and show the gap, which is
    itself the interesting quantity."""
    tmp_dir = Path(tmp_dir or tempfile.mkdtemp())
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # cd-hit-2d's word-size bands match cd-hit's.
    n = 5 if identity >= 0.7 else 4 if identity >= 0.6 else 3 if identity >= 0.5 else 2
    human_fa, ood_fa, out_fa = tmp_dir / "human.fa", tmp_dir / "ood.fa", tmp_dir / "ood_nr.fa"
    for path, df in [(human_fa, human_corpus), (ood_fa, ood_corpus)]:
        with open(path, "w") as f:
            for acc, seq in zip(df["accession"], df["sequence"]):
                f.write(f">{acc}\n{seq}\n")

    cmd = ["cd-hit-2d", "-i", str(human_fa), "-i2", str(ood_fa), "-o", str(out_fa),
           "-c", str(identity), "-n", str(n), "-M", "0", "-T", "0", "-d", "0",
           "-l", str(max(1, n - 1))]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        # Non-fatal: an unflagged OOD corpus is still usable, it just loses the strict variant.
        print(f"[M2-OOD] cd-hit-2d failed (rc={result.returncode}) -- homology flag not applied.\n"
              f"{result.stderr[:400]}")
        out = ood_corpus.copy()
        out["homologous_to_human"] = pd.NA
        return out, {"applied": False, "reason": f"cd-hit-2d rc={result.returncode}"}

    # Accessions surviving into the output have NO human homolog at this threshold.
    no_homolog = set()
    with open(out_fa) as f:
        for line in f:
            if line.startswith(">"):
                no_homolog.add(line[1:].strip().split()[0])
    # Sequences shorter than the word size are never emitted by cd-hit-2d; treat them as
    # non-homologous rather than silently flagging them as homologs.
    too_short = set(ood_corpus.loc[ood_corpus["sequence"].str.len() < n, "accession"])
    no_homolog |= too_short

    out = ood_corpus.copy()
    out["homologous_to_human"] = ~out["accession"].isin(no_homolog)
    n_hom = int(out["homologous_to_human"].sum())
    report = {
        "applied": True, "identity": identity,
        "n_ood_proteins": int(len(out)),
        "n_with_human_homolog": n_hom,
        "n_without_human_homolog": int(len(out) - n_hom),
        "pct_with_human_homolog": round(100 * n_hom / len(out), 2) if len(out) else 0.0,
        "n_too_short_to_compare": len(too_short),
    }
    return out, report


def build_ood_corpus(swissprot_tsv: Path, species: str, cfg, paths) -> Tuple[pd.DataFrame, dict]:
    """[N1-CL#25] OOD corpus for one non-human species: same length filter and same exact
    per-stratum residue counting as the human corpus, so background rates are computed
    identically and are directly comparable.

    Deliberately NO CD-HIT and NO discovery/holdout split. Both exist to prevent a homolog of a
    discovery protein leaking into holdout WITHIN one corpus; an OOD species is held out by
    construction, so there is no within-corpus split to protect. (Cross-species homology to the
    human corpus is NOT removed either -- that is a real caveat, but the OOD claim here is
    species-distribution generalisation, not sequence novelty, and ESM-2's own pretraining
    already saw all of these proteomes.)"""
    corpus = filter_swissprot(swissprot_tsv, cfg.max_protein_length)
    corpus["species"] = species
    stratum_counts = compute_stratum_counts(corpus, cfg.stratum_residues)
    print(f"[M2-OOD:{species}] {len(corpus)} proteins, {stratum_counts['total_residues']:,} residues")
    return corpus, stratum_counts


def run_m2_ood(ood_proteomes: Dict[str, Path], cfg, paths, force: bool = False,
               human_corpus: Optional[pd.DataFrame] = None) -> Dict[str, dict]:
    """Builds every OOD corpus, writing corpus_ood_<species>.parquet and
    stratum_counts_ood_<species>.json. Returns {species: {"corpus":..., "stratum_counts":...}}."""
    out = {}
    for species, tsv in (ood_proteomes or {}).items():
        corpus_path = paths.ROOT / f"corpus_ood_{species}.parquet"
        counts_path = paths.ROOT / f"stratum_counts_ood_{species}.json"
        if not force and corpus_path.exists() and counts_path.exists():
            corpus = pd.read_parquet(corpus_path)
            stratum_counts = json.loads(counts_path.read_text())
            print(f"[M2-OOD:{species}] CACHE HIT -- reused {len(corpus)} proteins.")
        else:
            corpus, stratum_counts = build_ood_corpus(tsv, species, cfg, paths)
            # [N1-CL#51] Flag proteins with a human homolog, so a homology-corrected ("strict")
            # OOD analysis is available alongside the full one.
            hom_report = {"applied": False, "reason": "no human corpus supplied"}
            if human_corpus is not None and len(human_corpus):
                corpus, hom_report = flag_human_homologs(
                    corpus, human_corpus, cfg.cdhit_identity,
                    tmp_dir=paths.TEMP / f"cdhit2d_{species}")
            corpus.to_parquet(corpus_path, index=False)
            counts_path.write_text(json.dumps(stratum_counts, indent=2))
            (paths.ROOT / f"homology_report_ood_{species}.json").write_text(
                json.dumps(hom_report, indent=2, default=str))
        hom_path = paths.ROOT / f"homology_report_ood_{species}.json"
        out[species] = {"corpus": corpus, "stratum_counts": stratum_counts,
                        "homology_report": (json.loads(hom_path.read_text())
                                            if hom_path.exists() else {"applied": False})}
    return out


def run_m2(swissprot_tsv: Path, m1_raw_paths: Dict[str, Path], parsers: Dict[str, callable],
           cfg, paths, tmp_dir: Path = None, force: bool = False) -> Tuple[pd.DataFrame, dict, dict]:
    """corpus.parquet + stratum_counts.json + split_manifest.json are cached and invalidated
    together as one unit, since they must stay mutually consistent."""
    raw_hashes = {label: sha256_file(p) for label, p in m1_raw_paths.items() if p.exists()}
    expected_fp = config_fingerprint({
        "swissprot_sha256": sha256_file(swissprot_tsv) if swissprot_tsv.exists() else None,
        "raw_source_hashes": raw_hashes,
    }, cfg=cfg)

    if not force:
        hit = find_cached("corpus.parquet")
        if hit is not None:
            fp_sidecar = hit.with_suffix(hit.suffix + ".fp")
            if fp_sidecar.exists() and fp_sidecar.read_text().strip() == expected_fp:
                src_dir = hit.parent
                for fname, dest in [("corpus.parquet", paths.CORPUS),
                                     ("stratum_counts.json", paths.STRATUM_COUNTS),
                                     ("split_manifest.json", paths.SPLIT_MANIFEST)]:
                    src = src_dir / fname
                    if src != dest:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(src, dest)
                if hit != paths.CORPUS:
                    shutil.copy2(fp_sidecar, paths.CORPUS.with_suffix(paths.CORPUS.suffix + ".fp"))
                corpus = pd.read_parquet(paths.CORPUS)
                gate_report = json.loads(paths.SPLIT_MANIFEST.read_text())["gate_report"]
                t3_path = paths.TABLES / "T3_split_composition.csv"
                t3_src = src_dir / "tables" / "T3_split_composition.csv"
                if t3_src.exists() and t3_src != t3_path:
                    t3_path.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(t3_src, t3_path)
                T3_split = pd.read_csv(t3_path) if t3_path.exists() else pd.DataFrame()
                print(f"[M2] CACHE HIT ({'input_dataset' if paths.INPUT in hit.parents else 'working'}): "
                      f"reused corpus.parquet + stratum_counts.json + split_manifest.json, "
                      f"skipped CD-HIT on {len(corpus)} proteins.")
                return corpus, gate_report, {"cached": True, "fingerprint": expected_fp, "T3_split": T3_split}
            print("[M2] found a cached corpus.parquet but its fingerprint is stale "
                  "(config or raw sources changed) -- rebuilding, including CD-HIT.")

    print("[M2] no valid cache -- building corpus from scratch (this runs CD-HIT).")
    corpus, gate_report, T3_split = build_corpus(swissprot_tsv, m1_raw_paths, parsers, cfg, paths, tmp_dir=tmp_dir)
    stratum_counts = compute_stratum_counts(corpus, cfg.stratum_residues)
    save_corpus_artefacts(corpus, stratum_counts, gate_report, paths, cfg, fp=expected_fp)
    paths.TABLES.mkdir(parents=True, exist_ok=True)
    T3_split.to_csv(paths.TABLES / "T3_split_composition.csv", index=False)
    return corpus, gate_report, {"cached": False, "fingerprint": expected_fp, "T3_split": T3_split}


In [8]:
# ---------------------------------------------------------------------------
# M2 -- raw type-hit table uses M1's actual resolved paths (never a hard-coded path).
# [N1-CL#40] Feeds EVERY positive-bearing source, not a three-source subset. The first real run
# stratified on only 25 types (CPLM's 24 + dbPTM N-glyco + O-GlcNAc) -- silently excluding
# phosphorylation_Y (~35,600 sites), methylation_R (~4,150), sulfoxidation (~3,900), the entire
# C stratum, and N-GlycositeAtlas's ~25,200 N-glyco sites. Since CL#26 widened stratification to
# "all canonical types", feeding it a narrow subset defeated the change: the split was balanced
# on a fraction of the data it needs to balance.
# ---------------------------------------------------------------------------
swissprot_path = resolved["swissprot"]
m1_raw_paths = {"cplm_human": resolved["cplm"][CPLM_SPECIES_HUMAN]}
parsers = {"cplm_human": lambda p: parse_cplm_zip(p)[["accession", "position", "type"]]}

if not MOCK_MODE:
    # All dbPTM types. `t=_t` default-binds the loop variable -- a bare closure would capture
    # the final value and label every file with the last PTM type.
    for _t, _p in resolved.get("dbptm", {}).items():
        _key = f"dbptm_{_t.replace(' ', '_').replace('/', '_')}"
        m1_raw_paths[_key] = _p
        parsers[_key] = (lambda p, t=_t:
                         parse_dbptm_flatfile(p, t)[["accession", "position", "type"]])

    if "Dataset-I" in resolved.get("oglcnac_atlas", {}):
        m1_raw_paths["oglcnac_dataset_i"] = resolved["oglcnac_atlas"]["Dataset-I"]
        parsers["oglcnac_dataset_i"] = (lambda p:
                                        parse_oglcnac_atlas_flatfile(p)[["accession", "position", "type"]])

    if "glycosite_atlas" in resolved:
        m1_raw_paths["glycosite_atlas"] = resolved["glycosite_atlas"]
        parsers["glycosite_atlas"] = (lambda p:
                                      parse_glycosite_atlas_flatfile(p)[["accession", "position", "type"]])

    # qPTM is registered for CACHE-FINGERPRINT purposes only -- deliberately NO parser, so it
    # does not drive stratification. It is a separate replication label set (CF-26), not part of
    # the main label population the split must balance. M3 reads its path from here.
    if "qptm" in resolved:
        m1_raw_paths["qptm"] = resolved["qptm"]

corpus, gate_report, m2_cache_info = run_m2(
    swissprot_path, m1_raw_paths, parsers, CFG, Paths, tmp_dir=Paths.TEMP / "cdhit",
    force=FORCE_REBUILD,
)
stratum_counts = json.loads(Paths.STRATUM_COUNTS.read_text())

print(f"M2: {len(corpus)} proteins, {corpus['cluster_id'].nunique()} clusters "
      f"({'CACHED' if m2_cache_info['cached'] else 'built fresh'})")
print(f"    split counts: {corpus['split'].value_counts().to_dict()}")
print(f"    gate (15k-20k proteins): {gate_report['gate_protein_count_15k_20k']}")
show_json(stratum_counts, "stratum_counts (g_ell -- the background denominator per stratum):")

# [N1-CL#42] CD-HIT diagnostics. A collapsed clustering (one giant cluster) and a no-op
# clustering (all singletons) both yield a plausible cluster COUNT -- only the size distribution
# distinguishes them.
show_json(cluster_size_report(corpus), "CD-HIT cluster diagnostics (40% identity):")
show_json(gate_report, "M2 gate report:")

# [N1-CL#26] Per-type split composition -- iterative stratification is a greedy heuristic, so the
# ACHIEVED holdout fraction per type is the only honest way to judge whether it worked.
T3_split = m2_cache_info.get("T3_split")
if T3_split is not None and len(T3_split):
    print(f"\nT3 -- split composition ({gate_report.get('n_types_stratified_on', 0)} types stratified on, "
          f"target holdout={CFG.holdout_fraction}, worst deviation="
          f"{gate_report.get('worst_abs_deviation_from_target')}):")
    show_table(T3_split)
    _bad = T3_split[T3_split["abs_deviation_from_target"] > 0.10]
    if len(_bad):
        print(f"NOTE: {len(_bad)} type(s) deviate >10pp from the target holdout fraction. "
              f"Expected for types with very few clusters (a single cluster can dominate); "
              f"check whether any PRIMARY type is affected before proceeding.")

# [N1-CL#25] OOD corpora (mouse / rat / yeast) -- no CD-HIT, no split; held out by construction.
ood_corpora = run_m2_ood(resolved.get("ood_proteomes", {}), CFG, Paths, force=FORCE_REBUILD,
                         human_corpus=corpus)
# [N1-CL#51] Cross-species homology to the human corpus -- flagged, not deleted, so both the full
# and the homology-corrected ("strict") OOD analyses stay available.
show_table(pd.DataFrame([
    dict(species=_s, **{k: v for k, v in (_d.get("homology_report") or {}).items()})
    for _s, _d in ood_corpora.items()
]), f"OOD cross-species homology vs the human corpus (cd-hit-2d at {CFG.cdhit_identity}):")
show_table(pd.DataFrame([
    dict(species=_s, n_proteins=_d["stratum_counts"]["n_proteins"],
         total_residues=_d["stratum_counts"]["total_residues"],
         **{f"g_{_k}": _v for _k, _v in _d["stratum_counts"].items()
            if _k not in ("n_proteins", "total_residues")})
    for _s, _d in ood_corpora.items()
]), "OOD corpora -- per-species background (g_ell) counts:")


[M2] no valid cache -- building corpus from scratch (this runs CD-HIT).


/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


M2: 18128 proteins, 11909 clusters (built fresh)
    split counts: {'discovery': 14384, 'holdout': 3744}
    gate (15k-20k proteins): True

stratum_counts (g_ell -- the background denominator per stratum):
{
  "K": 425120,
  "ST": 983282,
  "N": 263374,
  "C": 179280,
  "R": 432208,
  "Y": 209350,
  "E": 514269,
  "M": 165084,
  "W": 96344,
  "Q": 345562,
  "total_residues": 7474423,
  "n_proteins": 18128
}

CD-HIT cluster diagnostics (40% identity):
{
  "n_proteins": 18128,
  "n_clusters": 11909,
  "mean_cluster_size": 1.522,
  "median_cluster_size": 1,
  "max_cluster_size": 160,
  "n_singleton_clusters": 9110,
  "pct_proteins_in_singletons": 50.25,
  "size_distribution": {
    "1": 9110,
    "2": 1718,
    "3": 564,
    "4": 230,
    "5": 104,
    "6": 36,
    "7": 37,
    "8": 21,
    "9": 16,
    "10": 9,
    "11": 9,
    "12": 5,
    "13": 4,
    "14": 7,
    "15": 4
  },
  "largest_clusters": {
    "6264": 160,
    "219": 136,
    "6392": 118,
    "9269": 48,
    "720": 43,
    "

[M2-OOD:mouse] 15280 proteins, 6,492,949 residues


[M2-OOD:rat] 7546 proteins, 3,133,364 residues


[M2-OOD:yeast] 6229 proteins, 2,302,202 residues


[M2-OOD:ecoli] 4474 proteins, 1,315,770 residues

OOD cross-species homology vs the human corpus (cd-hit-2d at 0.4):
species  applied  identity  n_ood_proteins  n_with_human_homolog  n_without_human_homolog  pct_with_human_homolog  n_too_short_to_compare
  mouse     True       0.4           15280                 12031                     3249                   78.74                       0
    rat     True       0.4            7546                  6163                     1383                   81.67                       0
  yeast     True       0.4            6229                   609                     5620                    9.78                       0
  ecoli     True       0.4            4474                   314                     4160                    7.02                       0

OOD corpora -- per-species background (g_ell) counts:
species  n_proteins  total_residues    g_K   g_ST    g_N    g_C    g_R    g_Y    g_E    g_M   g_W    g_Q
  mouse       15280         64929

## M3 -- Label Construction: the Filtering Cascade

Isoform recovery -> residue-aware canonicalisation -> evidence -> site validation -> ambiguity masking -> dedup -> minimum frequency -> split inheritance. Plus qPTM replication labels, OOD labels, gold-negative contradiction filtering, and the cross-source agreement table.

In [9]:
"""
M3 -- Label construction: the filtering cascade
================================================
Inputs: M1 databases (raw site tables), M2 corpus (sequences, cluster_id, split).
Emits: labels_stratified.parquet, labels_naive.parquet, exclusion_mask.parquet,
       T1 (dataset provenance), T2 (label composition), F2 (dataset composition figure).
Gate: N-glycosylation, O-GlcNAcylation, succinylation must each clear >=1000 headline-bar sites.
Cost: hours, CPU only.

Six steps, each recording counts (-> T1), executed in order:
  1. Evidence tier      -- keep MS/MS + low-throughput experimental; drop "by similarity" etc.
  2. Site-mapping        -- residue at (accession,position) must match the type's chemistry
  3. Ambiguity masking    -- O-GlcNAcAtlas Dataset-II (binary) + CPLM LP classes II/III (continuous)
  4. Cross-source dedup   -- merge on (accession,position,type); source_count = n sources agreeing
  5. Minimum frequency    -- <200/stratum -> pooled to "other PTM"; >=1000 -> headline-eligible
  6. Redundancy           -- inherit cluster_id/split from M2 (cluster-level, not protein-level)

labels_stratified vs labels_naive: the underlying positive SITES are identical in both --
what differs is the grouping key for Step 5's pooling decision. `stratified` pools rare types
within their residue stratum; `naive` pools across the whole corpus regardless of chemistry. For
all three headline types (each occupies exactly one stratum) the two are identical; the
distinction only bites for rare cross-stratum pooling. Background population (g_ell) is not
stored in these files -- computed at query time downstream from M2's stratum_counts.json.
"""
from __future__ import annotations
import json
import shutil
from collections import defaultdict
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import pandas as pd

# Paths/CFG/fingerprint/find_cached/config_fingerprint come from 00_config.

# Chemistry ground truth: which residue identity each PTM type can occur on. Used by Step 2
# (site-mapping validation). Derived from cfg.ptm_type_to_stratum rather than hand-maintained,
# so every type in that dict is automatically covered here too.
PTM_VALID_RESIDUES: Dict[str, set] = {
    t: CFG.stratum_residues[s] for t, s in CFG.ptm_type_to_stratum.items()
    if s in CFG.stratum_residues
}

EVIDENCE_KEEP = {"ms/ms", "low-throughput", "experimental", "mass spectrometry"}
EVIDENCE_DROP = {"by similarity", "potential", "probable", "text-mining"}


# ---------------------------------------------------------------------------
# Type canonicalisation, PTM_TYPE_ALIASES, normalize_ptm_type: moved to 00_config.py
# [N1-CL#17] -- M_E needs the same mapping before M3 runs. canonicalize_types (below) is the
# only piece that's genuinely M3-specific (it's the cascade step, not the mapping itself).
# ---------------------------------------------------------------------------
def canonicalize_types(df: pd.DataFrame, cfg, corpus_seq: Optional[Dict[str, str]] = None
                        ) -> Tuple[pd.DataFrame, dict]:
    """[N1-CL#20] Now residue-aware. When `corpus_seq` is supplied, the real amino acid at each
    site is looked up FIRST and passed to normalize_ptm_type, so multi-chemistry source labels
    ("Phosphorylation", "Methylation", qPTM's bare "Glycosylation") resolve to the right sibling
    type instead of being collapsed onto one and then silently dropped by step2. Sites whose
    accession isn't in the corpus, or whose position is out of range, resolve with residue=None
    (alias-only) -- they're rejected downstream by step2 anyway, as before."""
    df = df.copy()
    known = set(cfg.ptm_type_to_stratum)

    if corpus_seq is not None:
        residues = []
        for acc, pos in zip(df["accession"].astype(str), df["position"]):
            seq = corpus_seq.get(acc)
            if seq is None or pd.isna(pos):
                residues.append(None); continue
            p = int(pos)
            residues.append(seq[p - 1] if 1 <= p <= len(seq) else None)
        # [N1-CL#38] Memoised on (raw_type, residue). normalize_ptm_type's known_types fallback
        # scans ~50 canonical keys per call; qPTM alone reaches ~5.4M rows after the human +
        # reliability filters, so the uncached form is ~270M string comparisons. The distinct
        # (type, residue) space is tiny (source types x 20 residues), so this is a pure win.
        _resolve_cache: Dict[tuple, str] = {}
        resolved_types = []
        for t, r in zip(df["type"], residues):
            key = (t, r)
            hit = _resolve_cache.get(key)
            if hit is None:
                hit = normalize_ptm_type(t, known_types=known, residue=r)
                _resolve_cache[key] = hit
            resolved_types.append(hit)
        df["type"] = resolved_types
    else:
        df["type"] = df["type"].map(lambda t: normalize_ptm_type(t, known_types=known))

    # [N1-CL#39] Two DIFFERENT kinds of unmapped row, which must not be conflated:
    #   (a) residue-unresolved -- the raw label IS in PTM_TYPE_RESIDUE_RESOLVED, but this row's
    #       residue could not be determined (accession absent from the corpus, position out of
    #       range) or is outside that label's known chemistries. Expected and benign: step2 would
    #       drop these anyway for exactly the same reason. DROPPED here, counted, not raised.
    #   (b) genuinely unknown spelling -- no table knows this label at all. Still RAISES, so a new
    #       source spelling can never silently vanish.
    # Conflating them is what crashed the real run: 8,009 qPTM "Glycosylation" rows whose residue
    # was unresolvable aborted the whole cascade. Only qPTM hit this because "glycosylation" has
    # no plain-alias fallback (unlike "phosphorylation"/"methylation") -- by design, since a bare
    # "Glycosylation" with an unknown residue genuinely cannot be assigned to N- or O-.
    unknown_mask = ~df["type"].isin(known)
    residue_resolved_labels = set(PTM_TYPE_RESIDUE_RESOLVED)
    raw_keys = df["type"].map(_normalize_type_key) if not unknown_mask.any() else \
        pd.Series([_normalize_type_key(t) for t in df["type"]], index=df.index)
    resolvable_mask = unknown_mask & raw_keys.isin(residue_resolved_labels)
    n_residue_unresolved = int(resolvable_mask.sum())
    if n_residue_unresolved:
        df = df[~resolvable_mask]
        unknown_mask = unknown_mask[~resolvable_mask]

    unknown_types = sorted(df.loc[unknown_mask, "type"].unique().tolist())
    return df, {"n_rows": len(df), "n_unmapped_rows": int(unknown_mask.sum()),
                "n_residue_unresolved_dropped": n_residue_unresolved,
                "unmapped_types": unknown_types, "residue_aware": corpus_seq is not None}


# ---------------------------------------------------------------------------
# Step 1 -- Evidence tier
# ---------------------------------------------------------------------------
def step1_evidence_filter(df: pd.DataFrame, evidence_col: str = "evidence") -> Tuple[pd.DataFrame, dict]:
    if evidence_col not in df.columns:
        # No per-site evidence field (e.g. CPLM's curated events) -> already filtered at the
        # database level, retained as-is.
        return df.copy(), {"n_in": len(df), "n_dropped": 0, "n_out": len(df), "reason": "no evidence field"}
    ev = df[evidence_col].astype(str).str.lower().str.strip()
    keep_mask = ev.apply(lambda s: any(k in s for k in EVIDENCE_KEEP) and not any(d in s for d in EVIDENCE_DROP))
    out = df[keep_mask].copy()  # unrecognised evidence strings default to DROP (conservative)
    return out, {"n_in": len(df), "n_dropped": int((~keep_mask).sum()), "n_out": len(out)}


# ---------------------------------------------------------------------------
# Step 2 -- Site-mapping validation
# ---------------------------------------------------------------------------
def step2_validate_sites(df: pd.DataFrame, corpus_seq: Dict[str, str],
                          valid_residues: Dict[str, set] = PTM_VALID_RESIDUES) -> Tuple[pd.DataFrame, dict]:
    """Residue at (accession, position) [1-indexed, UniProt convention] must match the type's
    known chemistry -- catches off-by-one / stale-isoform errors that are otherwise silent."""
    from tqdm.auto import tqdm
    accs = df["accession"].values
    poss = df["position"].values
    types = df["type"].values
    keep = []
    fail_reasons = defaultdict(int)
    for acc, pos, t in tqdm(zip(accs, poss, types), total=len(accs), desc="[M3] Site validation",
                           unit="site", leave=False):
        seq = corpus_seq.get(acc)
        if seq is None:
            keep.append(False); fail_reasons["accession_not_in_corpus"] += 1; continue
        pos = int(pos)
        if not (1 <= pos <= len(seq)):
            keep.append(False); fail_reasons["position_out_of_range"] += 1; continue
        residue = seq[pos - 1]
        valid = valid_residues.get(t)
        if valid is None:
            keep.append(True); continue  # unknown type -> chemistry unchecked, not rejected here
        if residue not in valid:
            keep.append(False); fail_reasons["residue_mismatch"] += 1; continue
        keep.append(True)
    out = df[pd.Series(keep, index=df.index)].copy()
    return out, {"n_in": len(df), "n_dropped": len(df) - len(out), "n_out": len(out),
                 "failure_breakdown": dict(fail_reasons)}


# ---------------------------------------------------------------------------
# Step 3 -- Ambiguity masking (source-specific implementations, common treatment)
# ---------------------------------------------------------------------------
def step3c_oglycosylation_ambiguity_mask(positives: pd.DataFrame,
                                          oglcnac_sites: set) -> pd.DataFrame:
    """[N1-CL#21] O-GlcNAcylation is a SUBTYPE of O-glycosylation, not a sibling -- dbPTM's broad
    "O-linked Glycosylation" bucket contains both mucin-type O-glycosylation AND O-GlcNAc.
    Measured on real data: 34.9% of dbPTM's O-linked sites are also in O-GlcNAcAtlas.

    The ~65% that are NOT in the Atlas are the problem: labelled O-glycosylation, they would sit
    in the O-GlcNAcylation BACKGROUND as apparent negatives, but an unknown share of them are
    genuinely O-GlcNAc the Atlas hasn't catalogued. A latent that truly detects O-GlcNAc would
    fire on them too, shrinking the positive-vs-background gap and biasing effect sizes toward
    null -- costing statistical power on a headline type.

    Fix: mask them for O-GlcNAcylation ONLY (same treatment Dataset-II ambiguous sites get).
    Their own O-glycosylation labels are untouched; they simply stop being usable as O-GlcNAc
    negatives. Sites already confirmed in the Atlas are excluded from the mask -- they're known
    positives, not ambiguous."""
    o_glyco = positives[positives["type"] == "O-glycosylation"]
    if not len(o_glyco):
        return pd.DataFrame(columns=["accession", "position", "type", "mask_reason"])
    keys = list(zip(o_glyco["accession"].astype(str), o_glyco["position"]))
    rows = [{"accession": a, "position": int(p), "type": "O-GlcNAcylation",
             "mask_reason": "dbptm_o_linked_glyco_may_be_uncatalogued_o_glcnac"}
            for a, p in keys if (a, int(p)) not in oglcnac_sites]
    return pd.DataFrame(rows, columns=["accession", "position", "type", "mask_reason"])


def step3a_oglcnac_mask(dataset_ii: pd.DataFrame) -> pd.DataFrame:
    """3a: every candidate S/T in a Dataset-II (ambiguous) peptide window -> exclusion mask row.
    Uses the annotated position (real Dataset-II ships the explicit candidate list per peptide;
    the reported position is used directly here as the conservative candidate)."""
    rows = []
    for _, r in dataset_ii.iterrows():
        rows.append({"accession": r["accession"], "position": int(r["position"]), "type": "O-GlcNAcylation",
                     "mask_reason": "oglcnac_atlas_dataset_ii_ambiguous"})
    return pd.DataFrame(rows)


# [N1-CL#50] step3b_cplm_lp_mask and _lp_to_class REMOVED -- CPLM's bulk download has no `lp`
# column (100% null, measured), so the Class II/III mask was unreachable code. CPLM sites are now
# treated exactly as they always effectively were: all retained as positives, with no
# confidence filter, which is a property of the source and is documented as such.


# ---------------------------------------------------------------------------
# Step 4 -- Cross-source deduplication
# ---------------------------------------------------------------------------
def step4_dedup(dfs: List[pd.DataFrame],
                 source_labels: Optional[Sequence[str]] = None) -> pd.DataFrame:
    """Merge on (accession, position, type).

    [N1-CL#62] BUGFIX. This returned `combined.groupby(key).size()` -- a count of contributing
    ROWS -- under the name `source_count`, documented as "number of independent sources". A source
    that lists one site under several PMIDs or peptides contributes once per row, so the field
    reached 52 for N-glycosylation and 25 for ubiquitination, which only two sources can supply.
    `n_sources_reporting` in `build_site_agreement` was computing the intended quantity correctly
    all along, from the same pre-dedup frames; the two simply disagreed and nothing compared them.

    Now emits BOTH, each named for what it measures:
      * `source_count`     -- distinct sources reporting the site. Max 2 on the human corpus.
      * `source_row_count` -- contributing rows. The old `source_count`; kept because it is a
                              usable proxy for how heavily a site is attested WITHIN a source.

    `source_labels` gives each row's source. Accepts one label per FRAME (len == len(dfs)) or one
    per ROW (len == total rows). If omitted, each frame in `dfs` is treated as one source -- which
    is correct for the human path, where `filtered_positives` is exactly one frame per source, but
    NOT for a caller that pre-concatenates sources into a single frame. Such a caller must pass
    per-row labels or it will get source_count == 1 everywhere.
    """
    combined = pd.concat(dfs, ignore_index=True, sort=False)
    key_cols = ["accession", "position", "type"]

    if source_labels is None:
        src = np.repeat(np.arange(len(dfs)), [len(d) for d in dfs])
    elif len(source_labels) == len(dfs) and len(dfs) != len(combined):
        src = np.repeat(np.asarray(source_labels, dtype=object), [len(d) for d in dfs])
    elif len(source_labels) == len(combined):
        src = np.asarray(source_labels, dtype=object)
    else:
        raise ValueError(
            f"source_labels has length {len(source_labels)}; expected {len(dfs)} (one per frame) "
            f"or {len(combined)} (one per row)")

    combined = combined.copy()
    combined["_src"] = src
    g = combined.groupby(key_cols)
    grouped = g["_src"].nunique().reset_index(name="source_count")
    grouped["source_row_count"] = g.size().reset_index(name="_n")["_n"].values
    combined = combined.drop(columns="_src")
    aux_cols = [c for c in combined.columns if c not in key_cols]  # first non-null pmid/lp_class/evidence
    first_vals = combined.groupby(key_cols)[aux_cols].first().reset_index()
    out = grouped.merge(first_vals, on=key_cols, how="left")
    return out


# ---------------------------------------------------------------------------
# Step 5 -- Minimum frequency (pool rare types, flag headline-eligible)
# ---------------------------------------------------------------------------
def step5_min_frequency(df: pd.DataFrame, cfg, group_by_stratum: bool,
                         type_to_stratum: Dict[str, str]) -> Tuple[pd.DataFrame, dict]:
    df = df.copy()
    df["stratum"] = df["type"].map(type_to_stratum)
    if group_by_stratum:
        counts = df.groupby(["stratum", "type"]).size().reset_index(name="n")
    else:
        counts = df.groupby(["type"]).size().reset_index(name="n")
        counts["stratum"] = "ALL"
        # [N1-CL#45] PRE-EXISTING BUG. "ALL" was set only on the COUNTS table, never on the data,
        # so labels_naive kept the real per-residue stratum. Combined with the fact that every
        # type maps to exactly ONE stratum -- making groupby(["stratum","type"]) and
        # groupby(["type"]) produce identical counts -- labels_naive came out a byte-identical
        # copy of labels_stratified. The real run proved it: both pooled the same 23 types and
        # both had 498,619 rows. The naive-vs-stratified comparison (plan section 0.4) was
        # therefore comparing a file against itself and could only ever agree.
        # The stratum column is what tells downstream code which background population to use,
        # so setting it to "ALL" here is what actually makes the naive variant naive: background
        # = every residue in the corpus, not only residues of the target type.
        df["stratum"] = "ALL"

    rare = counts[counts["n"] < cfg.min_sites_per_type]
    headline = set(counts.loc[counts["n"] >= cfg.min_sites_headline, "type"])

    rare_types = set(rare["type"])
    df["type_pooled"] = df["type"].where(~df["type"].isin(rare_types), df["stratum"] + "::other_PTM")
    df["headline_eligible"] = df["type"].isin(headline)

    report = {
        "n_types_before_pooling": int(counts["type"].nunique()),
        "n_types_pooled_to_other": int(len(rare_types)),
        "pooled_types": sorted(rare_types),
        "headline_eligible_types": sorted(headline),
        "counts_per_type": counts.to_dict(orient="records"),
    }
    return df, report


# ---------------------------------------------------------------------------
# Step 6 -- Redundancy (inherit cluster_id / split from M2 corpus)
# ---------------------------------------------------------------------------
def step6_inherit_split(df: pd.DataFrame, corpus: pd.DataFrame) -> pd.DataFrame:
    lookup = corpus.set_index("accession")[["cluster_id", "split"]]
    out = df.join(lookup, on="accession")
    n_missing = out["split"].isna().sum()
    if n_missing:
        raise RuntimeError(f"{n_missing} labelled sites have no corpus split assignment "
                            f"(accession not in M2 corpus)")
    return out


# ---------------------------------------------------------------------------
# Negatives + crosstalk (metadata only, not materialised as a giant table)
# ---------------------------------------------------------------------------
def strip_isoform_suffix(df: pd.DataFrame, corpus_seq: Dict[str, str],
                          accession_col: str = "accession") -> Tuple[pd.DataFrame, dict]:
    """[N1-CL#27] UniProt isoform accessions (P12345-2) never match the canonical corpus. The
    real isoform audit found 3,959 CPLM accessions (33.5% of its unmatched set) and 7,995 dbPTM
    accessions recoverable by stripping the suffix.

    CAVEAT, deliberately not hidden: the reported position refers to the ISOFORM's numbering, so
    stripping makes it a canonical-sequence position that may be wrong wherever the isoform
    differs. step2's chemistry check is the safety net (a shifted position usually lands on the
    wrong residue and is dropped), but a shifted position CAN coincidentally land on a valid
    residue. Every recovered row is therefore flagged `from_isoform_accession=True` so it can be
    excluded in a sensitivity re-run -- these should never silently become indistinguishable from
    directly-matched sites."""
    df = df.copy()
    accs = df[accession_col].astype(str)
    is_isoform = accs.str.contains("-") & accs.str.rsplit("-", n=1).str[-1].str.isdigit()
    stripped = accs.str.rsplit("-", n=1).str[0]
    recoverable = is_isoform & ~accs.isin(corpus_seq.keys()) & stripped.isin(corpus_seq.keys())
    df["from_isoform_accession"] = recoverable
    df.loc[recoverable, accession_col] = stripped[recoverable]
    return df, {"n_rows": len(df), "n_isoform_rows": int(is_isoform.sum()),  # [N1-CL#44] ROWS, not unique accessions
                "n_rows_recovered_by_stripping": int(recoverable.sum())}


def build_site_agreement(positives: pd.DataFrame, source_of_row: pd.Series) -> pd.DataFrame:
    """[N1-CL#28] Per-site cross-source agreement record, for the annotation-discrepancy analysis.

    Beyond `source_count`, this records which sources reported a site AND which sources covered
    that protein at all. That distinction is the whole point: a site absent from a source that
    never looked at the protein is a coverage gap, while a site absent from a source that
    annotated the same protein elsewhere is a genuine disagreement -- far stronger evidence. The
    real overlap data shows how common this is: only 26.9% of dbPTM's N-glyco sites appear in
    N-GlycositeAtlas, and only 50.3% the other way.

    Emitted so N4 can adjudicate discrepancies using latent activations (a site reported by one
    source that a validated PTM latent fires strongly on is evidence the site is real)."""
    df = positives.copy()
    df["_source"] = source_of_row.values

    # [N1-CL#43] "Covering" is keyed on (source, accession, TYPE), not (source, accession).
    # The type-blind version was meaningless: most sources are single-type files
    # (dbptm_Phosphorylation, dbptm_Acetylation, ...), so a phospho file "covered" any protein it
    # had a phospho site in -- and was then counted as DISAGREEING about a succinylation site it
    # could never have reported. The first real run showed the damage: 483,597 of 498,619 sites
    # (97%) flagged as disagreement, and 2-hydroxyisobutyrylation averaging 8.0 "disagreeing"
    # sources when only CPLM reports that type at all.
    # Correct semantics: a source covers (protein, type) if it reports >=1 site of THAT type in
    # THAT protein. Only then does the absence of a specific site mean the sources disagree.
    covered_protein_types = {}
    for (src, ptype), sub in df.groupby(["_source", "type"]):
        covered_protein_types.setdefault(ptype, {})[src] = set(sub["accession"])

    rows = []
    for (acc, pos, ptm_type), sub in df.groupby(["accession", "position", "type"]):
        reporting = sorted(set(sub["_source"]))
        covering = sorted([s for s, prots in covered_protein_types.get(ptm_type, {}).items()
                           if acc in prots])
        disagreeing = [s for s in covering if s not in reporting]
        rows.append({
            "accession": acc, "position": int(pos), "type": ptm_type,
            "n_sources_reporting": len(reporting), "sources_reporting": ",".join(reporting),
            "n_sources_covering_protein_type": len(covering),
            "n_sources_covering_but_not_reporting": len(disagreeing),
            "sources_disagreeing": ",".join(disagreeing),
            "agreement_ratio": round(len(reporting) / len(covering), 3) if covering else float("nan"),
        })
    return pd.DataFrame(rows)


def build_gold_negatives_nglyde(glyde_30pct: pd.DataFrame,
                                 positive_sites: Optional[pd.DataFrame] = None
                                 ) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """The N-GlyDE non-glycosylated sequons -- the only Gold-tier negatives in the project.

    [N1-CL#29] Removes sites CONTRADICTED by an annotated N-glycosylation positive elsewhere
    (checked against the full deduplicated positive set, not per-source). Measured on the real
    run: 222 of 1,310 (16.95%) contradicted, 1,088 clean negatives remain -- see N1_CHANGELOG.md.

    Returns (clean_negatives, contradicted_negatives, report). The contradicted set is kept and
    written out, not discarded, so the contamination rate stays reportable."""
    neg = glyde_30pct[glyde_30pct["Label"].str.lower().str.contains("non")].copy()
    neg = neg.rename(columns={"UniProt Accession": "accession", "Position": "position"})
    neg["type"] = "N-glycosylation"
    neg["tier"] = "gold"
    neg = neg[["accession", "position", "type", "tier"]]

    if positive_sites is None or not len(positive_sites):
        return neg, neg.iloc[0:0].copy(), {"n_gold": len(neg), "n_contradicted": 0,
                                            "contradiction_check_run": False}
    pos_n = positive_sites[positive_sites["type"] == "N-glycosylation"]
    pos_keys = set(zip(pos_n["accession"].astype(str), pos_n["position"].astype(int)))
    contradicted_mask = [(str(a), int(p)) in pos_keys
                         for a, p in zip(neg["accession"], neg["position"])]
    contradicted = neg[pd.Series(contradicted_mask, index=neg.index)]
    clean = neg[~pd.Series(contradicted_mask, index=neg.index)]
    report = {"n_gold_before": len(neg), "n_contradicted": len(contradicted),
              "n_gold_clean": len(clean),
              "pct_contradicted": round(100 * len(contradicted) / len(neg), 2) if len(neg) else 0.0,
              "contradiction_check_run": True}
    if len(contradicted):
        print(f"[M3] Gold-tier negatives: removed {len(contradicted)}/{len(neg)} "
              f"({report['pct_contradicted']}%) contradicted by an annotated N-glycosylation "
              f"positive. {len(clean)} clean gold negatives remain.")
    return clean, contradicted, report


def _enrich_label_schema(labels: pd.DataFrame, corpus_seq: Dict[str, str],
                          crosstalk: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """[N1-CL#33] Adds the per-site columns the implementation plan's labels_stratified schema
    specifies but the cascade wasn't emitting: `residue` (the actual amino acid, so downstream
    code never has to re-derive it from the corpus) and `crosstalk` (this residue also carries
    another PTM type).

    NOT added here, deliberately -- see N1_CHANGELOG.md #33: `label`, `negative_tier` and
    `corpus`. Those three only make sense on a materialised positive+negative row set, and these
    files store POSITIVES ONLY -- exactly as the ORIGINAL pipeline did. Materialising every
    testable (residue, type) pair would run to tens of millions of rows (K stratum alone: ~375k
    lysines x ~30 K-types). The negative population is reconstructed downstream from
    corpus.parquet + stratum_counts.json + exclusion_mask.parquet + accessions_carrying_type()
    (Hard tier) + gold_negatives_nglyco.parquet (Gold tier), which is how the original's M16/M21
    validation modules already did it."""
    out = labels.copy()
    if "residue" not in out.columns:
        out["residue"] = [
            corpus_seq[a][int(p) - 1] if a in corpus_seq and 1 <= int(p) <= len(corpus_seq[a]) else None
            for a, p in zip(out["accession"].astype(str), out["position"])
        ]
    if crosstalk is not None and len(crosstalk):
        ct_keys = set(zip(crosstalk["accession"].astype(str), crosstalk["position"].astype(int)))
        out["crosstalk"] = [(str(a), int(p)) in ct_keys
                            for a, p in zip(out["accession"], out["position"])]
    else:
        out["crosstalk"] = False
    return out


def build_T4_type_survival(stage_frames: Dict[str, pd.DataFrame], cfg,
                            final_labels: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """[N1-CL#31] Per-TYPE attrition through the cascade. T1 records per-SOURCE counts, which
    cannot answer "how much of N-glycosylation survived?" -- the two questions are orthogonal,
    and only the per-source view existed.

    `stage_frames` maps a stage name to the combined (all-sources) frame at that point. A type
    that loses most of its sites at one specific step is immediately visible (real data has
    several: dbPTM Ubiquitination loses ~33% at chemistry validation, Methylation ~70%)."""
    stages = list(stage_frames)
    counts = {}
    for stage, df in stage_frames.items():
        counts[stage] = (df["type"].value_counts().to_dict()
                         if df is not None and len(df) and "type" in df.columns else {})
    all_types = sorted({t for c in counts.values() for t in c})
    rows = []
    for t in all_types:
        row = {"type": t, "stratum": cfg.ptm_type_to_stratum.get(t, "UNMAPPED")}
        for stage in stages:
            row[stage] = int(counts[stage].get(t, 0))
        first, last = row[stages[0]], row[stages[-1]]
        row["pct_surviving"] = round(100 * last / first, 1) if first else None
        row["meets_200_floor"] = last >= cfg.min_sites_per_type
        row["meets_1000_headline"] = last >= cfg.min_sites_headline
        if final_labels is not None and len(final_labels) and "type_pooled" in final_labels.columns:
            pooled_names = set(final_labels.loc[final_labels["type"] == t, "type_pooled"])
            row["pooled_into"] = next((p for p in pooled_names if "other_PTM" in str(p)), "")
        rows.append(row)
    return pd.DataFrame(rows).sort_values(stages[-1], ascending=False).reset_index(drop=True)


def build_annotation_discrepancies(exclusion_mask: pd.DataFrame,
                                    gold_contradicted: pd.DataFrame,
                                    site_agreement: pd.DataFrame) -> pd.DataFrame:
    """[N1-CL#46] One artefact gathering every site where the source databases DISAGREE, for the
    annotation-discrepancy study.

    Masking a site removes it from the statistics -- it cannot be a positive (the annotation may
    belong to a neighbour or another type) and must not be a negative (that would inject
    guaranteed false negatives). But "excluded from the contingency table" is NOT "discarded":
    these are exactly the sites worth revisiting once validated latents exist. If a latent that
    demonstrably detects O-GlcNAc fires hard on a dbPTM O-linked site the Atlas never catalogued,
    that is positive evidence the site IS O-GlcNAc -- a finding, not noise.

    Three classes, each with a different question a latent can answer:

    | class | question |
    |---|---|
    | containment_uncatalogued_oglcnac | is this O-linked site actually O-GlcNAc? |
    | gold_negative_contradicted | is this "verified unmodified" site actually modified? |
    | single_source_despite_coverage | did the other source MISS this site, or is it spurious? |
    """
    frames = []

    if exclusion_mask is not None and len(exclusion_mask):
        containment = exclusion_mask[
            exclusion_mask["mask_reason"] == "dbptm_o_linked_glyco_may_be_uncatalogued_o_glcnac"].copy()
        if len(containment):
            containment = containment[["accession", "position", "type"]].copy()
            containment["discrepancy_class"] = "containment_uncatalogued_oglcnac"
            containment["evidence_for"] = "dbPTM O-linked Glycosylation"
            containment["evidence_against"] = "absent from O-GlcNAcAtlas"
            containment["question_for_latent"] = "does an O-GlcNAc latent fire here?"
            frames.append(containment)

    if gold_contradicted is not None and len(gold_contradicted):
        gc = gold_contradicted[["accession", "position", "type"]].copy()
        gc["discrepancy_class"] = "gold_negative_contradicted"
        gc["evidence_for"] = "annotated positive in dbPTM / N-GlycositeAtlas"
        gc["evidence_against"] = "N-GlyDE verified non-glycosylated"
        gc["question_for_latent"] = "does an N-glyco latent fire here?"
        frames.append(gc)

    if site_agreement is not None and len(site_agreement):
        dis = site_agreement[site_agreement["n_sources_covering_but_not_reporting"] > 0].copy()
        if len(dis):
            dis = dis.rename(columns={"sources_reporting": "evidence_for",
                                      "sources_disagreeing": "evidence_against"})
            dis = dis[["accession", "position", "type", "evidence_for", "evidence_against"]].copy()
            dis["discrepancy_class"] = "single_source_despite_coverage"
            dis["question_for_latent"] = "missed by the other source, or spurious?"
            frames.append(dis)

    if not frames:
        return pd.DataFrame(columns=["accession", "position", "type", "discrepancy_class",
                                      "evidence_for", "evidence_against", "question_for_latent"])
    return pd.concat(frames, ignore_index=True, sort=False)


def build_species_type_applicability(ood_labels: Dict[str, pd.DataFrame], human_labels: pd.DataFrame,
                                      cfg) -> pd.DataFrame:
    """[N1-CL#36] CF-22 species x type applicability matrix. A (species, type) pair is testable
    only when that species actually has >= cfg.ood_min_sites_per_type sites of that type.

    Required, not nice-to-have: running every type against every species produces meaningless
    nulls and uninterpretable failures. E. coli largely lacks N-glycosylation; several lysine
    acylations are far better characterised in bacteria than in yeast. Absence of a type in a
    species is a fact about biology, not a failure of the latent -- without this matrix the two
    are indistinguishable in the results.

    [N1-CL#53] Also reports `testable_strict`/`n_sites_species_strict`, the SAME test restricted
    to sites on proteins with no human homolog (`homologous_to_human == False`, from CL#51's
    cd-hit-2d flag). Needed because retention under that filter is highly uneven by species
    (mouse/rat lose ~80%+ of sites, yeast/E. coli lose ~15%) -- a pair testable in the full view
    can easily fall below the floor once restricted to genuinely novel (non-homologous) proteins,
    and `testable` alone cannot show that. Falls back to `testable_strict=None` for a species
    whose labels never got a `homologous_to_human` column (cd-hit-2d failed for that species;
    see CL#51's non-fatal handling)."""
    rows = []
    human_counts = human_labels["type"].value_counts().to_dict() if len(human_labels) else {}
    all_types = sorted(set(human_counts) | {t for df in ood_labels.values() if len(df)
                                            for t in df["type"].unique()})
    for species, df in ood_labels.items():
        counts = df["type"].value_counts().to_dict() if len(df) else {}
        has_hom_col = len(df) and "homologous_to_human" in df.columns
        strict_counts = (df.loc[~df["homologous_to_human"].fillna(False), "type"]
                         .value_counts().to_dict()) if has_hom_col else {}
        for t in all_types:
            n = int(counts.get(t, 0))
            n_strict = int(strict_counts.get(t, 0)) if has_hom_col else None
            rows.append({
                "species": species, "type": t,
                "stratum": cfg.ptm_type_to_stratum.get(t, "UNMAPPED"),
                "n_sites_species": n, "n_sites_human": int(human_counts.get(t, 0)),
                "testable": n >= cfg.ood_min_sites_per_type,
                "reason": ("ok" if n >= cfg.ood_min_sites_per_type
                           else ("absent_in_species" if n == 0 else "below_min_sites")),
                "n_sites_species_strict": n_strict,
                "testable_strict": (n_strict >= cfg.ood_min_sites_per_type
                                    if n_strict is not None else None),
            })
    return pd.DataFrame(rows)


def build_ood_labels(raw_sources: Dict[str, pd.DataFrame], ood_corpus: pd.DataFrame,
                      species: str, cfg, exclusion_mask: Optional[pd.DataFrame] = None
                      ) -> Tuple[pd.DataFrame, dict]:
    """[N1-CL#25/#35] OOD label set for one species. Runs the same cascade as the human corpus --
    isoform recovery, residue-aware canonicalisation, chemistry validation, species-specific
    ambiguity masking, dedup, minimum-frequency pooling -- and emits the same enriched schema.

    Exactly two steps are absent, and only these two:
      * NO split inheritance -- an OOD species has no discovery/holdout split (it is held out by
        construction, so there is no within-corpus boundary to protect).
      * NO CD-HIT upstream, for the same reason (see build_ood_corpus).
    Unmapped types are DROPPED rather than raising: OOD is a secondary analysis, and a species
    carrying a type the human config never needed should degrade coverage, not abort the run."""
    corpus_seq = dict(zip(ood_corpus["accession"], ood_corpus["sequence"]))
    log = {"species": species}
    if not raw_sources:
        return pd.DataFrame(), {"species": species, "note": "no OOD sources for this species"}

    # [N1-CL#62] Tag every row with its source BEFORE concatenating. This path merges all sources
    # into one frame, so without the tag step4_dedup cannot tell them apart -- which is why OOD
    # `source_count` reached 466 (yeast) and 418 (mouse) while at most 4 sources cover any species.
    _tagged = []
    for _src_name, _src_df in raw_sources.items():
        _d = _src_df.copy()
        _d["_ood_source"] = _src_name
        _tagged.append(_d)
    combined = pd.concat(_tagged, ignore_index=True, sort=False)
    log["n_raw_rows"] = len(combined)
    log["sources"] = sorted(raw_sources)

    stripped, log_iso = strip_isoform_suffix(combined, corpus_seq)
    log["step0_isoform_recovery"] = log_iso

    canon, log0 = canonicalize_types(stripped, cfg, corpus_seq=corpus_seq)
    log["step0_canonicalize"] = log0
    known = set(cfg.ptm_type_to_stratum)
    log["step0_unmapped_dropped"] = int((~canon["type"].isin(known)).sum())
    canon = canon[canon["type"].isin(known)]

    validated, log2 = step2_validate_sites(canon, corpus_seq)
    log["step2_validate"] = log2

    if exclusion_mask is not None and len(exclusion_mask):
        mask_keys = set(zip(exclusion_mask["accession"].astype(str),
                            exclusion_mask["position"].astype(int), exclusion_mask["type"]))
        keep = [(str(a), int(p), t) not in mask_keys
                for a, p, t in zip(validated["accession"], validated["position"], validated["type"])]
        n_before = len(validated)
        validated = validated[pd.Series(keep, index=validated.index)].copy()
        log["step3_ambiguity_mask"] = {"n_in": n_before, "n_masked": n_before - len(validated)}

    # [N1-CL#62] per-ROW labels: this path pre-concatenated its sources, so frame membership
    # carries no information. `_ood_source` survives the cascade because every step above returns
    # a filtered copy rather than a rebuilt frame.
    _ood_src = validated["_ood_source"] if "_ood_source" in validated.columns else None
    deduped = step4_dedup([validated.drop(columns=["_ood_source"], errors="ignore")],
                          source_labels=list(_ood_src) if _ood_src is not None else None)
    log["step4_dedup"] = {"n_in": len(validated), "n_out": len(deduped)}

    pooled, log5 = step5_min_frequency(deduped, cfg, group_by_stratum=True,
                                        type_to_stratum=cfg.ptm_type_to_stratum)
    log["step5_min_frequency"] = log5
    pooled = _enrich_label_schema(pooled, corpus_seq, crosstalk_flags(deduped))
    pooled["species"] = species
    pooled["split"] = "ood"
    return pooled, log


def crosstalk_flags(positive_sites: pd.DataFrame, source_frame: Optional[pd.DataFrame] = None,
                     source_labels: Optional[pd.Series] = None) -> pd.DataFrame:
    """Mark every (accession, position) carrying >1 type; record the type combination.

    [N1-CL#52] When `source_frame` + `source_labels` are supplied, also records WHICH sources
    reported each co-occurring type, and classifies the evidence:

    | `evidence_class` | Meaning |
    |---|---|
    | `same_source_multi_type` | at least one source reported >=2 of the co-occurring types -- that source's own curation asserts genuine co-modification |
    | `cross_source_only` | every type came from a different source, and no single source saw more than one -- consistent with co-modification, but ALSO consistent with two sources disagreeing about which modification was observed |

    Why this matters: K-stratum crosstalk runs above 90% for the acylations, and that number is
    load-bearing for interpreting any succinylation claim. Without this split, genuine competing
    acylation and inter-source disagreement are indistinguishable. `cross_source_only` does not
    prove disagreement -- single-type sources (dbPTM ships one file per type) *cannot* report a
    second type even when it is real -- but it bounds how much of the crosstalk could be
    artefactual."""
    grp = positive_sites.groupby(["accession", "position"])["type"].apply(lambda s: sorted(set(s)))
    crosstalk = grp[grp.apply(len) > 1].reset_index()
    crosstalk = crosstalk.rename(columns={"type": "types"})
    crosstalk["n_types"] = crosstalk["types"].apply(len)
    crosstalk["type_combination"] = crosstalk["types"].apply(lambda ts: " + ".join(ts))
    crosstalk["crosstalk"] = True

    if source_frame is None or source_labels is None or not len(source_frame):
        crosstalk["sources_per_type"] = None
        crosstalk["evidence_class"] = "unattributed"
        return crosstalk

    sf = source_frame.copy()
    sf["_source"] = list(source_labels)
    # (accession, position) -> {type: {sources}}
    per_site: Dict[tuple, Dict[str, set]] = {}
    for acc, pos, typ, src in zip(sf["accession"].astype(str), sf["position"],
                                   sf["type"], sf["_source"]):
        if pd.isna(pos):
            continue
        per_site.setdefault((acc, int(pos)), {}).setdefault(typ, set()).add(src)

    sources_per_type, evidence_class = [], []
    for acc, pos in zip(crosstalk["accession"].astype(str), crosstalk["position"]):
        tmap = per_site.get((acc, int(pos)), {})
        sources_per_type.append("; ".join(f"{t}={','.join(sorted(v))}" for t, v in sorted(tmap.items())))
        src_counts: Dict[str, int] = {}
        for t, srcs in tmap.items():
            for sv in srcs:
                src_counts[sv] = src_counts.get(sv, 0) + 1
        evidence_class.append("same_source_multi_type" if any(v > 1 for v in src_counts.values())
                              else "cross_source_only")
    crosstalk["sources_per_type"] = sources_per_type
    crosstalk["evidence_class"] = evidence_class
    return crosstalk


def accessions_carrying_type(positive_sites: pd.DataFrame, ptm_type: str) -> set:
    """Defines the 'Hard' negative tier: proteins with >=1 annotated site of this type."""
    return set(positive_sites.loc[positive_sites["type"] == ptm_type, "accession"])


# ---------------------------------------------------------------------------
# qPTM: a separate, independent-curation replication pipeline (CF-26)
# ---------------------------------------------------------------------------
def build_qptm_labels(qptm_raw: pd.DataFrame, corpus: pd.DataFrame, cfg,
                       exclusion_mask: Optional[pd.DataFrame] = None) -> Tuple[pd.DataFrame, dict]:
    """Reuses canonicalisation + site-chemistry validation + min-frequency pooling +
    split-inheritance (same steps as the main cascade) on qPTM's own site table alone, filtered
    to human (qPTM also covers mouse/rat/yeast). Never merged into labels_stratified/
    labels_naive -- schema-identical, kept separate for replication comparison.
    [N1-CL#11] Species filter matches "human" -- the real qPTM Organism column says "Human", not
    "Homo sapiens"; the old filter would have silently produced zero qPTM labels on real data."""
    corpus_seq = dict(zip(corpus["accession"], corpus["sequence"]))
    log = {}
    species_norm = qptm_raw["species"].astype(str).str.strip().str.lower()
    human_only = qptm_raw[species_norm.isin({"human", "homo sapiens"})].copy()
    log["step0_species_filter"] = {"n_in": len(qptm_raw), "n_human": len(human_only)}

    # [N1-CL#22] Reliability floor -- qPTM has no `evidence` column, so step1_evidence_filter is
    # a no-op on it; without this, qPTM would be the only source with NO quality filter at all.
    if "reliability" in human_only.columns and human_only["reliability"].notna().any():
        n_before = len(human_only)
        rel = pd.to_numeric(human_only["reliability"], errors="coerce")
        human_only = human_only[rel >= cfg.qptm_reliability_floor].copy()
        log["step1_reliability_floor"] = {
            "floor": cfg.qptm_reliability_floor, "n_in": n_before, "n_out": len(human_only),
            "n_dropped": n_before - len(human_only),
        }
    else:
        log["step1_reliability_floor"] = {"applied": False, "reason": "no reliability column"}

    canon, log0 = canonicalize_types(human_only, cfg, corpus_seq=corpus_seq)
    log["step0_canonicalize"] = log0
    if log0["unmapped_types"]:
        raise ValueError(f"[qptm] {log0['n_unmapped_rows']} rows have unmapped PTM type spellings: "
                         f"{log0['unmapped_types']}. Add them to PTM_TYPE_ALIASES.")

    validated, log2 = step2_validate_sites(canon, corpus_seq)
    log["step2_validate"] = log2

    deduped = validated.drop_duplicates(subset=["accession", "position", "type"]).copy()
    log["step4_dedup"] = {"n_in": len(validated), "n_out": len(deduped)}

    # [N1-CL#32] Apply the SAME ambiguity mask the main cascade uses. Previously skipped, which
    # meant a site masked as ambiguous in labels_stratified stayed an unmasked positive in the
    # qPTM replication set -- making the CF-26 comparison not apples-to-apples.
    if exclusion_mask is not None and len(exclusion_mask):
        mask_keys = set(zip(exclusion_mask["accession"].astype(str),
                            exclusion_mask["position"].astype(int),
                            exclusion_mask["type"]))
        keep = [(str(a), int(p), t) not in mask_keys
                for a, p, t in zip(deduped["accession"], deduped["position"], deduped["type"])]
        n_before = len(deduped)
        deduped = deduped[pd.Series(keep, index=deduped.index)].copy()
        log["step3_ambiguity_mask"] = {"n_in": n_before, "n_masked": n_before - len(deduped),
                                       "n_out": len(deduped)}

    pooled, log5 = step5_min_frequency(deduped, cfg, group_by_stratum=True,
                                       type_to_stratum=cfg.ptm_type_to_stratum)
    log["step5_min_frequency"] = log5

    labels_qptm = step6_inherit_split(pooled, corpus)
    log["step6_split_inherited"] = {"n_rows": len(labels_qptm)}
    labels_qptm = _enrich_label_schema(labels_qptm, corpus_seq, crosstalk_flags(deduped))
    return labels_qptm, log


def compute_qptm_dbptm_overlap(labels_qptm: pd.DataFrame, labels_stratified: pd.DataFrame) -> pd.DataFrame:
    """qPTM and dbPTM share primary literature, so the replication is partially (not fully)
    independent -- this overlap fraction tells a reader how much to trust it. Site-set overlap
    per type, both directions (asymmetric coverage is common enough that one direction alone
    would understate it)."""
    rows = []
    for t in sorted(set(labels_qptm["type"]) | set(labels_stratified["type"])):
        q_sites = set(zip(labels_qptm.loc[labels_qptm["type"] == t, "accession"],
                          labels_qptm.loc[labels_qptm["type"] == t, "position"]))
        d_sites = set(zip(labels_stratified.loc[labels_stratified["type"] == t, "accession"],
                          labels_stratified.loc[labels_stratified["type"] == t, "position"]))
        shared = q_sites & d_sites
        rows.append({
            "type": t, "n_qptm_sites": len(q_sites), "n_dbptm_sites": len(d_sites), "n_shared": len(shared),
            "overlap_fraction_of_qptm": (len(shared) / len(q_sites)) if q_sites else float("nan"),
            "overlap_fraction_of_dbptm": (len(shared) / len(d_sites)) if d_sites else float("nan"),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Orchestrator
# ---------------------------------------------------------------------------
def run_m3(raw_sources: Dict[str, pd.DataFrame], corpus: pd.DataFrame, cfg,
           glyde_30pct: Optional[pd.DataFrame] = None, qptm_raw: Optional[pd.DataFrame] = None) -> dict:
    """raw_sources: dict of label -> raw DataFrame, each with at least [accession, position,
    type] and optionally [evidence, lp, pmid]. This is the cascade itself (always recomputes) --
    see run_m3_cached below for the cache-aware entry point."""
    corpus_seq = dict(zip(corpus["accession"], corpus["sequence"]))
    step_log = {}
    masks = []

    canonicalized = {}
    for label, df in raw_sources.items():
        # [N1-CL#27] Isoform-suffix recovery runs BEFORE canonicalisation, so residue-aware type
        # resolution sees an accession that actually resolves against the corpus.
        stripped, log_iso = strip_isoform_suffix(df, corpus_seq)
        if log_iso["n_rows_recovered_by_stripping"]:
            step_log[f"step0_isoform_recovery::{label}"] = log_iso
        out0, log0 = canonicalize_types(stripped, cfg, corpus_seq=corpus_seq)
        step_log[f"step0_canonicalize::{label}"] = log0
        if log0["unmapped_types"]:
            raise ValueError(
                f"[{label}] {log0['n_unmapped_rows']} rows have PTM type spellings not in "
                f"PTM_TYPE_ALIASES or cfg.ptm_type_to_stratum: {log0['unmapped_types']}. "
                f"Add them to PTM_TYPE_ALIASES rather than letting them silently vanish.")
        canonicalized[label] = out0

    after_step1 = {}
    for label, df in canonicalized.items():
        out1, log1 = step1_evidence_filter(df)
        step_log[f"step1::{label}"] = log1
        after_step1[label] = out1

    after_step2 = {}
    for label, df in after_step1.items():
        out2, log2 = step2_validate_sites(df, corpus_seq)
        step_log[f"step2::{label}"] = log2
        after_step2[label] = out2

    # [N1-CL#37] `positive_source_labels` is tracked in lockstep with positives_for_dedup.
    # Previously build_site_agreement did zip(after_step2.keys(), filtered_positives) -- but
    # oglcnac_dataset_ii is skipped here (mask-only), so the two lists differ in length AND
    # order, silently misattributing every source after it. The length guard could not catch it:
    # zip truncates, so the totals always matched.
    positives_for_dedup, positive_source_labels = [], []
    for label, df in after_step2.items():
        if "oglcnac_dataset_ii" in label:
            masks.append(step3a_oglcnac_mask(df))
            continue  # Dataset-II contributes only to the mask, never to positives
        positives_for_dedup.append(df)  # no source has a usable per-site confidence field
        positive_source_labels.append(label)

    # [N1-CL#21] O-GlcNAc containment mask -- needs every source's positives in hand, so it runs
    # after the per-source loop rather than inside it.
    if positives_for_dedup:
        all_positives = pd.concat(positives_for_dedup, ignore_index=True, sort=False)
        oglcnac_confirmed = set(
            zip(all_positives.loc[all_positives["type"] == "O-GlcNAcylation", "accession"].astype(str),
                all_positives.loc[all_positives["type"] == "O-GlcNAcylation", "position"].astype(int)))
        oglyco_mask = step3c_oglycosylation_ambiguity_mask(all_positives, oglcnac_confirmed)
        if len(oglyco_mask):
            masks.append(oglyco_mask)
        step_log["step3c_oglcnac_containment"] = {
            "n_oglcnac_confirmed_sites": len(oglcnac_confirmed),
            "n_oglyco_sites_masked_for_oglcnac": len(oglyco_mask),
        }

    exclusion_mask = pd.concat(masks, ignore_index=True, sort=False) if masks else pd.DataFrame(
        columns=["accession", "position", "type", "mask_reason"])

    mask_keys = set(zip(exclusion_mask["accession"], exclusion_mask["position"], exclusion_mask["type"]))
    filtered_positives = []
    for df in positives_for_dedup:
        keep = ~df.apply(lambda r: (r["accession"], r["position"], r["type"]) in mask_keys, axis=1)
        filtered_positives.append(df[keep])

    # [N1-CL#62] Pass the per-source labels so `source_count` counts SOURCES, not rows. These are
    # the same labels build_site_agreement uses, so the two artefacts can no longer disagree.
    deduped = step4_dedup(filtered_positives, source_labels=positive_source_labels)
    step_log["step4::dedup"] = {"n_in": sum(len(d) for d in filtered_positives), "n_out": len(deduped)}

    labels_stratified, report_strat = step5_min_frequency(deduped, cfg, group_by_stratum=True,
                                                            type_to_stratum=cfg.ptm_type_to_stratum)
    labels_naive, report_naive = step5_min_frequency(deduped, cfg, group_by_stratum=False,
                                                       type_to_stratum=cfg.ptm_type_to_stratum)
    step_log["step5::stratified"] = report_strat
    step_log["step5::naive"] = report_naive

    labels_stratified = step6_inherit_split(labels_stratified, corpus)
    labels_naive = step6_inherit_split(labels_naive, corpus)
    step_log["step6::split_inherited"] = {"n_rows": len(labels_stratified)}

    # [N1-CL#31] Per-type survival waterfall. Frames captured at each stage above; T1's
    # per-source view cannot answer "how much of type X survived", which is the question that
    # actually determines whether a type clears the 200/1000 thresholds.
    # [N1-CL#44] Mask-only sources are excluded from the early stages. O-GlcNAcAtlas Dataset-II
    # rows are ambiguity-mask input and can NEVER become positives, but they were being counted
    # in raw_canonicalized/after_evidence/after_chemistry -- inflating O-GlcNAcylation's
    # denominator by 5,300 rows and understating its true survival (15.5% reported against a
    # denominator that included rows never eligible to survive).
    _pos_only = lambda d: {k: v for k, v in d.items() if "oglcnac_dataset_ii" not in k}
    _stage_frames = {
        "raw_canonicalized": pd.concat(list(_pos_only(canonicalized).values()), ignore_index=True, sort=False) if canonicalized else pd.DataFrame(),
        "after_evidence": pd.concat(list(_pos_only(after_step1).values()), ignore_index=True, sort=False) if after_step1 else pd.DataFrame(),
        "after_chemistry": pd.concat(list(_pos_only(after_step2).values()), ignore_index=True, sort=False) if after_step2 else pd.DataFrame(),
        "after_ambiguity_mask": pd.concat(filtered_positives, ignore_index=True, sort=False) if filtered_positives else pd.DataFrame(),
        "after_dedup": deduped,
    }
    T4_survival = build_T4_type_survival(_stage_frames, cfg, final_labels=labels_stratified)

    # [N1-CL#28/#52] Source attribution is built FIRST, because both the agreement table and the
    # crosstalk evidence classification need to know which source contributed each row -- that
    # information only exists pre-dedup.
    source_labels = []
    for label, df in zip(positive_source_labels, filtered_positives):
        source_labels.extend([label] * len(df))
    agreement_input = pd.concat(filtered_positives, ignore_index=True, sort=False) if filtered_positives else pd.DataFrame()

    ct = crosstalk_flags(deduped, source_frame=agreement_input,
                         source_labels=pd.Series(source_labels) if len(source_labels) == len(agreement_input) else None)
    labels_stratified = _enrich_label_schema(labels_stratified, corpus_seq, ct)
    labels_naive = _enrich_label_schema(labels_naive, corpus_seq, ct)
    assert len(positive_source_labels) == len(filtered_positives), \
        "source-label/frame misalignment -- see N1_CHANGELOG.md #37"
    if len(agreement_input) and len(source_labels) == len(agreement_input):
        site_agreement = build_site_agreement(agreement_input, pd.Series(source_labels))
    else:
        site_agreement = pd.DataFrame()
    step_log["step4::site_agreement"] = {"n_sites": len(site_agreement)}

    # [N1-CL#29] Gold negatives now contradiction-filtered against the real positive set.
    if glyde_30pct is not None:
        gold_negatives, gold_contradicted, gold_report = build_gold_negatives_nglyde(
            glyde_30pct, positive_sites=deduped)
        step_log["gold_negatives::contradiction_filter"] = gold_report
    else:
        gold_negatives = pd.DataFrame()
        gold_contradicted = pd.DataFrame()

    # qPTM: separate pipeline, never merged into labels_stratified/labels_naive above. Empty
    # DataFrame with the same schema if not supplied, so downstream code sees "no qPTM data" as
    # a structural fact (every type reports not_applicable) rather than crashing.
    if qptm_raw is not None and len(qptm_raw):
        labels_qptm, qptm_log = build_qptm_labels(qptm_raw, corpus, cfg, exclusion_mask=exclusion_mask)
        step_log["step_qptm::build"] = qptm_log
        qptm_dbptm_overlap = compute_qptm_dbptm_overlap(labels_qptm, labels_stratified)
    else:
        labels_qptm = labels_stratified.iloc[0:0].copy()
        qptm_dbptm_overlap = pd.DataFrame(columns=["type", "n_qptm_sites", "n_dbptm_sites", "n_shared",
                                                    "overlap_fraction_of_qptm", "overlap_fraction_of_dbptm"])

    return {
        "labels_stratified": labels_stratified, "labels_naive": labels_naive, "labels_qptm": labels_qptm,
        "exclusion_mask": exclusion_mask, "crosstalk": ct, "gold_negatives_n_glycosylation": gold_negatives,
        "gold_negatives_contradicted": gold_contradicted, "site_agreement": site_agreement,
        "annotation_discrepancies": build_annotation_discrepancies(
            exclusion_mask, gold_contradicted, site_agreement),
        "T4_survival": T4_survival,
        "qptm_dbptm_overlap": qptm_dbptm_overlap, "step_log": step_log,
    }


def _dataframe_fingerprint(df: Optional[pd.DataFrame]) -> str:
    """Content hash of a DataFrame's actual values (not just shape/columns), so the cache
    invalidates on real data changes -- falls back to a shape/columns fingerprint only if
    hash_pandas_object can't handle the dtypes present."""
    if df is None or len(df) == 0:
        return "empty"
    try:
        h = pd.util.hash_pandas_object(df, index=True).values
        import hashlib as _hashlib
        return _hashlib.sha256(h.tobytes()).hexdigest()[:16]
    except Exception:
        return fingerprint({"shape": list(df.shape), "columns": list(df.columns)})


def run_m3_cached(raw_sources: Dict[str, pd.DataFrame], corpus: pd.DataFrame, cfg, paths,
                   glyde_30pct: Optional[pd.DataFrame] = None, qptm_raw: Optional[pd.DataFrame] = None,
                   corpus_fingerprint: Optional[str] = None, force: bool = False) -> Tuple[dict, dict]:
    """Cache-aware entry point. Emits labels_stratified/labels_naive/labels_qptm/
    exclusion_mask/crosstalk (parquet, under paths.ROOT) + step_log.json + qptm_dbptm_overlap.csv,
    either by copying a valid cache hit or by running the cascade and writing fresh output."""
    sources_fp = {label: _dataframe_fingerprint(df) for label, df in sorted(raw_sources.items())}
    corpus_fp = corpus_fingerprint if corpus_fingerprint is not None else _dataframe_fingerprint(corpus)
    expected_fp = config_fingerprint({
        "corpus_fingerprint": corpus_fp,
        "raw_sources_fingerprint": sources_fp,
        "glyde_fingerprint": _dataframe_fingerprint(glyde_30pct),
        "qptm_fingerprint": _dataframe_fingerprint(qptm_raw),
    }, cfg=cfg)

    companion_files = {
        "labels_stratified": "labels_stratified.parquet",
        "labels_naive": "labels_naive.parquet",
        "labels_qptm": "labels_qptm.parquet",
        "exclusion_mask": "exclusion_mask.parquet",
        "crosstalk": "crosstalk.parquet",
        # [N1-CL#28/#29] new persisted outputs
        "site_agreement": "site_agreement.parquet",
        "gold_negatives_n_glycosylation": "gold_negatives_nglyco.parquet",
        "gold_negatives_contradicted": "gold_negatives_contradicted.parquet",
        "annotation_discrepancies": "annotation_discrepancies.parquet",  # [N1-CL#46]
    }

    if not force:
        hit = find_cached(companion_files["labels_stratified"])
        if hit is not None:
            fp_sidecar = hit.with_suffix(hit.suffix + ".fp")
            step_log_src = hit.parent / "step_log.json"
            if fp_sidecar.exists() and fp_sidecar.read_text().strip() == expected_fp and step_log_src.exists():
                src_dir = hit.parent
                result = {}
                for key, fname in companion_files.items():
                    src = src_dir / fname
                    dest = paths.ROOT / fname
                    if src != dest:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        if src.exists():
                            shutil.copy2(src, dest)
                    result[key] = pd.read_parquet(dest) if dest.exists() else pd.DataFrame()
                if hit != paths.ROOT / companion_files["labels_stratified"]:
                    shutil.copy2(fp_sidecar, (paths.ROOT / companion_files["labels_stratified"])
                                 .with_suffix(".parquet.fp"))
                dest_step_log = paths.ROOT / "step_log.json"
                if step_log_src != dest_step_log:
                    shutil.copy2(step_log_src, dest_step_log)
                result["step_log"] = json.loads(dest_step_log.read_text())
                overlap_src, overlap_dest = src_dir / "qptm_dbptm_overlap.csv", paths.ROOT / "qptm_dbptm_overlap.csv"
                if overlap_src.exists() and overlap_src != overlap_dest:
                    shutil.copy2(overlap_src, overlap_dest)
                result["qptm_dbptm_overlap"] = pd.read_csv(overlap_dest) if overlap_dest.exists() else pd.DataFrame()
                t4_src, t4_dest = src_dir / "tables" / "T4_type_survival.csv", paths.TABLES / "T4_type_survival.csv"
                if t4_src.exists() and t4_src != t4_dest:
                    t4_dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(t4_src, t4_dest)
                result["T4_survival"] = pd.read_csv(t4_dest) if t4_dest.exists() else pd.DataFrame()
                print(f"[M3] CACHE HIT ({'input_dataset' if paths.INPUT in hit.parents else 'working'}): "
                      f"reused labels_stratified/labels_naive/labels_qptm/exclusion_mask/crosstalk, "
                      f"skipped the six-step cascade on {sum(len(d) for d in raw_sources.values())} raw rows.")
                return result, {"cached": True, "fingerprint": expected_fp}
            print("[M3] found cached labels but fingerprint is stale (config/corpus/raw sources "
                  "changed) -- rebuilding the six-step cascade.")

    print("[M3] no valid cache -- running the six-step filtering cascade.")
    result = run_m3(raw_sources, corpus, cfg, glyde_30pct=glyde_30pct, qptm_raw=qptm_raw)
    for key, fname in companion_files.items():
        dest = paths.ROOT / fname
        dest.parent.mkdir(parents=True, exist_ok=True)
        result[key].to_parquet(dest, index=False)
    (paths.ROOT / companion_files["labels_stratified"]).with_suffix(".parquet.fp").write_text(expected_fp)
    (paths.ROOT / "step_log.json").write_text(json.dumps(result["step_log"], indent=2, default=str))
    paths.TABLES.mkdir(parents=True, exist_ok=True)
    result["T4_survival"].to_csv(paths.TABLES / "T4_type_survival.csv", index=False)
    result["qptm_dbptm_overlap"].to_csv(paths.ROOT / "qptm_dbptm_overlap.csv", index=False)
    return result, {"cached": False, "fingerprint": expected_fp}


# ---------------------------------------------------------------------------
# T1 / T2 / F2
# ---------------------------------------------------------------------------
def build_T1(step_log: dict) -> pd.DataFrame:
    rows = []
    for key, log in step_log.items():
        step, source = (key.split("::", 1) + [""])[:2]
        row = {"step": step, "source": source}
        row.update({k: v for k, v in log.items() if not isinstance(v, (list, dict))})
        rows.append(row)
    return pd.DataFrame(rows)


def build_T2(labels_stratified: pd.DataFrame, stratum_counts: dict, cfg) -> pd.DataFrame:
    # crosstalk_fraction computed once over the full table, not inside a per-type-filtered
    # loop (where a "duplicate" can never actually be found since the loop already filters it out).
    site_type_counts = labels_stratified.groupby(["accession", "position"])["type"].transform("nunique")
    is_crosstalk_site = site_type_counts > 1

    rows = []
    for (stratum, ptm_type), sub in labels_stratified.groupby(["stratum", "type"]):
        n_pos = len(sub)
        n_disc = (sub["split"] == "discovery").sum()
        n_hold = (sub["split"] == "holdout").sum()
        stratum_size = stratum_counts.get(stratum, None)
        crosstalk_frac = is_crosstalk_site.loc[sub.index].mean() if n_pos else 0.0
        rows.append({
            "type": ptm_type, "stratum": stratum,
            "positive_sites": n_pos, "sites_discovery": int(n_disc), "sites_holdout": int(n_hold),
            "stratum_size": stratum_size,
            "positive_rate": (n_pos / stratum_size) if stratum_size else None,
            "crosstalk_fraction": round(float(crosstalk_frac), 4),
            "meets_200_floor": n_pos >= cfg.min_sites_per_type,
            "meets_1000_headline": n_pos >= cfg.min_sites_headline,
            "v12b_eligible": (n_disc >= cfg.holdout_min_sites_per_type and n_hold >= cfg.holdout_min_sites_per_type),
        })
    return pd.DataFrame(rows).sort_values("positive_sites", ascending=False).reset_index(drop=True)


def build_F2(labels_stratified: pd.DataFrame, corpus: pd.DataFrame, step_log: dict, out_path: Path,
             stratum_counts: Optional[dict] = None, n_gold_clean: Optional[int] = None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(3, 2, figsize=(12, 13))
    axes[2, 1].axis("off")  # (a)-(e) is five panels on a 3x2 grid

    ax = axes[0, 0]  # (a) filtering waterfall (steps 1-2)
    steps = [k for k in step_log if k.startswith("step1::") or k.startswith("step2::")]
    labels_ = [s.split("::")[1] for s in steps]
    n_in = [step_log[s]["n_in"] for s in steps]
    n_out = [step_log[s]["n_out"] for s in steps]
    x = range(len(steps))
    ax.bar(x, n_in, alpha=0.4, label="in")
    ax.bar(x, n_out, alpha=0.8, label="out")
    ax.set_xticks(list(x)); ax.set_xticklabels(labels_, rotation=45, ha="right", fontsize=7)
    ax.set_title("(a) Filtering waterfall (steps 1-2)"); ax.legend(fontsize=7, loc="upper right")

    ax = axes[0, 1]  # (b) label frequency per type, log scale
    counts = labels_stratified.groupby("type").size().sort_values(ascending=False)
    ax.bar(range(len(counts)), counts.values)
    ax.set_yscale("log")
    ax.set_xticks(range(len(counts))); ax.set_xticklabels(counts.index, rotation=90, fontsize=6)
    ax.set_title("(b) Label frequency per type (log scale)")

    ax = axes[1, 0]  # (c) crosstalk fraction per type
    _site_type_counts_f2 = labels_stratified.groupby(["accession", "position"])["type"].transform("nunique")
    ct_frac = (_site_type_counts_f2 > 1).groupby(labels_stratified["type"]).mean()
    ax.bar(range(len(ct_frac)), ct_frac.values)
    ax.set_xticks(range(len(ct_frac))); ax.set_xticklabels(ct_frac.index, rotation=90, fontsize=6)
    ax.set_title("(c) Crosstalk fraction per type")

    # [N1-CL#48] Panel (d) -- three-tier negative counts for N-glycosylation. The plan specifies
    # F2 panels (a)-(e); (d) was never implemented. Counts are derived, not stored, exactly as the
    # downstream validation modules derive them: Gold from the cleaned N-GlyDE set, Hard = target
    # residues inside proteins that carry >=1 annotated site of this type (Corpus A), Background
    # = target residues everywhere else (Corpus B).
    ax = axes[1, 1]
    if len(labels_stratified) and stratum_counts is not None:
        _t = "N-glycosylation"
        _pos = labels_stratified[labels_stratified["type"] == _t]
        _carrier_acc = set(_pos["accession"])
        _carrier_seqs = corpus[corpus["accession"].isin(_carrier_acc)]["sequence"]
        _n_in_carriers = int(sum(sq.count("N") for sq in _carrier_seqs))
        _n_total = int(stratum_counts.get("N", 0))
        _tiers = {
            "Gold\n(N-GlyDE)": int(n_gold_clean or 0),
            "Hard\n(Corpus A)": max(0, _n_in_carriers - len(_pos)),
            "Background\n(Corpus B)": max(0, _n_total - _n_in_carriers),
        }
        ax.bar(range(len(_tiers)), list(_tiers.values()))
        ax.set_yscale("log")
        ax.set_xticks(range(len(_tiers))); ax.set_xticklabels(list(_tiers), fontsize=7)
        ax.set_title(f"(d) Three-tier negatives, {_t} (log)")
        for _i, _v in enumerate(_tiers.values()):
            ax.text(_i, _v, f"{_v:,}", ha="center", va="bottom", fontsize=6)

    ax = axes[2, 0]  # (e) protein-length distribution, positive vs negative, top type
    if len(labels_stratified):
        top_type = labels_stratified["type"].value_counts().idxmax()
        pos_acc = set(labels_stratified.loc[labels_stratified["type"] == top_type, "accession"])
        pos_len = corpus[corpus["accession"].isin(pos_acc)]["length"]
        neg_len = corpus[~corpus["accession"].isin(pos_acc)]["length"]
        ax.hist(neg_len, bins=30, alpha=0.5, label="no annotated site", density=True)
        ax.hist(pos_len, bins=30, alpha=0.5, label=f"has {top_type} site", density=True)
        ax.set_title(f"(e) Protein-length distribution ({top_type})"); ax.legend(fontsize=7, loc="upper right")

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return out_path


In [10]:
# ---------------------------------------------------------------------------
# M3
# ---------------------------------------------------------------------------
cplm_raw = parse_cplm_zip(m1_raw_paths["cplm_human"])
cplm_raw["evidence"] = "MS/MS"  # CPLM's compact schema has no per-site evidence field -> treat as verified

raw_sources = {"cplm_human": cplm_raw[["accession", "position", "type", "evidence"]]}

if MOCK_MODE:
    # Synthetic N-glycosylation + qPTM sources for a real, if tiny, cascade run without network.
    import random as _random
    _rng = _random.Random(7)
    _rows = []
    _accs = corpus["accession"].tolist()
    for i in range(300):
        acc = _rng.choice(_accs)
        seq = corpus.loc[corpus["accession"] == acc, "sequence"].iloc[0]
        n_positions = [j + 1 for j, ch in enumerate(seq) if ch == "N"]
        if not n_positions:
            continue
        pos = _rng.choice(n_positions) if _rng.random() > 0.1 else _rng.randint(1, len(seq))
        _rows.append({"accession": acc, "position": pos, "type": "N-linked Glycosylation",
                      "evidence": "MS/MS", "pmid": str(40000000 + i)})
    raw_sources["dbptm_nglyco_MOCK"] = pd.DataFrame(_rows)
    # Mock qPTM deliberately overlaps ~50% of the mock dbPTM rows (not all, not none), so
    # compute_qptm_dbptm_overlap exercises a genuine partial-overlap fraction.
    _qptm_rows = []
    for i, r in enumerate(_rows):
        if _rng.random() < 0.5:
            _qptm_rows.append({"accession": r["accession"], "position": r["position"],
                              "type": "N-linked Glycosylation", "species": "Homo sapiens",
                              "pmid": str(42000000 + i)})
    for i in range(80):  # plus qPTM-only sites, independent of the dbPTM subset
        acc = _rng.choice(_accs)
        seq = corpus.loc[corpus["accession"] == acc, "sequence"].iloc[0]
        n_positions = [j + 1 for j, ch in enumerate(seq) if ch == "N"]
        if not n_positions:
            continue
        _qptm_rows.append({"accession": acc, "position": _rng.choice(n_positions),
                          "type": "N-linked Glycosylation", "species": "Homo sapiens",
                          "pmid": str(43000000 + i)})
    qptm_raw = pd.DataFrame(_qptm_rows)
    glyde_30pct = None  # Gold-tier negatives not exercised in MOCK_MODE's synthetic fixture
else:
    for _dbptm_type, _dbptm_path in resolved.get("dbptm", {}).items():
        _key = f"dbptm_{_dbptm_type.replace(' ', '_').replace('/', '_')}"
        raw_sources[_key] = parse_dbptm_flatfile(_dbptm_path, _dbptm_type)

    # O-GlcNAcAtlas: Dataset-I (unambiguous) feeds positives; Dataset-II (ambiguous) MUST be
    # keyed with "oglcnac_dataset_ii" -- run_m3's step-3 dispatch matches on that exact
    # substring to route it to the ambiguity mask, never by anything in the parsed data itself.
    _oglcnac = resolved.get("oglcnac_atlas", {})
    if "Dataset-I" in _oglcnac:
        raw_sources["oglcnac_dataset_i"] = parse_oglcnac_atlas_flatfile(_oglcnac["Dataset-I"])
    if "Dataset-II" in _oglcnac:
        raw_sources["oglcnac_dataset_ii"] = parse_oglcnac_atlas_flatfile(_oglcnac["Dataset-II"])

    if "glycosite_atlas" in resolved:
        raw_sources["glycosite_atlas"] = parse_glycosite_atlas_flatfile(resolved["glycosite_atlas"])

    # CF-26: qPTM, parsed via M1's parse_qptm_flatfile -- required before M6's streaming pass.
    qptm_raw = (parse_qptm_flatfile(m1_raw_paths["qptm"]) if "qptm" in m1_raw_paths
               else pd.DataFrame(columns=["accession", "position", "type", "species", "pmid"]))

    # N-GlyDE (via StackGlyEmbed, train+test concatenated): the only Gold-tier negative source.
    if "glyde" in resolved:
        _glyde_parts = [parse_glyde_flatfile(p) for p in resolved["glyde"].values()]
        glyde_30pct = pd.concat(_glyde_parts, ignore_index=True) if _glyde_parts else None
    else:
        glyde_30pct = None

result, m3_cache_info = run_m3_cached(
    raw_sources, corpus, CFG, Paths, glyde_30pct=glyde_30pct, qptm_raw=qptm_raw,
    corpus_fingerprint=m2_cache_info["fingerprint"], force=FORCE_REBUILD,
)

T1 = build_T1(result["step_log"])
T2 = build_T2(result["labels_stratified"], stratum_counts, CFG)
Paths.TABLES.mkdir(parents=True, exist_ok=True)
T1.to_csv(Paths.TABLES / "T1_dataset_provenance.csv", index=False)
T2.to_csv(Paths.TABLES / "T2_label_composition.csv", index=False)
f2_path = build_F2(result["labels_stratified"], corpus, result["step_log"],
                   Paths.FIGURES / "F2_dataset_composition.png",
                   stratum_counts=stratum_counts,
                   n_gold_clean=len(result.get("gold_negatives_n_glycosylation", [])))

# [N1-CL#35/#36] OOD label sets for all four species. Source coverage per species is MEASURED,
# not assumed -- a None label in CFG.ood_species means that source does not cover that species
# (qPTM has no E. coli; O-GlcNAcAtlas has no E. coli). N-GlycositeAtlas is human-only and is
# deliberately not an OOD source.
ood_labels = {}
if not MOCK_MODE:
    for _sp, _meta in CFG.ood_species.items():
        if _sp not in ood_corpora:
            print(f"    OOD [{_sp}]: no corpus (proteome download missing) -- skipped")
            continue
        _ood_sources, _ood_mask_parts = {}, []

        # dbPTM -- species via entry_name suffix
        for _t, _p in resolved.get("dbptm", {}).items():
            _d = parse_dbptm_flatfile(_p, _t)
            _d = _d[_d["species_suffix"] == _meta["uniprot_suffix"]]
            if len(_d):
                _ood_sources[f"dbptm_{_t}"] = _d

        # CPLM -- verified strain-qualified per-species file
        if _sp in resolved.get("ood_cplm", {}):
            _c = parse_cplm_zip(resolved["ood_cplm"][_sp])
            _c["evidence"] = "MS/MS"
            if len(_c):
                _ood_sources[f"cplm_{_sp}"] = _c

        # qPTM -- species column; same reliability floor as the human replication set
        if _meta.get("qptm_label") and len(qptm_raw):
            _q = qptm_raw[qptm_raw["species"].astype(str).str.strip().str.lower() == _meta["qptm_label"]]
            if "reliability" in _q.columns:
                _q = _q[pd.to_numeric(_q["reliability"], errors="coerce") >= CFG.qptm_reliability_floor]
            if len(_q):
                _ood_sources["qptm"] = _q

        # O-GlcNAcAtlas -- Dataset-I positives, Dataset-II into this species' ambiguity mask
        _og = resolved.get("oglcnac_atlas", {})
        if _meta.get("oglcnac_label"):
            if "Dataset-I" in _og:
                _o1 = parse_oglcnac_atlas_flatfile(_og["Dataset-I"], species=_meta["oglcnac_label"])
                if len(_o1):
                    _ood_sources["oglcnac_dataset_i"] = _o1
            if "Dataset-II" in _og:
                _o2 = parse_oglcnac_atlas_flatfile(_og["Dataset-II"], species=_meta["oglcnac_label"])
                if len(_o2):
                    _ood_mask_parts.append(step3a_oglcnac_mask(_o2))

        _ood_mask = pd.concat(_ood_mask_parts, ignore_index=True) if _ood_mask_parts else None
        _lab, _log = build_ood_labels(_ood_sources, ood_corpora[_sp]["corpus"], _sp, CFG,
                                      exclusion_mask=_ood_mask)
        # [N1-CL#51] Carry homologous_to_human onto the labels, so a strict-OOD analysis is a
        # one-column filter downstream rather than a re-join against the corpus.
        if len(_lab) and "homologous_to_human" in ood_corpora[_sp]["corpus"].columns:
            _hom = ood_corpora[_sp]["corpus"].set_index("accession")["homologous_to_human"]
            _lab["homologous_to_human"] = _lab["accession"].map(_hom)
        ood_labels[_sp] = _lab
        if len(_lab):
            _lab.to_parquet(Paths.ROOT / f"labels_ood_{_sp}.parquet", index=False)
        print(f"    OOD [{_sp}]: {len(_lab)} sites from {len(_ood_sources)} source(s) "
              f"({', '.join(sorted({k.split('_')[0] for k in _ood_sources}))})")
        show_json(_log, f"    OOD [{_sp}] cascade log:")
        if len(_lab):
            show_table(_lab.groupby(["stratum", "type"]).size().reset_index(name="n_sites")
                       .sort_values("n_sites", ascending=False),
                       f"    OOD [{_sp}] -- sites per type:")

    # CF-22 applicability matrix -- which (species, type) pairs are actually testable, in both
    # the full view and the [N1-CL#53] homology-corrected (strict) view.
    if ood_labels:
        TA = build_species_type_applicability(ood_labels, result["labels_stratified"], CFG)
        TA.to_csv(Paths.TABLES / "T5_ood_applicability.csv", index=False)
        _strict = pd.DataFrame([
            dict(species=_sp,
                 n_sites_full=int(len(_l)),
                 n_sites_strict=int((~_l["homologous_to_human"].fillna(False)).sum())
                                 if "homologous_to_human" in _l.columns else None)
            for _sp, _l in ood_labels.items() if len(_l)])
        if len(_strict):
            _strict["pct_retained_strict"] = (100 * _strict["n_sites_strict"]
                                              / _strict["n_sites_full"]).round(1)
            show_table(_strict, "OOD sites: full vs homology-corrected (strict) -- [N1-CL#51]")
        _testable = TA[TA["testable"]]
        _testable_strict = TA[TA["testable_strict"] == True]  # noqa: E712 (None must not count)
        print(f"\nT5 -- OOD applicability (CF-22): {len(_testable)} of {len(TA)} (species, type) "
              f"pairs have >={CFG.ood_min_sites_per_type} sites and are testable (full view).")
        print(f"T5 -- STRICT (no human homolog): {len(_testable_strict)} of {len(TA)} pairs "
              f"still clear the floor once restricted to non-homologous proteins [N1-CL#53] -- "
              f"this is the number to use for a genuine species-novelty claim, not the full-view "
              f"count above.")
        show_table(TA.sort_values(["species", "n_sites_species"], ascending=[True, False]),
                   "T5 -- FULL + STRICT OOD applicability matrix (all pairs, testable and not):")

print(f"M3: labels_stratified={len(result['labels_stratified'])} rows "
      f"({'CACHED' if m3_cache_info['cached'] else 'built fresh'}), "
      f"exclusion_mask={len(result['exclusion_mask'])} rows, crosstalk={len(result['crosstalk'])} rows")
print(f"    CF-26: labels_qptm={len(result['labels_qptm'])} rows (independent-curation replication set)")
if len(result["qptm_dbptm_overlap"]):
    print("\nqPTM x dbPTM site-set overlap (CF-26 -- required alongside the label set itself, since "
         "the two share primary literature and the replication is only partially independent):")
    show_table(result["qptm_dbptm_overlap"])
if len(result.get("gold_negatives_contradicted", [])):
    _g = result["step_log"].get("gold_negatives::contradiction_filter", {})
    print(f"\nGold-tier negatives: {_g.get('n_gold_clean')} clean, "
          f"{_g.get('n_contradicted')} removed ({_g.get('pct_contradicted')}% contradicted by an "
          f"annotated positive). Removed set saved to gold_negatives_contradicted.parquet.")
# [N1-CL#46] Annotation-discrepancy candidates -- masked/contradicted sites are excluded from
# the STATISTICS but retained here as the study set for latent-based adjudication in N4.
if len(result.get("annotation_discrepancies", [])):
    _ad = result["annotation_discrepancies"]
    show_table(_ad.groupby("discrepancy_class").size().reset_index(name="n_sites"),
               "Annotation-discrepancy candidates (saved to annotation_discrepancies.parquet -- "
               "these are NOT discarded, they are the N4 adjudication set):")
    show_table(_ad[_ad["discrepancy_class"] == "containment_uncatalogued_oglcnac"]
               .head(10)[["accession", "position", "type", "evidence_for", "evidence_against"]],
               "  sample -- O-linked sites that may be uncatalogued O-GlcNAc:")

if len(result.get("site_agreement", [])):
    _sa = result["site_agreement"]
    _disagree = _sa[_sa["n_sources_covering_but_not_reporting"] > 0]
    print(f"\nSite agreement: {len(_sa)} sites; {len(_disagree)} have >=1 source that covered the "
          f"protein but did NOT report the site (genuine disagreement, not a coverage gap). "
          f"Saved to site_agreement.parquet for the N4 latent-adjudication analysis.")

# [N1-CL#52] Crosstalk composition -- WHICH type combinations co-occur, and whether a single
# source asserted the co-modification or it was assembled from different sources.
_ct = result["crosstalk"]
if len(_ct):
    show_table(_ct["type_combination"].value_counts().head(25).reset_index()
               .rename(columns={"index": "type_combination", "type_combination": "n_sites",
                                "count": "n_sites"}),
               "Crosstalk -- 25 most common type combinations:")
    if "evidence_class" in _ct.columns:
        show_table(_ct["evidence_class"].value_counts().reset_index()
                   .rename(columns={"index": "evidence_class", "evidence_class": "n_sites",
                                    "count": "n_sites"}),
                   "Crosstalk evidence class (same_source_multi_type = one source asserted both; "
                   "cross_source_only = assembled from different sources, so co-modification and "
                   "inter-source disagreement are not separable):")
        show_table(_ct.groupby("n_types").size().reset_index(name="n_sites"),
                   "Crosstalk -- number of distinct PTM types per site:")

# [N1-CL#31] Per-type survival -- the direct answer to "how many PTM types actually survive".
T4 = result.get("T4_survival")
if T4 is not None and len(T4):
    _n_entered = len(T4)
    _n_floor = int(T4["meets_200_floor"].sum())
    _n_headline = int(T4["meets_1000_headline"].sum())
    print(f"\nT4 -- per-type survival through the cascade: {_n_entered} types entered, "
          f"{_n_floor} clear the >={CFG.min_sites_per_type}-site floor, "
          f"{_n_headline} clear the >={CFG.min_sites_headline}-site headline bar "
          f"({_n_entered - _n_floor} pooled into <stratum>::other_PTM).")
    show_table(T4)
    _heavy_loss = T4[(T4["pct_surviving"].notna()) & (T4["pct_surviving"] < 70)]
    if len(_heavy_loss):
        print(f"NOTE: {len(_heavy_loss)} type(s) lost >30% of sites in the cascade -- check "
              f"which stage in the table above (chemistry validation is the usual culprit).")

# [N1-CL#42] step_log holds the per-source, per-step counts (including the new
# residue-unresolved drops, the LP/O-GlcNAc mask counts and the whole qPTM sub-cascade). It was
# written to disk only -- printed here so it survives in the saved notebook.
show_json(result["step_log"], "STEP LOG -- per-source, per-step counts for the whole cascade:")

if len(result.get("site_agreement", [])):
    _sa = result["site_agreement"]
    show_table(_sa["n_sources_reporting"].value_counts().sort_index().reset_index()
               .rename(columns={"index": "n_sources_reporting", 0: "n_sites"}),
               "Site agreement -- how many sources report each site:")
    show_table(_sa.groupby("type")["n_sources_covering_but_not_reporting"]
               .agg(["size", "mean", "max"]).reset_index()
               .rename(columns={"size": "n_sites", "mean": "mean_disagreeing_sources",
                                "max": "max_disagreeing_sources"}),
               "Site agreement -- genuine disagreement per type (source covered the protein but did not report the site):")

print("\nT1 -- dataset provenance:")
show_table(T1)
print("\nT2 -- label composition:")
show_table(T2)
print(f"\nF2 saved to {f2_path}")

# M2's discovery-partition gate (each primary type must retain >=1,000 sites) uses T2's real,
# M3-filtered per-partition counts, not M2's own pre-M3 raw estimate.
discovery_headline_ok = {
    t: bool((T2.loc[T2["type"] == t, "sites_discovery"] >= CFG.min_sites_headline).any())
    for t in CFG.primary_types if t in T2["type"].values
}
print(f"\nGate -- primary types clearing the >=1000-site headline bar IN DISCOVERY: "
     f"{discovery_headline_ok}")
if not all(discovery_headline_ok.values()) and not MOCK_MODE:
    print("ACTION REQUIRED: at least one primary type falls below 1,000 discovery-partition sites. "
         "Reduce CFG.holdout_fraction to 0.15 and re-run M2+M3 before proceeding to N2.")
if MOCK_MODE:
    print("(MOCK_MODE: gate is expected to fail on this tiny synthetic corpus -- this checks "
          "the reporting logic, not real statistical power. Run on Kaggle for a real gate check.)")


/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)
/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


[M3] no valid cache -- running the six-step filtering cascade.


[M3] Gold-tier negatives: removed 222/1310 (16.95%) contradicted by an annotated N-glycosylation positive. 1088 clean gold negatives remain.


/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


    OOD [mouse]: 191577 sites from 31 source(s) (cplm, dbptm, oglcnac, qptm)

    OOD [mouse] cascade log:
{
  "species": "mouse",
  "n_raw_rows": 3080596,
  "sources": [
    "cplm_mouse",
    "dbptm_ADP-ribosylation",
    "dbptm_Acetylation",
    "dbptm_Butyrylation",
    "dbptm_C-linked Glycosylation",
    "dbptm_Carboxylation",
    "dbptm_Citrullination",
    "dbptm_Crotonylation",
    "dbptm_Deamidation",
    "dbptm_Gamma-carboxyglutamic acid",
    "dbptm_Glutarylation",
    "dbptm_Glutathionylation",
    "dbptm_Hydroxylation",
    "dbptm_Malonylation",
    "dbptm_Methylation",
    "dbptm_N-linked Glycosylation",
    "dbptm_Nitration",
    "dbptm_O-linked Glycosylation",
    "dbptm_Phosphorylation",
    "dbptm_Propionylation",
    "dbptm_S-nitrosylation",
    "dbptm_S-palmitoylation",
    "dbptm_Serotonylation",
    "dbptm_Succinylation",
    "dbptm_Sulfation",
    "dbptm_Sulfhydration",
    "dbptm_Sulfoxidation",
    "dbptm_Sumoylation",
    "dbptm_Ubiquitination",
    "oglcnac_da

/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


    OOD [rat]: 59356 sites from 23 source(s) (cplm, dbptm, oglcnac, qptm)

    OOD [rat] cascade log:
{
  "species": "rat",
  "n_raw_rows": 328548,
  "sources": [
    "cplm_rat",
    "dbptm_ADP-ribosylation",
    "dbptm_Acetylation",
    "dbptm_Biotinylation",
    "dbptm_C-linked Glycosylation",
    "dbptm_Deamidation",
    "dbptm_Glutathionylation",
    "dbptm_Hydroxylation",
    "dbptm_Lipoylation",
    "dbptm_Methylation",
    "dbptm_N-linked Glycosylation",
    "dbptm_Nitration",
    "dbptm_O-linked Glycosylation",
    "dbptm_Phosphorylation",
    "dbptm_S-nitrosylation",
    "dbptm_S-palmitoylation",
    "dbptm_Succinylation",
    "dbptm_Sulfation",
    "dbptm_Sulfoxidation",
    "dbptm_Sumoylation",
    "dbptm_Ubiquitination",
    "oglcnac_dataset_i",
    "qptm"
  ],
  "step0_isoform_recovery": {
    "n_rows": 328548,
    "n_isoform_rows": 1094,
    "n_rows_recovered_by_stripping": 605
  },
  "step0_canonicalize": {
    "n_rows": 328548,
    "n_unmapped_rows": 0,
    "n_residue_u

/tmp/ipykernel_251512/471168607.py:451: DtypeWarning: Columns (11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(io.StringIO(_decode_bytes_robust(data)), **kwargs)


    OOD [yeast]: 89583 sites from 24 source(s) (cplm, dbptm, oglcnac, qptm)

    OOD [yeast] cascade log:
{
  "species": "yeast",
  "n_raw_rows": 2366689,
  "sources": [
    "cplm_yeast",
    "dbptm_Acetylation",
    "dbptm_Biotinylation",
    "dbptm_Butyrylation",
    "dbptm_Crotonylation",
    "dbptm_Formylation",
    "dbptm_Glutarylation",
    "dbptm_Glutathionylation",
    "dbptm_Hydroxylation",
    "dbptm_Lipoylation",
    "dbptm_Malonylation",
    "dbptm_Methylation",
    "dbptm_N-linked Glycosylation",
    "dbptm_O-linked Glycosylation",
    "dbptm_Phosphorylation",
    "dbptm_Propionylation",
    "dbptm_S-nitrosylation",
    "dbptm_S-palmitoylation",
    "dbptm_Succinylation",
    "dbptm_Sulfoxidation",
    "dbptm_Sumoylation",
    "dbptm_Ubiquitination",
    "oglcnac_dataset_i",
    "qptm"
  ],
  "step0_isoform_recovery": {
    "n_rows": 2366689,
    "n_isoform_rows": 8,
    "n_rows_recovered_by_stripping": 8
  },
  "step0_canonicalize": {
    "n_rows": 2366689,
    "n_unmappe

    OOD [ecoli]: 17197 sites from 17 source(s) (cplm, dbptm)

    OOD [ecoli] cascade log:
{
  "species": "ecoli",
  "n_raw_rows": 31421,
  "sources": [
    "cplm_ecoli",
    "dbptm_ADP-ribosylation",
    "dbptm_Acetylation",
    "dbptm_Biotinylation",
    "dbptm_Carboxylation",
    "dbptm_Formylation",
    "dbptm_Glutathionylation",
    "dbptm_Hydroxylation",
    "dbptm_Lipoylation",
    "dbptm_Malonylation",
    "dbptm_Methylation",
    "dbptm_Phosphorylation",
    "dbptm_S-nitrosylation",
    "dbptm_Succinylation",
    "dbptm_Sulfation",
    "dbptm_Sulfhydration",
    "dbptm_Sulfoxidation"
  ],
  "step0_isoform_recovery": {
    "n_rows": 31421,
    "n_isoform_rows": 0,
    "n_rows_recovered_by_stripping": 0
  },
  "step0_canonicalize": {
    "n_rows": 31421,
    "n_unmapped_rows": 0,
    "n_residue_unresolved_dropped": 0,
    "unmapped_types": [],
    "residue_aware": true
  },
  "step0_unmapped_dropped": 0,
  "step2_validate": {
    "n_in": 31421,
    "n_dropped": 1114,
    "n_out"

## Self-check

In [11]:
assert corpus["cluster_id"].notna().all()
assert set(corpus["split"].unique()) <= {"discovery", "holdout"}
per_cluster_splits = corpus.groupby("cluster_id")["split"].nunique()
assert (per_cluster_splits == 1).all(), "a CD-HIT cluster was split across discovery/holdout -- leakage bug"
assert result["labels_stratified"]["split"].isin(["discovery", "holdout"]).all()
assert not result["labels_stratified"]["type"].isna().any(), "unmapped PTM type slipped through canonicalisation"

# [N1-CL#48] Stronger structural checks -- each of these corresponds to a bug actually found
# during development, so they are regression guards, not decoration.
assert set(result["labels_naive"]["stratum"].unique()) == {"ALL"}, \
    "labels_naive must use the ALL placeholder stratum, else it duplicates labels_stratified (CL#45)"
assert set(result["labels_stratified"]["stratum"].unique()) <= set(CFG.stratum_residues), \
    "a stratum appeared that is not in cfg.stratum_residues"
_mask_keys = set(zip(result["exclusion_mask"]["accession"].astype(str),
                     result["exclusion_mask"]["position"].astype(int),
                     result["exclusion_mask"]["type"])) if len(result["exclusion_mask"]) else set()
_pos_keys = set(zip(result["labels_stratified"]["accession"].astype(str),
                    result["labels_stratified"]["position"].astype(int),
                    result["labels_stratified"]["type"]))
assert not (_mask_keys & _pos_keys), "a masked site survived as a positive -- step 3 leak"
if len(result.get("gold_negatives_n_glycosylation", [])):
    _g = set(zip(result["gold_negatives_n_glycosylation"]["accession"].astype(str),
                 result["gold_negatives_n_glycosylation"]["position"].astype(int)))
    _n = set(zip(result["labels_stratified"].loc[result["labels_stratified"]["type"] == "N-glycosylation", "accession"].astype(str),
                 result["labels_stratified"].loc[result["labels_stratified"]["type"] == "N-glycosylation", "position"].astype(int)))
    assert not (_g & _n), "a gold-tier negative is also an annotated positive -- contradiction filter failed (CL#29)"
for _sp, _lab in (ood_labels or {}).items():
    if len(_lab):
        assert set(_lab["split"].unique()) == {"ood"}, f"OOD labels for {_sp} carry a non-ood split"
print("N1 structural self-check: OK")
print(f"Cache summary this run -- M2: {'HIT' if m2_cache_info['cached'] else 'built'}, "
      f"M3: {'HIT' if m3_cache_info['cached'] else 'built'}")

# ===========================================================================
# [N1-CL#42] CONSOLIDATED RUN SUMMARY -- one block containing every number needed to review this
# run from the saved notebook alone, without the output dataset.
# ===========================================================================
print("\n" + "=" * 78 + "\nN1 RUN SUMMARY\n" + "=" * 78)

_summary = {
    "corpus": {
        "n_proteins": int(len(corpus)),
        "n_clusters": int(corpus["cluster_id"].nunique()),
        "split_counts": {k: int(v) for k, v in corpus["split"].value_counts().items()},
        "achieved_holdout_fraction": round(float((corpus["split"] == "holdout").mean()), 4),
        "gate_15k_20k_proteins": bool(gate_report["gate_protein_count_15k_20k"]),
        "cdhit_identity": CFG.cdhit_identity,
    },
    "labels": {
        "labels_stratified_rows": int(len(result["labels_stratified"])),
        "labels_naive_rows": int(len(result["labels_naive"])),
        "labels_qptm_rows": int(len(result["labels_qptm"])),
        "exclusion_mask_rows": int(len(result["exclusion_mask"])),
        "crosstalk_sites": int(len(result["crosstalk"])),
        "site_agreement_sites": int(len(result.get("site_agreement", []))),
    },
    "types": {
        "n_types_stratified_on": gate_report.get("n_types_stratified_on"),
        "n_types_entered_cascade": int(len(T4)) if T4 is not None and len(T4) else 0,
        "n_types_meeting_200_floor": int(T4["meets_200_floor"].sum()) if T4 is not None and len(T4) else 0,
        "n_types_meeting_1000_headline": int(T4["meets_1000_headline"].sum()) if T4 is not None and len(T4) else 0,
    },
    "primary_type_gate": discovery_headline_ok,
    "gold_negatives": result["step_log"].get("gold_negatives::contradiction_filter", {}),
    "qptm": {
        "reliability_floor": CFG.qptm_reliability_floor,
        "step_log": result["step_log"].get("step_qptm::build", {}),
    },
    "ood": {
        _sp: {
            "n_proteins": int(len(_d["corpus"])),
            "n_labelled_sites": int(len(ood_labels.get(_sp, []))),
            # [N1-CL#53] Strict = no human homolog (CL#51). None if cd-hit-2d didn't run/failed.
            "n_labelled_sites_strict": (
                int((~ood_labels[_sp]["homologous_to_human"].fillna(False)).sum())
                if len(ood_labels.get(_sp, [])) and "homologous_to_human" in ood_labels[_sp].columns
                else None
            ),
        }
        for _sp, _d in (ood_corpora or {}).items()
    },
    "cache": {"M2": "HIT" if m2_cache_info["cached"] else "built",
              "M3": "HIT" if m3_cache_info["cached"] else "built"},
}
show_json(_summary)
print("\nAll diagnostic tables above are printed in FULL (show_table uses to_string), so this "
      "notebook alone is sufficient to review the run -- the output dataset is not needed.")


N1 structural self-check: OK
Cache summary this run -- M2: built, M3: built

N1 RUN SUMMARY
{
  "corpus": {
    "n_proteins": 18128,
    "n_clusters": 11909,
    "split_counts": {
      "discovery": 14384,
      "holdout": 3744
    },
    "achieved_holdout_fraction": 0.2065,
    "gate_15k_20k_proteins": true,
    "cdhit_identity": 0.4
  },
  "labels": {
    "labels_stratified_rows": 498619,
    "labels_naive_rows": 498619,
    "labels_qptm_rows": 131700,
    "exclusion_mask_rows": 6382,
    "crosstalk_sites": 49495,
    "site_agreement_sites": 498619
  },
  "types": {
    "n_types_stratified_on": 44,
    "n_types_entered_cascade": 47,
    "n_types_meeting_200_floor": 21,
    "n_types_meeting_1000_headline": 20
  },
  "primary_type_gate": {
    "N-glycosylation": true,
    "O-GlcNAcylation": true,
    "succinylation": true
  },
  "gold_negatives": {
    "n_gold_before": 1310,
    "n_contradicted": 222,
    "n_gold_clean": 1088,
    "pct_contradicted": 16.95,
    "contradiction_check_run

## Output contract / packaging

In [12]:
# ===========================================================================
# N1 OUTPUT CONTRACT -- everything downstream notebooks may assume exists, with no other
# guarantee about N1's internals. N2 (M4-M6) needs corpus.parquet + labels_*.parquet only; N3
# needs the full set below (N1's dataset is attached to N3 separately -- see N1_CHANGELOG.md).
# After a successful run: save a notebook version, then Kaggle "New Dataset" -> "Notebook
# Output" (preserves the tables/exploration folder structure this checklist depends on).
# ===========================================================================
save_config()

REQUIRED_FILES = [
    "manifest.json",                        # M1 provenance
    "corpus.parquet", "corpus.parquet.fp",  # M2
    "stratum_counts.json", "split_manifest.json",
    "labels_stratified.parquet", "labels_naive.parquet", "labels_qptm.parquet",  # M3
    "exclusion_mask.parquet", "crosstalk.parquet", "step_log.json", "qptm_dbptm_overlap.csv",
    "site_agreement.parquet", "gold_negatives_nglyco.parquet",  # N1-CL#28/#29
    "gold_negatives_contradicted.parquet",  # [N1-CL#49] written all along, was simply unlisted
    "annotation_discrepancies.parquet",  # N1-CL#46
    "tables/T1_dataset_provenance.csv", "tables/T2_label_composition.csv",
    "tables/T3_split_composition.csv", "tables/T4_type_survival.csv",  # N1-CL#26/#31
    "tables/T5_ood_applicability.csv",  # N1-CL#36 (CF-22)
    "tables/TE_dataset_overview.csv",       # M_E
    "figures/F2_dataset_composition.png",
    "config.json",
]

print(f"{'file':<55}{'present':<10}{'size':>12}")
missing_required = []
for rel in REQUIRED_FILES:
    f = Paths.ROOT / rel
    present = f.exists()
    size = f.stat().st_size if present else 0
    if not present:
        missing_required.append(rel)
    print(f"{rel:<55}{str(present):<10}{size:>12,}")

# [N1-CL#25] OOD artefacts -- informational, one set per configured species.
for _sp in CFG.ood_species:
    for _rel in [f"corpus_ood_{_sp}.parquet", f"stratum_counts_ood_{_sp}.json",
                 f"labels_ood_{_sp}.parquet", f"homology_report_ood_{_sp}.json"]:
        _f = Paths.ROOT / _rel
        print(f"{_rel:<55}{str(_f.exists()):<10}{(_f.stat().st_size if _f.exists() else 0):>12,}")

n_exploration_files = len(list(Paths.EXPLORATION.glob("*.json"))) if Paths.EXPLORATION.exists() else 0
print(f"{'exploration/*.json (M_E per-source profiles)':<55}{str(n_exploration_files > 0):<10}{n_exploration_files:>12} files")

# [N1-CL#18] Informational, not blocking -- these are for your own review, not a downstream
# notebook dependency, so a missing one (e.g. cross-dataset overlap with <2 comparable sets)
# doesn't fail the packaging check the way a missing REQUIRED_FILES entry does.
for rel in ["tables/TE_type_coverage.csv", "tables/TE_dbptm_species_audit.csv",
            "tables/TE_cross_dataset_overlap.csv", "tables/TE_chemistry_failures.csv",
            "tables/TE_isoform_audit.csv", "tables/TE_gold_negative_contradictions.csv",
            "tables/TE_qptm_reliability_sensitivity.csv"]:
    f = Paths.ROOT / rel
    print(f"{rel:<55}{str(f.exists()):<10}{(f.stat().st_size if f.exists() else 0):>12,}")

# [N1-CL#12] Real recursive total of Paths.ROOT, not just the REQUIRED_FILES list above -- this
# is the number that matters, since it's exactly what "New Dataset" -> "Notebook Output" uploads.
output_dir_bytes = path_size_bytes(Paths.ROOT)
print(f"\nOutput dataset size (Paths.ROOT = {Paths.ROOT}, what gets uploaded to Kaggle as a "
      f"dataset): {human_mb(output_dir_bytes)}")
if missing_required:
    print(f"\nWARNING -- required files missing: {missing_required}")
    print("Do not package this as a dataset yet.")
else:
    print("\nAll required files present. Save a notebook version, then create a Kaggle Dataset "
          "from this notebook's output and attach it to N2 (and separately to N3).")


file                                                   present           size
manifest.json                                          True            22,229
corpus.parquet                                         True         7,658,835
corpus.parquet.fp                                      True                16
stratum_counts.json                                    True               204
split_manifest.json                                    True           294,155
labels_stratified.parquet                              True         9,660,751
labels_naive.parquet                                   True         9,490,564
labels_qptm.parquet                                    True           973,511
exclusion_mask.parquet                                 True            31,093
crosstalk.parquet                                      True           422,146
step_log.json                                          True            35,044
qptm_dbptm_overlap.csv                                 True     